In [ ]:
#Marios Ellinidis 4926
#Nikolaos Konstantinidis 5155

In [2]:
# Jupyter Notebook Cell 1: Imports
import networkx as nx
import pandas as pd
import numpy as np
from itertools import combinations
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
import pickle
import os

# Cell 2: Load Citation Graph
edges = pd.read_csv("edgelist.txt", header=None, names=["source", "target"])
positive_edges = list(edges.itertuples(index=False, name=None))
G = nx.DiGraph()
G.add_edges_from(edges.itertuples(index=False, name=None))
print(f"Graph loaded with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

# Load negative edges from file
negative_edges = set()
with open("negative_edgelist2.txt", "r") as f:
    for line in f:
        u, v = map(int, line.strip().split(','))
        negative_edges.add((u, v))

print(f"Loaded {len(negative_edges)} negative edges from file.")

undirected_G = G.to_undirected()
print("loading abstracts.txt...")
abstracts = {}
with open("abstracts.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("|--|")
        abstracts[int(parts[0])] = parts[1]
        
print("loading authors.txt...")
authors = {}
with open("authors.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("|--|")
        authors[int(parts[0])] = set(parts[1].split(","))


Graph loaded with 138499 nodes and 1091955 edges.
Loaded 1091955 negative edges from file.
loading abstracts.txt...
loading authors.txt...


**generating negative edges , already implemented ignore**

In [ ]:
all_nodes = list(G.nodes)
negative_edges = set()
while len(negative_edges) < len(positive_edges):
    u, v = np.random.choice(all_nodes, 2, replace=False)
    if (u, v) not in positive_edges and (u, v) not in negative_edges and (u, v) not in test_edges :
        negative_edges.add((u, v))
        if len(negative_edges) % 100_000 == 0:
            print(f"{len(negative_edges)} negative edges generated so far...")


print(f"Positive examples: {len(positive_edges)}, Negative examples: {len(negative_edges)}")
with open("negative_edgelist.txt", "w") as f:
    for u, v in negative_edges:
        f.write(f"{u},{v}\n")

print("Negative edges saved to 'negative_edgelist.txt'")

In [8]:
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
import numpy as np

w2v = KeyedVectors.load_word2vec_format("GoogleNews-vectors-negative300.bin", binary=True)


from tqdm import tqdm
import numpy as np

def get_embedding(text):
    words = text.split()
    vectors = [w2v[w] for w in words if w in w2v]
    return np.mean(vectors, axis=0) if vectors else np.zeros(w2v.vector_size)

doc_embeddings = {}
for pid, text in tqdm(abstracts.items(), desc="Embedding documents"):
    doc_embeddings[pid] = get_embedding(text)

def get_author_based_embedding(pid, authors, doc_embeddings, embedding_dim=300):
    author_set = authors.get(pid, set())
    author_docs = [
        doc_id for doc_id, auth_set in authors.items()
        if doc_id != pid and auth_set & author_set
    ]
    valid_embeddings = [
        doc_embeddings[doc_id]
        for doc_id in author_docs
        if np.linalg.norm(doc_embeddings[doc_id]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)




X = []
y = []

embedding_dim = 300  # or whatever your embedding size is

# Replace average neighbor fallback with author-based embedding
for a, b in tqdm(positive_edges, desc="Creating features"):

    # Get embedding for node a
    if a in doc_embeddings and np.linalg.norm(doc_embeddings[a]) > 0:
        a_embedding = doc_embeddings[a]
    else:
        a_embedding = get_author_based_embedding(a, authors, doc_embeddings, embedding_dim)
        
    # Get embedding for node b
    if b in doc_embeddings and np.linalg.norm(doc_embeddings[b]) > 0:
        b_embedding = doc_embeddings[b]
    else:
        b_embedding = get_author_based_embedding(b, authors, doc_embeddings, embedding_dim)
       

    # Feature vector
    X.append([
        len(list(nx.common_neighbors(undirected_G, a, b))),
        cosine_similarity([a_embedding], [b_embedding])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    
    y.append(1)

for a, b in tqdm(negative_edges, desc="Creating features for negative edges"):
    if a in doc_embeddings and np.linalg.norm(doc_embeddings[a]) > 0:
        a_embedding = doc_embeddings[a]
    else:
        a_embedding = get_author_based_embedding(a, authors, doc_embeddings, embedding_dim)
        

    if b in doc_embeddings and np.linalg.norm(doc_embeddings[b]) > 0:
        b_embedding = doc_embeddings[b]
    else:
        b_embedding = get_author_based_embedding(b, authors, doc_embeddings, embedding_dim)


    X.append([
        len(list(nx.common_neighbors(undirected_G, a, b))),
        cosine_similarity([a_embedding], [b_embedding])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    
    y.append(0)

from sklearn.linear_model import LogisticRegression


X_np = np.array(X)
y_np = np.array(y)


model = LogisticRegression(max_iter=1000, random_state=42)


model.fit(X_np, y_np)

print("Model trained on full dataset.")


# Test set feature extraction
X_test = []
missing_nodes = []

test_edges = []

with open("test.txt", "r") as f:
    for line in f:
        a, b = map(int, line.strip().split(","))
        test_edges.append((a, b))

# Print how many edges were loaded
print(f"Loaded {len(test_edges)} edges from test.txt")

for a, b in tqdm(test_edges, desc="Extracting features from test set"):
    if a in doc_embeddings and np.linalg.norm(doc_embeddings[a]) > 0:
        a_embedding = doc_embeddings[a]
    else:
        a_embedding = get_author_based_embedding(a, authors, doc_embeddings, embedding_dim)
        missing_nodes.append(a)

    if b in doc_embeddings and np.linalg.norm(doc_embeddings[b]) > 0:
        b_embedding = doc_embeddings[b]
    else:
        b_embedding = get_author_based_embedding(b, authors, doc_embeddings, embedding_dim)
        missing_nodes.append(b)

    X_test.append([
        len(list(nx.common_neighbors(undirected_G, a, b))),
        cosine_similarity([a_embedding], [b_embedding])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])


X_test_np = np.array(X_test)

y_test_probs = model.predict_proba(X_test_np)[:, 1]

submission_df = pd.DataFrame({
    "ID": range(len(y_test_probs)),
    "Label": y_test_probs
})


submission_df.to_csv("author_based_submission.csv", index=False)

print("Submission file saved as 'author_based_submission.csv'")


X = np.array(X)
y = np.array(y)

# Initialize Stratified K-Fold
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

log_losses = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y), 1):
    print(f"\nFold {fold}")
    
    # Split data
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    # Train logistic regression
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    
    # Predict probabilities
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    
    # Compute log loss
    loss = log_loss(y_val, y_pred_proba)
    log_losses.append(loss)
    print(f"Log Loss: {loss:.4f}")

# Final average log loss
avg_loss = np.mean(log_losses)
print(f"\nAverage Log Loss over 5 folds: {avg_loss:.4f}")

Creating features for negative edges: 100%|████████████████| 1091955/1091955 [24:31<00:00, 742.19it/s]


Model trained on full dataset.
Loaded 106692 edges from test.txt


Extracting features from test set: 100%|█████████████████████| 106692/106692 [01:47<00:00, 995.87it/s]


Submission file saved as 'author_based_submission.csv'

Fold 1
Log Loss: 0.2645

Fold 2
Log Loss: 0.2639

Fold 3
Log Loss: 0.2633

Fold 4
Log Loss: 0.2641

Fold 5
Log Loss: 0.2653

Average Log Loss over 5 folds: 0.2642


In [8]:
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
import numpy as np

# Load Word2Vec
w2v = KeyedVectors.load_word2vec_format("GoogleNews-vectors-negative300.bin", binary=True)

# Get embedding from text
def get_embedding(text):
    words = text.split()
    vectors = [w2v[w] for w in words if w in w2v]
    return np.mean(vectors, axis=0) if vectors else np.zeros(w2v.vector_size)

# Create document embeddings
doc_embeddings = {}
for pid, text in tqdm(abstracts.items(), desc="Embedding documents"):
    doc_embeddings[pid] = get_embedding(text)

# Fallback: average neighbor embedding
def get_average_neighbor_embedding(node, graph, doc_embeddings, embedding_dim=300):
    neighbors = list(graph.neighbors(node))
    neighbor_embeds = [
        doc_embeddings[n]
        for n in neighbors if n in doc_embeddings and np.linalg.norm(doc_embeddings[n]) > 0
    ]
    if neighbor_embeds:
        return np.mean(neighbor_embeds, axis=0)
    else:
        return np.zeros(embedding_dim)

# Feature and label construction
X = []
y = []
embedding_dim = 300

# Positive edges
for a, b in tqdm(positive_edges, desc="Creating features for positive edges"):

    a_embedding = doc_embeddings.get(a)
    if a_embedding is None or np.linalg.norm(a_embedding) == 0:
        a_embedding = get_average_neighbor_embedding(a, undirected_G, doc_embeddings, embedding_dim)

    b_embedding = doc_embeddings.get(b)
    if b_embedding is None or np.linalg.norm(b_embedding) == 0:
        b_embedding = get_average_neighbor_embedding(b, undirected_G, doc_embeddings, embedding_dim)

    X.append([
        len(list(nx.common_neighbors(undirected_G, a, b))),
        cosine_similarity([a_embedding], [b_embedding])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(1)

# Negative edges
for a, b in tqdm(negative_edges, desc="Creating features for negative edges"):

    a_embedding = doc_embeddings.get(a)
    if a_embedding is None or np.linalg.norm(a_embedding) == 0:
        a_embedding = get_average_neighbor_embedding(a, undirected_G, doc_embeddings, embedding_dim)

    b_embedding = doc_embeddings.get(b)
    if b_embedding is None or np.linalg.norm(b_embedding) == 0:
        b_embedding = get_average_neighbor_embedding(b, undirected_G, doc_embeddings, embedding_dim)

    X.append([
        len(list(nx.common_neighbors(undirected_G, a, b))),
        cosine_similarity([a_embedding], [b_embedding])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(0)

# Train classifier
X_np = np.array(X)
y_np = np.array(y)

model = LogisticRegression(max_iter=500, random_state=42)
model.fit(X_np, y_np)

print("Model trained on full dataset.")

# Test set feature extraction
X_test = []
missing_nodes = []
test_edges = []

with open("test.txt", "r") as f:
    for line in f:
        a, b = map(int, line.strip().split(","))
        test_edges.append((a, b))

print(f"Loaded {len(test_edges)} edges from test.txt")

for a, b in tqdm(test_edges, desc="Extracting features from test set"):

    a_embedding = doc_embeddings.get(a)
    if a_embedding is None or np.linalg.norm(a_embedding) == 0:
        a_embedding = get_average_neighbor_embedding(a, undirected_G, doc_embeddings, embedding_dim)
        missing_nodes.append(a)

    b_embedding = doc_embeddings.get(b)
    if b_embedding is None or np.linalg.norm(b_embedding) == 0:
        b_embedding = get_average_neighbor_embedding(b, undirected_G, doc_embeddings, embedding_dim)
        missing_nodes.append(b)

    X_test.append([
        len(list(nx.common_neighbors(undirected_G, a, b))),
        cosine_similarity([a_embedding], [b_embedding])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])

# Predict
X_test_np = np.array(X_test)
y_test_probs = model.predict_proba(X_test_np)[:, 1]

submission_df = pd.DataFrame({
    "ID": range(len(y_test_probs)),
    "Label": y_test_probs
})

submission_df.to_csv("submission.csv", index=False)
print("Submission file saved as 'submission.csv'")



X = np.array(X)
y = np.array(y)

# Initialize Stratified K-Fold
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

log_losses = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y), 1):
    print(f"\nFold {fold}")
    
    # Split data
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    # Train logistic regression
    model = LogisticRegression(max_iter=500)
    model.fit(X_train, y_train)
    
    # Predict probabilities
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    
    # Compute log loss
    loss = log_loss(y_val, y_pred_proba)
    log_losses.append(loss)
    print(f"Log Loss: {loss:.4f}")

# Final average log loss
avg_loss = np.mean(log_losses)
print(f"\nAverage Log Loss over 5 folds: {avg_loss:.4f}")

Creating features for negative edges: 100%|███████████████| 1091955/1091955 [02:33<00:00, 7109.86it/s]


Model trained on full dataset.
Loaded 106692 edges from test.txt


Extracting features from test set: 100%|████████████████████| 106692/106692 [00:38<00:00, 2752.30it/s]


Submission file saved as 'submission2.csv'

Fold 1
Log Loss: 0.2554

Fold 2
Log Loss: 0.2558

Fold 3
Log Loss: 0.2541

Fold 4
Log Loss: 0.2554

Fold 5
Log Loss: 0.2557

Average Log Loss over 5 folds: 0.2553


**hybrid approach**

In [8]:
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

# Load Word2Vec
w2v = KeyedVectors.load_word2vec_format("GoogleNews-vectors-negative300.bin", binary=True)

# Get embedding from text
def get_embedding(text):
    words = text.split()
    vectors = [w2v[w] for w in words if w in w2v]
    return np.mean(vectors, axis=0) if vectors else np.zeros(w2v.vector_size)

# Create document embeddings
doc_embeddings = {}
for pid, text in tqdm(abstracts.items(), desc="Embedding documents"):
    doc_embeddings[pid] = get_embedding(text)

# Author-based embedding
def get_author_based_embedding(pid, authors, doc_embeddings, embedding_dim=300):
    author_set = authors.get(pid, set())
    author_docs = [
        doc_id for doc_id, auth_set in authors.items()
        if doc_id != pid and auth_set & author_set
    ]
    valid_embeddings = [
        doc_embeddings[doc_id]
        for doc_id in author_docs
        if np.linalg.norm(doc_embeddings[doc_id]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)

# Neighbor-based embedding
def get_average_neighbor_embedding(node, graph, doc_embeddings, embedding_dim=300):
    neighbors = list(graph.neighbors(node))
    neighbor_embeds = [
        doc_embeddings[n]
        for n in neighbors if n in doc_embeddings and np.linalg.norm(doc_embeddings[n]) > 0
    ]
    return np.mean(neighbor_embeds, axis=0) if neighbor_embeds else np.zeros(embedding_dim)

# Hybrid embedding: weighted average
def get_hybrid_embedding(pid, graph, authors, doc_embeddings, embedding_dim=300, w_neigh=0.6, w_auth=0.4):
    neighbor_emb = get_average_neighbor_embedding(pid, graph, doc_embeddings, embedding_dim)
    author_emb = get_author_based_embedding(pid, authors, doc_embeddings, embedding_dim)
    if np.linalg.norm(neighbor_emb) == 0 and np.linalg.norm(author_emb) == 0:
        return np.zeros(embedding_dim)
    return w_neigh * neighbor_emb + w_auth * author_emb


# Feature and label construction
X = []
y = []
embedding_dim = 300

# Positive edges
for a, b in tqdm(positive_edges, desc="Creating features for positive edges"):

    a_embedding = doc_embeddings.get(a)
    if a_embedding is None or np.linalg.norm(a_embedding) == 0:
        a_embedding = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)

    b_embedding = doc_embeddings.get(b)
    if b_embedding is None or np.linalg.norm(b_embedding) == 0:
        b_embedding = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)

    X.append([
        len(list(nx.common_neighbors(undirected_G, a, b))),
        cosine_similarity([a_embedding], [b_embedding])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(1)

# Negative edges
for a, b in tqdm(negative_edges, desc="Creating features for negative edges"):

    a_embedding = doc_embeddings.get(a)
    if a_embedding is None or np.linalg.norm(a_embedding) == 0:
        a_embedding = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)

    b_embedding = doc_embeddings.get(b)
    if b_embedding is None or np.linalg.norm(b_embedding) == 0:
        b_embedding = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)

    X.append([
        len(list(nx.common_neighbors(undirected_G, a, b))),
        cosine_similarity([a_embedding], [b_embedding])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(0)

# Train classifier
X_np = np.array(X)
y_np = np.array(y)

model = LogisticRegression(max_iter=500, random_state=42)
model.fit(X_np, y_np)

print("Model trained on full dataset.")

# Test set feature extraction
X_test = []
missing_nodes = []
test_edges = []

with open("test.txt", "r") as f:
    for line in f:
        a, b = map(int, line.strip().split(","))
        test_edges.append((a, b))

print(f"Loaded {len(test_edges)} edges from test.txt")

for a, b in tqdm(test_edges, desc="Extracting features from test set"):

    a_embedding = doc_embeddings.get(a)
    if a_embedding is None or np.linalg.norm(a_embedding) == 0:
        a_embedding = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)
        missing_nodes.append(a)

    b_embedding = doc_embeddings.get(b)
    if b_embedding is None or np.linalg.norm(b_embedding) == 0:
        b_embedding = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)
        missing_nodes.append(b)

    X_test.append([
        len(list(nx.common_neighbors(undirected_G, a, b))),
        cosine_similarity([a_embedding], [b_embedding])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])

# Predict
X_test_np = np.array(X_test)
y_test_probs = model.predict_proba(X_test_np)[:, 1]

submission_df = pd.DataFrame({
    "ID": range(len(y_test_probs)),
    "Label": y_test_probs
})

submission_df.to_csv("hybrid_submission.csv", index=False)
print("Submission file saved as 'hybrid_submission.csv'")

# Cross-validation
X = np.array(X)
y = np.array(y)
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
log_losses = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y), 1):
    print(f"\nFold {fold}")
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    model = LogisticRegression(max_iter=500)
    model.fit(X_train, y_train)
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    
    loss = log_loss(y_val, y_pred_proba)
    log_losses.append(loss)
    print(f"Log Loss: {loss:.4f}")

print(f"\nAverage Log Loss over 5 folds: {np.mean(log_losses):.4f}")


Creating features for negative edges: 100%|████████████████| 1091955/1091955 [24:19<00:00, 748.07it/s]


Model trained on full dataset.
Loaded 106692 edges from test.txt


Extracting features from test set: 100%|█████████████████████| 106692/106692 [01:46<00:00, 999.33it/s]


Submission file saved as 'hybrid_submission.csv'

Fold 1
Log Loss: 0.2570

Fold 2
Log Loss: 0.2576

Fold 3
Log Loss: 0.2560

Fold 4
Log Loss: 0.2573

Fold 5
Log Loss: 0.2575

Average Log Loss over 5 folds: 0.2571


**fall back hybrid**

In [8]:
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

# Load Word2Vec
w2v = KeyedVectors.load_word2vec_format("GoogleNews-vectors-negative300.bin", binary=True)

embedding_stats = {
    "abstract": 0,
    "neighbor_fallback": 0,
    "author_fallback": 0,
    "zero_vector": 0  # Optional: track how many times all methods failed
}


# Get embedding from text
def get_embedding(text):
    words = text.split()
    vectors = [w2v[w] for w in words if w in w2v]
    return np.mean(vectors, axis=0) if vectors else np.zeros(w2v.vector_size)

# Create document embeddings
doc_embeddings = {}
for pid, text in tqdm(abstracts.items(), desc="Embedding documents"):
    doc_embeddings[pid] = get_embedding(text)

# Author-based embedding
def get_author_based_embedding(pid, authors, doc_embeddings, embedding_dim=300):
    author_set = authors.get(pid, set())
    author_docs = [
        doc_id for doc_id, auth_set in authors.items()
        if doc_id != pid and auth_set & author_set
    ]
    valid_embeddings = [
        doc_embeddings[doc_id]
        for doc_id in author_docs
        if np.linalg.norm(doc_embeddings[doc_id]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)

# Neighbor-based embedding
def get_average_neighbor_embedding(node, graph, doc_embeddings, embedding_dim=300):
    neighbors = list(graph.neighbors(node))
    neighbor_embeds = [
        doc_embeddings[n]
        for n in neighbors if n in doc_embeddings and np.linalg.norm(doc_embeddings[n]) > 0
    ]
    return np.mean(neighbor_embeds, axis=0) if neighbor_embeds else np.zeros(embedding_dim)



def get_hybrid_embedding(pid, graph, authors, doc_embeddings, dim=300):
    emb = doc_embeddings.get(pid)
    embedding_stats["abstract"] += 1
    if emb is not None and np.linalg.norm(emb) > 0:
        return emb

    # Try neighbor-based embedding
    emb = get_average_neighbor_embedding(pid, graph, doc_embeddings, dim)
    if np.linalg.norm(emb) > 0:
        embedding_stats["neighbor_fallback"] += 1
        return emb

    # Try author-based embedding
    emb = get_author_based_embedding(pid, authors, doc_embeddings)
    if np.linalg.norm(emb) > 0:
        embedding_stats["author_fallback"] += 1
        return emb

    # All failed
    embedding_stats["zero_vector"] += 1
    return emb


# Feature and label construction
X = []
y = []
embedding_dim = 300

# Positive edges
for a, b in tqdm(positive_edges, desc="Creating features for positive edges"):

    a_embedding = doc_embeddings.get(a)
    if a_embedding is None or np.linalg.norm(a_embedding) == 0:
        a_embedding = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)

    b_embedding = doc_embeddings.get(b)
    if b_embedding is None or np.linalg.norm(b_embedding) == 0:
        b_embedding = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)

    X.append([
        len(list(nx.common_neighbors(undirected_G, a, b))),
        cosine_similarity([a_embedding], [b_embedding])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(1)

# Negative edges
for a, b in tqdm(negative_edges, desc="Creating features for negative edges"):

    a_embedding = doc_embeddings.get(a)
    if a_embedding is None or np.linalg.norm(a_embedding) == 0:
        a_embedding = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)

    b_embedding = doc_embeddings.get(b)
    if b_embedding is None or np.linalg.norm(b_embedding) == 0:
        b_embedding = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)

    X.append([
        len(list(nx.common_neighbors(undirected_G, a, b))),
        cosine_similarity([a_embedding], [b_embedding])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(0)

# Train classifier
X_np = np.array(X)
y_np = np.array(y)

model = LogisticRegression(max_iter=500, random_state=42)
model.fit(X_np, y_np)

print("Model trained on full dataset.")

# Test set feature extraction
X_test = []
missing_nodes = []
test_edges = []

with open("test.txt", "r") as f:
    for line in f:
        a, b = map(int, line.strip().split(","))
        test_edges.append((a, b))

print(f"Loaded {len(test_edges)} edges from test.txt")

for a, b in tqdm(test_edges, desc="Extracting features from test set"):

    a_embedding = doc_embeddings.get(a)
    if a_embedding is None or np.linalg.norm(a_embedding) == 0:
        a_embedding = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)
        missing_nodes.append(a)

    b_embedding = doc_embeddings.get(b)
    if b_embedding is None or np.linalg.norm(b_embedding) == 0:
        b_embedding = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)
        missing_nodes.append(b)

    X_test.append([
        len(list(nx.common_neighbors(undirected_G, a, b))),
        cosine_similarity([a_embedding], [b_embedding])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])



# Predict
X_test_np = np.array(X_test)
y_test_probs = model.predict_proba(X_test_np)[:, 1]

submission_df = pd.DataFrame({
    "ID": range(len(y_test_probs)),
    "Label": y_test_probs
})

submission_df.to_csv("fallback_hybrid_submission.csv", index=False)
print("Submission file saved as 'fallbakc_hybrid_submission.csv'")

print("\n=== Embedding Source Statistics ===")
print(f"Direct abstract embeddings used: {embedding_stats['abstract']}")
print(f"Neighbor-based fallback used:    {embedding_stats['neighbor_fallback']}")
print(f"Author-based fallback used:      {embedding_stats['author_fallback']}")
print(f"Zero-vector used (all failed):   {embedding_stats['zero_vector']}")

# Cross-validation
X = np.array(X)
y = np.array(y)
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
log_losses = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y), 1):
    print(f"\nFold {fold}")
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    model = LogisticRegression(max_iter=500)
    model.fit(X_train, y_train)
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    
    loss = log_loss(y_val, y_pred_proba)
    log_losses.append(loss)
    print(f"Log Loss: {loss:.4f}")

print(f"\nAverage Log Loss over 5 folds: {np.mean(log_losses):.4f}")



Creating features for negative edges: 100%|███████████████| 1091955/1091955 [03:38<00:00, 5000.12it/s]


Model trained on full dataset.
Loaded 106692 edges from test.txt


Extracting features from test set: 100%|████████████████████| 106692/106692 [00:19<00:00, 5565.16it/s]


Submission file saved as 'fallbakc_hybrid_submission.csv'

=== Embedding Source Statistics ===
Direct abstract embeddings used: 172881
Neighbor-based fallback used:    166998
Author-based fallback used:      5546
Zero-vector used (all failed):   337

Fold 1
Log Loss: 0.2538

Fold 2
Log Loss: 0.2544

Fold 3
Log Loss: 0.2528

Fold 4
Log Loss: 0.2540

Fold 5
Log Loss: 0.2541

Average Log Loss over 5 folds: 0.2538


**fallback author first**

In [4]:
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

# Load Word2Vec
w2v = KeyedVectors.load_word2vec_format("GoogleNews-vectors-negative300.bin", binary=True)

embedding_stats = {
    "abstract": 0,
    "neighbor_fallback": 0,
    "author_fallback": 0,
    "zero_vector": 0  # Optional: track how many times all methods failed
}


# Get embedding from text
def get_embedding(text):
    words = text.split()
    vectors = [w2v[w] for w in words if w in w2v]
    return np.mean(vectors, axis=0) if vectors else np.zeros(w2v.vector_size)

# Create document embeddings
doc_embeddings = {}
for pid, text in tqdm(abstracts.items(), desc="Embedding documents"):
    doc_embeddings[pid] = get_embedding(text)

# Author-based embedding
def get_author_based_embedding(pid, authors, doc_embeddings, embedding_dim=300):
    author_set = authors.get(pid, set())
    author_docs = [
        doc_id for doc_id, auth_set in authors.items()
        if doc_id != pid and auth_set & author_set
    ]
    valid_embeddings = [
        doc_embeddings[doc_id]
        for doc_id in author_docs
        if np.linalg.norm(doc_embeddings[doc_id]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)

# Neighbor-based embedding
def get_average_neighbor_embedding(node, graph, doc_embeddings, embedding_dim=300):
    neighbors = list(graph.neighbors(node))
    neighbor_embeds = [
        doc_embeddings[n]
        for n in neighbors if n in doc_embeddings and np.linalg.norm(doc_embeddings[n]) > 0
    ]
    return np.mean(neighbor_embeds, axis=0) if neighbor_embeds else np.zeros(embedding_dim)



def get_hybrid_embedding(pid, graph, authors, doc_embeddings, dim=300):
    emb = doc_embeddings.get(pid)
    # Count every time we try abstract embedding
    if emb is not None and np.linalg.norm(emb) > 0:
        embedding_stats["abstract"] += 1 
        return emb

    # Try author-based embedding first
    emb = get_author_based_embedding(pid, authors, doc_embeddings)
    
    if np.linalg.norm(emb) > 0:
        embedding_stats["author_fallback"] += 1
        return emb

    # Then try neighbor-based embedding
    emb = get_average_neighbor_embedding(pid, graph, doc_embeddings, dim)
    
    if np.linalg.norm(emb) > 0:
        embedding_stats["neighbor_fallback"] += 1
        return emb

    embedding_stats["zero_vector"] += 1
    return emb



# Feature and label construction
X = []
y = []
embedding_dim = 300

# Positive edges
for a, b in tqdm(positive_edges, desc="Creating features for positive edges"):

    a_embedding = doc_embeddings.get(a)
    if a_embedding is None or np.linalg.norm(a_embedding) == 0:
        a_embedding = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)

    b_embedding = doc_embeddings.get(b)
    if b_embedding is None or np.linalg.norm(b_embedding) == 0:
        b_embedding = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)

    X.append([
        len(list(nx.common_neighbors(undirected_G, a, b))),
        cosine_similarity([a_embedding], [b_embedding])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(1)

# Negative edges
for a, b in tqdm(negative_edges, desc="Creating features for negative edges"):

    a_embedding = doc_embeddings.get(a)
    if a_embedding is None or np.linalg.norm(a_embedding) == 0:
        a_embedding = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)

    b_embedding = doc_embeddings.get(b)
    if b_embedding is None or np.linalg.norm(b_embedding) == 0:
        b_embedding = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)

    X.append([
        len(list(nx.common_neighbors(undirected_G, a, b))),
        cosine_similarity([a_embedding], [b_embedding])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(0)

# Train classifier
X_np = np.array(X)
y_np = np.array(y)

model = LogisticRegression(max_iter=500, random_state=42)
model.fit(X_np, y_np)

print("Model trained on full dataset.")

# Test set feature extraction
X_test = []
missing_nodes = []
test_edges = []

with open("test.txt", "r") as f:
    for line in f:
        a, b = map(int, line.strip().split(","))
        test_edges.append((a, b))

print(f"Loaded {len(test_edges)} edges from test.txt")

for a, b in tqdm(test_edges, desc="Extracting features from test set"):

    a_embedding = doc_embeddings.get(a)
    if a_embedding is None or np.linalg.norm(a_embedding) == 0:
        a_embedding = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)
        missing_nodes.append(a)

    b_embedding = doc_embeddings.get(b)
    if b_embedding is None or np.linalg.norm(b_embedding) == 0:
        b_embedding = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)
        missing_nodes.append(b)

    X_test.append([
        len(list(nx.common_neighbors(undirected_G, a, b))),
        cosine_similarity([a_embedding], [b_embedding])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])



# Predict
X_test_np = np.array(X_test)
y_test_probs = model.predict_proba(X_test_np)[:, 1]

submission_df = pd.DataFrame({
    "ID": range(len(y_test_probs)),
    "Label": y_test_probs
})

submission_df.to_csv("author_firsst_fallback_hybrid_submission.csv", index=False)
print("Submission file saved as 'authorfirst_fallback_hybrid_submission.csv'")

print("\n=== Embedding Source Statistics ===")
print(f"Direct abstract embeddings used: {embedding_stats['abstract']}")
print(f"Neighbor-based fallback used:    {embedding_stats['neighbor_fallback']}")
print(f"Author-based fallback used:      {embedding_stats['author_fallback']}")
print(f"Zero-vector used (all failed):   {embedding_stats['zero_vector']}")

# Cross-validation
X = np.array(X)
y = np.array(y)
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
log_losses = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y), 1):
    print(f"\nFold {fold}")
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    model = LogisticRegression(max_iter=500)
    model.fit(X_train, y_train)
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    
    loss = log_loss(y_val, y_pred_proba)
    log_losses.append(loss)
    print(f"Log Loss: {loss:.4f}")

print(f"\nAverage Log Loss over 5 folds: {np.mean(log_losses):.4f}")

Embedding documents: 100%|████████████| 138499/138499 [00:19<00:00, 7031.22it/s]
Creating features for positive edges: 100%|█| 1091955/1091955 [12:04<00:00, 1507
Creating features for negative edges: 100%|█| 1091955/1091955 [24:08<00:00, 753.


Model trained on full dataset.
Loaded 106692 edges from test.txt


Extracting features from test set: 100%|█| 106692/106692 [01:45<00:00, 1015.89it


Submission file saved as 'authorfirst_fallback_hybrid_submission.csv'

=== Embedding Source Statistics ===
Direct abstract embeddings used: 0
Neighbor-based fallback used:    6845
Author-based fallback used:      165700
Zero-vector used (all failed):   337

Fold 1
Log Loss: 0.2597

Fold 2
Log Loss: 0.2589

Fold 3
Log Loss: 0.2591

Fold 4
Log Loss: 0.2610

Fold 5
Log Loss: 0.2595

Average Log Loss over 5 folds: 0.2597


**multifallbacks**

In [4]:
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

# Load Word2Vec
w2v = KeyedVectors.load_word2vec_format("GoogleNews-vectors-negative300.bin", binary=True)

embedding_stats = {
    "abstract": 0,
    "neighbor_fallback": 0,
    "2hop_fallback": 0,
    "author_fallback": 0,
    "global_mean_fallback": 0,
    "zero_vector": 0
}

# Get embedding from text
def get_embedding(text):
    words = text.split()
    vectors = [w2v[w] for w in words if w in w2v]
    return np.mean(vectors, axis=0) if vectors else np.zeros(w2v.vector_size)

# Create document embeddings
doc_embeddings = {}
for pid, text in tqdm(abstracts.items(), desc="Embedding documents"):
    doc_embeddings[pid] = get_embedding(text)

# Precompute global mean
valid_embeds = [v for v in doc_embeddings.values() if np.linalg.norm(v) > 0]
global_mean = np.mean(valid_embeds, axis=0) if valid_embeds else np.zeros(w2v.vector_size)

# Author-based embedding
def get_author_based_embedding(pid, authors, doc_embeddings, embedding_dim=300):
    author_set = authors.get(pid, set())
    author_docs = [
        doc_id for doc_id, auth_set in authors.items()
        if doc_id != pid and auth_set & author_set
    ]
    valid_embeddings = [
        doc_embeddings[doc_id]
        for doc_id in author_docs
        if np.linalg.norm(doc_embeddings[doc_id]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)

# 1-hop neighbor-based embedding
def get_average_neighbor_embedding(node, graph, doc_embeddings, embedding_dim=300):
    neighbors = list(graph.neighbors(node))
    neighbor_embeds = [
        doc_embeddings[n]
        for n in neighbors if n in doc_embeddings and np.linalg.norm(doc_embeddings[n]) > 0
    ]
    return np.mean(neighbor_embeds, axis=0) if neighbor_embeds else np.zeros(embedding_dim)

# 2-hop neighbor-based embedding
def get_2hop_neighbor_embedding(node, graph, doc_embeddings, embedding_dim=300):
    two_hop_neighbors = set()
    for neighbor in graph.neighbors(node):
        two_hop_neighbors.update(graph.neighbors(neighbor))
    two_hop_neighbors.discard(node)
    valid_embeddings = [
        doc_embeddings[n]
        for n in two_hop_neighbors if n in doc_embeddings and np.linalg.norm(doc_embeddings[n]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)

# Hybrid embedding with full fallback chain
def get_hybrid_embedding(pid, graph, authors, doc_embeddings, dim=300):
    emb = doc_embeddings.get(pid)
    if emb is not None and np.linalg.norm(emb) > 0:
        embedding_stats["abstract"] += 1
        return emb

    emb = get_average_neighbor_embedding(pid, graph, doc_embeddings, dim)
    if np.linalg.norm(emb) > 0:
        embedding_stats["neighbor_fallback"] += 1
        return emb

    emb = get_2hop_neighbor_embedding(pid, graph, doc_embeddings, dim)
    if np.linalg.norm(emb) > 0:
        embedding_stats["2hop_fallback"] += 1
        return emb

    emb = get_author_based_embedding(pid, authors, doc_embeddings, dim)
    if np.linalg.norm(emb) > 0:
        embedding_stats["author_fallback"] += 1
        return emb

    if np.linalg.norm(global_mean) > 0:
        embedding_stats["global_mean_fallback"] += 1
        return global_mean

    embedding_stats["zero_vector"] += 1
    return np.zeros(dim)

# Feature and label construction
X = []
y = []
embedding_dim = 300

for a, b in tqdm(positive_edges, desc="Creating features for positive edges"):
    a_emb = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)
    b_emb = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)

    X.append([
        len(list(nx.common_neighbors(undirected_G, a, b))),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(1)

for a, b in tqdm(negative_edges, desc="Creating features for negative edges"):
    a_emb = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)
    b_emb = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)

    X.append([
        len(list(nx.common_neighbors(undirected_G, a, b))),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(0)

# Train model
X_np = np.array(X)
y_np = np.array(y)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_np, y_np)
print("Model trained on full dataset.")

# Test prediction
X_test = []
missing_nodes = []
test_edges = []

with open("test.txt", "r") as f:
    for line in f:
        a, b = map(int, line.strip().split(","))
        test_edges.append((a, b))

print(f"Loaded {len(test_edges)} edges from test.txt")

for a, b in tqdm(test_edges, desc="Extracting features from test set"):
    a_emb = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)
    b_emb = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)

    X_test.append([
        len(list(nx.common_neighbors(undirected_G, a, b))),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])

# Predict and save
X_test_np = np.array(X_test)
y_test_probs = model.predict_proba(X_test_np)[:, 1]

submission_df = pd.DataFrame({
    "ID": range(len(y_test_probs)),
    "Label": y_test_probs
})
submission_df.to_csv("multifallback_hybrid_submission.csv", index=False)
print("Submission file saved as 'multifallback_hybrid_submission.csv'")

# Embedding stats summary
print("\n=== Embedding Source Statistics ===")
for k, v in embedding_stats.items():
    print(f"{k.replace('_', ' ').capitalize():<28}: {v}")

# Cross-validation
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
log_losses = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_np, y_np), 1):
    print(f"\nFold {fold}")
    model = LogisticRegression(max_iter=1000)
    model.fit(X_np[train_idx], y_np[train_idx])
    y_pred_proba = model.predict_proba(X_np[val_idx])[:, 1]
    loss = log_loss(y_np[val_idx], y_pred_proba)
    log_losses.append(loss)
    print(f"Log Loss: {loss:.4f}")

print(f"\nAverage Log Loss over 5 folds: {np.mean(log_losses):.4f}")


Embedding documents: 100%|████████████| 138499/138499 [00:18<00:00, 7478.15it/s]
Creating features for positive edges: 100%|█| 1091955/1091955 [02:39<00:00, 6827
Creating features for negative edges: 100%|█| 1091955/1091955 [02:32<00:00, 7139


Model trained on full dataset.
Loaded 106692 edges from test.txt


Extracting features from test set: 100%|█| 106692/106692 [00:15<00:00, 6722.89it


Submission file saved as 'multifallback_hybrid_submission.csv'

=== Embedding Source Statistics ===
Abstract                    : 4408322
Neighbor fallback           : 166999
2hop fallback               : 5817
Author fallback             : 54
Global mean fallback        : 12
Zero vector                 : 0

Fold 1
Log Loss: 0.2533

Fold 2
Log Loss: 0.2529

Fold 3
Log Loss: 0.2532

Fold 4
Log Loss: 0.2547

Fold 5
Log Loss: 0.2533

Average Log Loss over 5 folds: 0.2535


**jaccard similarity**

In [8]:
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

# Load Word2Vec
w2v = KeyedVectors.load_word2vec_format("GoogleNews-vectors-negative300.bin", binary=True)


# Get embedding from text
def get_embedding(text):
    words = text.split()
    vectors = [w2v[w] for w in words if w in w2v]
    return np.mean(vectors, axis=0) if vectors else np.zeros(w2v.vector_size)

# Create document embeddings
doc_embeddings = {}
for pid, text in tqdm(abstracts.items(), desc="Embedding documents"):
    doc_embeddings[pid] = get_embedding(text)

# Precompute global mean
valid_embeds = [v for v in doc_embeddings.values() if np.linalg.norm(v) > 0]
global_mean = np.mean(valid_embeds, axis=0) if valid_embeds else np.zeros(w2v.vector_size)

# Author-based embedding
def get_author_based_embedding(pid, authors, doc_embeddings, embedding_dim=300):
    author_set = authors.get(pid, set())
    author_docs = [
        doc_id for doc_id, auth_set in authors.items()
        if doc_id != pid and auth_set & author_set
    ]
    valid_embeddings = [
        doc_embeddings[doc_id]
        for doc_id in author_docs
        if np.linalg.norm(doc_embeddings[doc_id]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)

# 1-hop neighbor-based embedding
def get_average_neighbor_embedding(node, graph, doc_embeddings, embedding_dim=300):
    neighbors = list(graph.neighbors(node))
    neighbor_embeds = [
        doc_embeddings[n]
        for n in neighbors if n in doc_embeddings and np.linalg.norm(doc_embeddings[n]) > 0
    ]
    return np.mean(neighbor_embeds, axis=0) if neighbor_embeds else np.zeros(embedding_dim)

# 2-hop neighbor-based embedding
def get_2hop_neighbor_embedding(node, graph, doc_embeddings, embedding_dim=300):
    two_hop_neighbors = set()
    for neighbor in graph.neighbors(node):
        two_hop_neighbors.update(graph.neighbors(neighbor))
    two_hop_neighbors.discard(node)
    valid_embeddings = [
        doc_embeddings[n]
        for n in two_hop_neighbors if n in doc_embeddings and np.linalg.norm(doc_embeddings[n]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)

# Hybrid embedding with full fallback chain
def get_hybrid_embedding(pid, graph, authors, doc_embeddings, dim=300):
    emb = doc_embeddings.get(pid)
    if emb is not None and np.linalg.norm(emb) > 0:
        return emb

    emb = get_average_neighbor_embedding(pid, graph, doc_embeddings, dim)
    if np.linalg.norm(emb) > 0:
        return emb

    emb = get_2hop_neighbor_embedding(pid, graph, doc_embeddings, dim)
    if np.linalg.norm(emb) > 0:
        return emb

    emb = get_author_based_embedding(pid, authors, doc_embeddings, dim)
    if np.linalg.norm(emb) > 0:
        return emb

    if np.linalg.norm(global_mean) > 0:
        return global_mean

    embedding_stats["zero_vector"] += 1
    return np.zeros(dim)

from sklearn.metrics import jaccard_score

# Function to calculate Jaccard similarity between two sets
def jaccard_similarity(set_a, set_b):
    intersection = len(set_a & set_b)
    union = len(set_a | set_b)
    return intersection / union if union != 0 else 0.0

# Feature and label construction with Jaccard similarity
X = []
y = []
embedding_dim = 300

# For positive edges
for a, b in tqdm(positive_edges, desc="Creating features for positive edges"):
    a_emb = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)
    b_emb = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)
    
    # Get the neighbors for both nodes
    a_neighbors = set(undirected_G.neighbors(a))
    b_neighbors = set(undirected_G.neighbors(b))
    
  
    
    # Add the Jaccard similarity to the feature vector
    X.append([
        #len(list(nx.common_neighbors(undirected_G, a, b))),
        jaccard_similarity(a_neighbors, b_neighbors),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set())),
        
    ])
    y.append(1)

# For negative edges
for a, b in tqdm(negative_edges, desc="Creating features for negative edges"):
    a_emb = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)
    b_emb = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)
    
    # Get the neighbors for both nodes
    a_neighbors = set(undirected_G.neighbors(a))
    b_neighbors = set(undirected_G.neighbors(b))
    
   
    # Add the Jaccard similarity to the feature vector
    X.append([
        #len(list(nx.common_neighbors(undirected_G, a, b))),
        jaccard_similarity(a_neighbors, b_neighbors),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set())),
        
    ])
    y.append(0)

# Train model
X_np = np.array(X)
y_np = np.array(y)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_np, y_np)
print("Model trained on full dataset.")

# Test prediction with Jaccard similarity
X_test = []
missing_nodes = []
test_edges = []

with open("test.txt", "r") as f:
    for line in f:
        a, b = map(int, line.strip().split(","))
        test_edges.append((a, b))

print(f"Loaded {len(test_edges)} edges from test.txt")

for a, b in tqdm(test_edges, desc="Extracting features from test set"):
    a_emb = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)
    b_emb = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)

    # Get the neighbors for both nodes in the test set
    a_neighbors = set(undirected_G.neighbors(a))
    b_neighbors = set(undirected_G.neighbors(b))


    # Add the Jaccard similarity to the feature vector for test edges
    X_test.append([
        len(list(nx.common_neighbors(undirected_G, a, b))),
        jaccard_similarity(a_neighbors, b_neighbors),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set())),
        
    ])

# Predict and save
X_test_np = np.array(X_test)
y_test_probs = model.predict_proba(X_test_np)[:, 1]

submission_df = pd.DataFrame({
    "ID": range(len(y_test_probs)),
    "Label": y_test_probs
})
submission_df.to_csv("submission_jaccard.csv", index=False)
print("Submission file saved as 'submission_jaccard.csv'")



# Cross-validation
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
log_losses = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_np, y_np), 1):
    print(f"\nFold {fold}")
    model = LogisticRegression(max_iter=1000)
    model.fit(X_np[train_idx], y_np[train_idx])
    y_pred_proba = model.predict_proba(X_np[val_idx])[:, 1]
    loss = log_loss(y_np[val_idx], y_pred_proba)
    log_losses.append(loss)
    print(f"Log Loss: {loss:.4f}")

print(f"\nAverage Log Loss over 5 folds: {np.mean(log_losses):.4f}")

Creating features for negative edges: 100%|███████████████| 1091955/1091955 [02:35<00:00, 7016.76it/s]


Model trained on full dataset.
Loaded 106692 edges from test.txt


Extracting features from test set: 100%|████████████████████| 106692/106692 [00:15<00:00, 6865.37it/s]


Submission file saved as 'submission_jaccard.csv'

Fold 1
Log Loss: 0.3127

Fold 2
Log Loss: 0.3131

Fold 3
Log Loss: 0.3114

Fold 4
Log Loss: 0.3135

Fold 5
Log Loss: 0.3108

Average Log Loss over 5 folds: 0.3123


**adamic adar**

In [8]:
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

# Load Word2Vec
w2v = KeyedVectors.load_word2vec_format("GoogleNews-vectors-negative300.bin", binary=True)

embedding_stats = {
    "abstract": 0,
    "neighbor_fallback": 0,
    "2hop_fallback": 0,
    "author_fallback": 0,
    "global_mean_fallback": 0,
    "zero_vector": 0
}

# Get embedding from text
def get_embedding(text):
    words = text.split()
    vectors = [w2v[w] for w in words if w in w2v]
    return np.mean(vectors, axis=0) if vectors else np.zeros(w2v.vector_size)

# Create document embeddings
doc_embeddings = {}
for pid, text in tqdm(abstracts.items(), desc="Embedding documents"):
    doc_embeddings[pid] = get_embedding(text)

# Precompute global mean
valid_embeds = [v for v in doc_embeddings.values() if np.linalg.norm(v) > 0]
global_mean = np.mean(valid_embeds, axis=0) if valid_embeds else np.zeros(w2v.vector_size)

# Author-based embedding
def get_author_based_embedding(pid, authors, doc_embeddings, embedding_dim=300):
    author_set = authors.get(pid, set())
    author_docs = [
        doc_id for doc_id, auth_set in authors.items()
        if doc_id != pid and auth_set & author_set
    ]
    valid_embeddings = [
        doc_embeddings[doc_id]
        for doc_id in author_docs
        if np.linalg.norm(doc_embeddings[doc_id]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)

# 1-hop neighbor-based embedding
def get_average_neighbor_embedding(node, graph, doc_embeddings, embedding_dim=300):
    neighbors = list(graph.neighbors(node))
    neighbor_embeds = [
        doc_embeddings[n]
        for n in neighbors if n in doc_embeddings and np.linalg.norm(doc_embeddings[n]) > 0
    ]
    return np.mean(neighbor_embeds, axis=0) if neighbor_embeds else np.zeros(embedding_dim)

# 2-hop neighbor-based embedding
def get_2hop_neighbor_embedding(node, graph, doc_embeddings, embedding_dim=300):
    two_hop_neighbors = set()
    for neighbor in graph.neighbors(node):
        two_hop_neighbors.update(graph.neighbors(neighbor))
    two_hop_neighbors.discard(node)
    valid_embeddings = [
        doc_embeddings[n]
        for n in two_hop_neighbors if n in doc_embeddings and np.linalg.norm(doc_embeddings[n]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)

# Hybrid embedding with full fallback chain
def get_hybrid_embedding(pid, graph, authors, doc_embeddings, dim=300):
    emb = doc_embeddings.get(pid)
    if emb is not None and np.linalg.norm(emb) > 0:
        embedding_stats["abstract"] += 1
        return emb

    emb = get_average_neighbor_embedding(pid, graph, doc_embeddings, dim)
    if np.linalg.norm(emb) > 0:
        embedding_stats["neighbor_fallback"] += 1
        return emb

    emb = get_2hop_neighbor_embedding(pid, graph, doc_embeddings, dim)
    if np.linalg.norm(emb) > 0:
        embedding_stats["2hop_fallback"] += 1
        return emb

    emb = get_author_based_embedding(pid, authors, doc_embeddings, dim)
    if np.linalg.norm(emb) > 0:
        embedding_stats["author_fallback"] += 1
        return emb

    if np.linalg.norm(global_mean) > 0:
        embedding_stats["global_mean_fallback"] += 1
        return global_mean

    embedding_stats["zero_vector"] += 1
    return np.zeros(dim)

from networkx.algorithms.link_prediction import adamic_adar_index

# Feature and label construction with Adamic-Adar
X = []
y = []
embedding_dim = 300

def compute_adamic_adar(graph, u, v):
    try:
        return list(adamic_adar_index(graph, [(u, v)]))[0][2]
    except:
        return 0.0

for a, b in tqdm(positive_edges, desc="Creating features for positive edges (Adamic-Adar)"):
    a_emb = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)
    b_emb = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)

    X.append([
        compute_adamic_adar(undirected_G, a, b),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(1)

for a, b in tqdm(negative_edges, desc="Creating features for negative edges (Adamic-Adar)"):
    a_emb = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)
    b_emb = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)

    X.append([
        compute_adamic_adar(undirected_G, a, b),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(0)


# Train model
X_np = np.array(X)
y_np = np.array(y)

model = LogisticRegression(max_iter=500, random_state=42)
model.fit(X_np, y_np)
print("Model trained on full dataset.")

# Test prediction
X_test = []
missing_nodes = []
test_edges = []

with open("test.txt", "r") as f:
    for line in f:
        a, b = map(int, line.strip().split(","))
        test_edges.append((a, b))

print(f"Loaded {len(test_edges)} edges from test.txt")

for a, b in tqdm(test_edges, desc="Extracting features from test set"):
    a_emb = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)
    b_emb = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)

    X_test.append([
        compute_adamic_adar(undirected_G, a, b),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])

# Predict and save
X_test_np = np.array(X_test)
y_test_probs = model.predict_proba(X_test_np)[:, 1]

submission_df = pd.DataFrame({
    "ID": range(len(y_test_probs)),
    "Label": y_test_probs
})
submission_df.to_csv("submission_adamic_adar.csv", index=False)
print("Submission file saved as 'submission_adamic_adar.csv'")

# Embedding stats summary
print("\n=== Embedding Source Statistics ===")
for k, v in embedding_stats.items():
    print(f"{k.replace('_', ' ').capitalize():<28}: {v}")

# Cross-validation
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
log_losses = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_np, y_np), 1):
    print(f"\nFold {fold}")
    model = LogisticRegression(max_iter=500)
    model.fit(X_np[train_idx], y_np[train_idx])
    y_pred_proba = model.predict_proba(X_np[val_idx])[:, 1]
    loss = log_loss(y_np[val_idx], y_pred_proba)
    log_losses.append(loss)
    print(f"Log Loss: {loss:.4f}")

print(f"\nAverage Log Loss over 5 folds: {np.mean(log_losses):.4f}")


Embedding documents: 100%|██████████████████████████████████| 138499/138499 [00:18<00:00, 7549.29it/s]
Creating features for positive edges (Adamic-Adar): 100%|█| 1091955/1091955 [02:43<00:00, 6694.37it/s]
Creating features for negative edges (Adamic-Adar): 100%|█| 1091955/1091955 [02:37<00:00, 6937.71it/s]


Model trained on full dataset.
Loaded 106692 edges from test.txt


Extracting features from test set: 100%|████████████████████| 106692/106692 [00:16<00:00, 6424.21it/s]


Submission file saved as 'submission_adamic_adar.csv'

=== Embedding Source Statistics ===
Abstract                    : 4408323
Neighbor fallback           : 166998
2hop fallback               : 5817
Author fallback             : 54
Global mean fallback        : 12
Zero vector                 : 0

Fold 1
Log Loss: 0.2502

Fold 2
Log Loss: 0.2509

Fold 3
Log Loss: 0.2495

Fold 4
Log Loss: 0.2508

Fold 5
Log Loss: 0.2509

Average Log Loss over 5 folds: 0.2504


**Preferential Attachment**

In [10]:
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

# Load Word2Vec
w2v = KeyedVectors.load_word2vec_format("GoogleNews-vectors-negative300.bin", binary=True)

embedding_stats = {
    "abstract": 0,
    "neighbor_fallback": 0,
    "2hop_fallback": 0,
    "author_fallback": 0,
    "global_mean_fallback": 0,
    "zero_vector": 0
}

# Get embedding from text
def get_embedding(text):
    words = text.split()
    vectors = [w2v[w] for w in words if w in w2v]
    return np.mean(vectors, axis=0) if vectors else np.zeros(w2v.vector_size)

# Create document embeddings
doc_embeddings = {}
for pid, text in tqdm(abstracts.items(), desc="Embedding documents"):
    doc_embeddings[pid] = get_embedding(text)

# Precompute global mean
valid_embeds = [v for v in doc_embeddings.values() if np.linalg.norm(v) > 0]
global_mean = np.mean(valid_embeds, axis=0) if valid_embeds else np.zeros(w2v.vector_size)

# Author-based embedding
def get_author_based_embedding(pid, authors, doc_embeddings, embedding_dim=300):
    author_set = authors.get(pid, set())
    author_docs = [
        doc_id for doc_id, auth_set in authors.items()
        if doc_id != pid and auth_set & author_set
    ]
    valid_embeddings = [
        doc_embeddings[doc_id]
        for doc_id in author_docs
        if np.linalg.norm(doc_embeddings[doc_id]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)

# 1-hop neighbor-based embedding
def get_average_neighbor_embedding(node, graph, doc_embeddings, embedding_dim=300):
    neighbors = list(graph.neighbors(node))
    neighbor_embeds = [
        doc_embeddings[n]
        for n in neighbors if n in doc_embeddings and np.linalg.norm(doc_embeddings[n]) > 0
    ]
    return np.mean(neighbor_embeds, axis=0) if neighbor_embeds else np.zeros(embedding_dim)

# 2-hop neighbor-based embedding
def get_2hop_neighbor_embedding(node, graph, doc_embeddings, embedding_dim=300):
    two_hop_neighbors = set()
    for neighbor in graph.neighbors(node):
        two_hop_neighbors.update(graph.neighbors(neighbor))
    two_hop_neighbors.discard(node)
    valid_embeddings = [
        doc_embeddings[n]
        for n in two_hop_neighbors if n in doc_embeddings and np.linalg.norm(doc_embeddings[n]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)

# Hybrid embedding with full fallback chain
def get_hybrid_embedding(pid, graph, authors, doc_embeddings, dim=300):
    emb = doc_embeddings.get(pid)
    if emb is not None and np.linalg.norm(emb) > 0:
        embedding_stats["abstract"] += 1
        return emb

    emb = get_average_neighbor_embedding(pid, graph, doc_embeddings, dim)
    if np.linalg.norm(emb) > 0:
        embedding_stats["neighbor_fallback"] += 1
        return emb

    emb = get_2hop_neighbor_embedding(pid, graph, doc_embeddings, dim)
    if np.linalg.norm(emb) > 0:
        embedding_stats["2hop_fallback"] += 1
        return emb

    emb = get_author_based_embedding(pid, authors, doc_embeddings, dim)
    if np.linalg.norm(emb) > 0:
        embedding_stats["author_fallback"] += 1
        return emb

    if np.linalg.norm(global_mean) > 0:
        embedding_stats["global_mean_fallback"] += 1
        return global_mean

    embedding_stats["zero_vector"] += 1
    return np.zeros(dim)
    
embedding_dim = 300

from networkx.algorithms.link_prediction import preferential_attachment

# Define Preferential Attachment function
def compute_preferential_attachment(graph, u, v):
    try:
        return list(preferential_attachment(graph, [(u, v)]))[0][2]
    except:
        return 0.0

# Build features and labels
X = []
y = []

for a, b in tqdm(positive_edges, desc="Creating features for positive edges (PA)"):
    a_emb = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)
    b_emb = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)

    X.append([
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(1)

for a, b in tqdm(negative_edges, desc="Creating features for negative edges (PA)"):
    a_emb = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)
    b_emb = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)

    X.append([
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(0)

# Train model
X_np = np.array(X)
y_np = np.array(y)

model = LogisticRegression(max_iter=500, random_state=42)
model.fit(X_np, y_np)
print("Model trained on full dataset with Preferential Attachment.")

# Test set prediction
X_test = []
test_edges = []

with open("test.txt", "r") as f:
    for line in f:
        a, b = map(int, line.strip().split(","))
        test_edges.append((a, b))

print(f"Loaded {len(test_edges)} edges from test.txt")

for a, b in tqdm(test_edges, desc="Extracting features from test set"):
    a_emb = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)
    b_emb = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)

    X_test.append([
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])

# Predict and save
X_test_np = np.array(X_test)
y_test_probs = model.predict_proba(X_test_np)[:, 1]

submission_df = pd.DataFrame({
    "ID": range(len(y_test_probs)),
    "Label": y_test_probs
})
submission_df.to_csv("submission_preferential_attachment.csv", index=False)
print("Submission file saved as 'submission_preferential_attachment.csv'")

# Embedding stats summary
print("\n=== Embedding Source Statistics ===")
for k, v in embedding_stats.items():
    print(f"{k.replace('_', ' ').capitalize():<28}: {v}")

# Cross-validation
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
log_losses = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_np, y_np), 1):
    print(f"\nFold {fold}")
    model = LogisticRegression(max_iter=500)
    model.fit(X_np[train_idx], y_np[train_idx])
    y_pred_proba = model.predict_proba(X_np[val_idx])[:, 1]
    loss = log_loss(y_np[val_idx], y_pred_proba)
    log_losses.append(loss)
    print(f"Log Loss: {loss:.4f}")

print(f"\nAverage Log Loss over 5 folds: {np.mean(log_losses):.4f}")


Creating features for negative edges (PA): 100%|██████████| 1091955/1091955 [02:35<00:00, 7006.69it/s]


Model trained on full dataset with Preferential Attachment.
Loaded 106692 edges from test.txt


Extracting features from test set: 100%|████████████████████| 106692/106692 [00:15<00:00, 7040.59it/s]


Submission file saved as 'submission_preferential_attachment.csv'

=== Embedding Source Statistics ===
Abstract                    : 4408323
Neighbor fallback           : 166998
2hop fallback               : 5817
Author fallback             : 54
Global mean fallback        : 12
Zero vector                 : 0

Fold 1
Log Loss: 0.4113

Fold 2
Log Loss: 0.4108

Fold 3
Log Loss: 0.4092

Fold 4
Log Loss: 0.4095

Fold 5
Log Loss: 0.4131

Average Log Loss over 5 folds: 0.4108


**adamic adar+preferential**

In [8]:
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

# Load Word2Vec
w2v = KeyedVectors.load_word2vec_format("GoogleNews-vectors-negative300.bin", binary=True)

embedding_stats = {
    "abstract": 0,
    "neighbor_fallback": 0,
    "2hop_fallback": 0,
    "author_fallback": 0,
    "global_mean_fallback": 0,
    "zero_vector": 0
}

# Get embedding from text
def get_embedding(text):
    words = text.split()
    vectors = [w2v[w] for w in words if w in w2v]
    return np.mean(vectors, axis=0) if vectors else np.zeros(w2v.vector_size)

# Create document embeddings
doc_embeddings = {}
for pid, text in tqdm(abstracts.items(), desc="Embedding documents"):
    doc_embeddings[pid] = get_embedding(text)

# Precompute global mean
valid_embeds = [v for v in doc_embeddings.values() if np.linalg.norm(v) > 0]
global_mean = np.mean(valid_embeds, axis=0) if valid_embeds else np.zeros(w2v.vector_size)

# Author-based embedding
def get_author_based_embedding(pid, authors, doc_embeddings, embedding_dim=300):
    author_set = authors.get(pid, set())
    author_docs = [
        doc_id for doc_id, auth_set in authors.items()
        if doc_id != pid and auth_set & author_set
    ]
    valid_embeddings = [
        doc_embeddings[doc_id]
        for doc_id in author_docs
        if np.linalg.norm(doc_embeddings[doc_id]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)

# 1-hop neighbor-based embedding
def get_average_neighbor_embedding(node, graph, doc_embeddings, embedding_dim=300):
    neighbors = list(graph.neighbors(node))
    neighbor_embeds = [
        doc_embeddings[n]
        for n in neighbors if n in doc_embeddings and np.linalg.norm(doc_embeddings[n]) > 0
    ]
    return np.mean(neighbor_embeds, axis=0) if neighbor_embeds else np.zeros(embedding_dim)

# 2-hop neighbor-based embedding
def get_2hop_neighbor_embedding(node, graph, doc_embeddings, embedding_dim=300):
    two_hop_neighbors = set()
    for neighbor in graph.neighbors(node):
        two_hop_neighbors.update(graph.neighbors(neighbor))
    two_hop_neighbors.discard(node)
    valid_embeddings = [
        doc_embeddings[n]
        for n in two_hop_neighbors if n in doc_embeddings and np.linalg.norm(doc_embeddings[n]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)

# Hybrid embedding with full fallback chain
def get_hybrid_embedding(pid, graph, authors, doc_embeddings, dim=300):
    emb = doc_embeddings.get(pid)
    if emb is not None and np.linalg.norm(emb) > 0:
        embedding_stats["abstract"] += 1
        return emb

    emb = get_average_neighbor_embedding(pid, graph, doc_embeddings, dim)
    if np.linalg.norm(emb) > 0:
        embedding_stats["neighbor_fallback"] += 1
        return emb

    emb = get_2hop_neighbor_embedding(pid, graph, doc_embeddings, dim)
    if np.linalg.norm(emb) > 0:
        embedding_stats["2hop_fallback"] += 1
        return emb

    emb = get_author_based_embedding(pid, authors, doc_embeddings, dim)
    if np.linalg.norm(emb) > 0:
        embedding_stats["author_fallback"] += 1
        return emb

    if np.linalg.norm(global_mean) > 0:
        embedding_stats["global_mean_fallback"] += 1
        return global_mean

    embedding_stats["zero_vector"] += 1
    return np.zeros(dim)

embedding_dim = 300

from networkx.algorithms.link_prediction import adamic_adar_index, preferential_attachment

def compute_adamic_adar(graph, u, v):
    try:
        return list(adamic_adar_index(graph, [(u, v)]))[0][2]
    except:
        return 0.0

def compute_preferential_attachment(graph, u, v):
    try:
        return list(preferential_attachment(graph, [(u, v)]))[0][2]
    except:
        return 0.0

X = []
y = []


# Positive edges
for a, b in tqdm(positive_edges, desc="Creating features for positive edges"):
    a_emb = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)
    b_emb = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)

    X.append([
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(1)

# Negative edges
for a, b in tqdm(negative_edges, desc="Creating features for negative edges"):
    a_emb = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)
    b_emb = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)

    X.append([
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(0)

# Convert to NumPy
X_np = np.array(X)
y_np = np.array(y)


model = LogisticRegression(max_iter=500, random_state=42)
model.fit(X_np, y_np)
print("Model trained on full dataset.")

X_test = []
test_edges = []

with open("test.txt", "r") as f:
    for line in f:
        a, b = map(int, line.strip().split(","))
        test_edges.append((a, b))

print(f"Loaded {len(test_edges)} edges from test.txt")

for a, b in tqdm(test_edges, desc="Extracting features from test set"):
    a_emb = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)
    b_emb = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)

    X_test.append([
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])

X_test_np = np.array(X_test)
y_test_probs = model.predict_proba(X_test_np)[:, 1]

submission_df = pd.DataFrame({
    "ID": range(len(y_test_probs)),
    "Label": y_test_probs
})
submission_df.to_csv("adar_preferential.csv", index=False)
print("Submission file saved as 'adar_preferential.csv'")

# Embedding stats summary
print("\n=== Embedding Source Statistics ===")
for k, v in embedding_stats.items():
    print(f"{k.replace('_', ' ').capitalize():<28}: {v}")

# Cross-validation
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
log_losses = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_np, y_np), 1):
    print(f"\nFold {fold}")
    model = LogisticRegression(max_iter=500)
    model.fit(X_np[train_idx], y_np[train_idx])
    y_pred_proba = model.predict_proba(X_np[val_idx])[:, 1]
    loss = log_loss(y_np[val_idx], y_pred_proba)
    log_losses.append(loss)
    print(f"Log Loss: {loss:.4f}")

print(f"\nAverage Log Loss over 5 folds: {np.mean(log_losses):.4f}")


Creating features for negative edges: 100%|███████████████| 1091955/1091955 [02:42<00:00, 6729.90it/s]


Model trained on full dataset.
Loaded 106692 edges from test.txt


Extracting features from test set: 100%|████████████████████| 106692/106692 [00:17<00:00, 6052.02it/s]


Submission file saved as 'adar_preferential.csv'

=== Embedding Source Statistics ===
Abstract                    : 4408323
Neighbor fallback           : 166998
2hop fallback               : 5817
Author fallback             : 54
Global mean fallback        : 12
Zero vector                 : 0

Fold 1
Log Loss: 0.2344

Fold 2
Log Loss: 0.2342

Fold 3
Log Loss: 0.2323

Fold 4
Log Loss: 0.2336

Fold 5
Log Loss: 0.2345

Average Log Loss over 5 folds: 0.2338


**jaccard+adar+preferrential**

In [11]:
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

# Load Word2Vec
w2v = KeyedVectors.load_word2vec_format("GoogleNews-vectors-negative300.bin", binary=True)

embedding_stats = {
    "abstract": 0,
    "neighbor_fallback": 0,
    "2hop_fallback": 0,
    "author_fallback": 0,
    "global_mean_fallback": 0,
    "zero_vector": 0
}

# Get embedding from text
def get_embedding(text):
    words = text.split()
    vectors = [w2v[w] for w in words if w in w2v]
    return np.mean(vectors, axis=0) if vectors else np.zeros(w2v.vector_size)

# Create document embeddings
doc_embeddings = {}
for pid, text in tqdm(abstracts.items(), desc="Embedding documents"):
    doc_embeddings[pid] = get_embedding(text)

# Precompute global mean
valid_embeds = [v for v in doc_embeddings.values() if np.linalg.norm(v) > 0]
global_mean = np.mean(valid_embeds, axis=0) if valid_embeds else np.zeros(w2v.vector_size)

# Author-based embedding
def get_author_based_embedding(pid, authors, doc_embeddings, embedding_dim=300):
    author_set = authors.get(pid, set())
    author_docs = [
        doc_id for doc_id, auth_set in authors.items()
        if doc_id != pid and auth_set & author_set
    ]
    valid_embeddings = [
        doc_embeddings[doc_id]
        for doc_id in author_docs
        if np.linalg.norm(doc_embeddings[doc_id]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)

# 1-hop neighbor-based embedding
def get_average_neighbor_embedding(node, graph, doc_embeddings, embedding_dim=300):
    neighbors = list(graph.neighbors(node))
    neighbor_embeds = [
        doc_embeddings[n]
        for n in neighbors if n in doc_embeddings and np.linalg.norm(doc_embeddings[n]) > 0
    ]
    return np.mean(neighbor_embeds, axis=0) if neighbor_embeds else np.zeros(embedding_dim)

# 2-hop neighbor-based embedding
def get_2hop_neighbor_embedding(node, graph, doc_embeddings, embedding_dim=300):
    two_hop_neighbors = set()
    for neighbor in graph.neighbors(node):
        two_hop_neighbors.update(graph.neighbors(neighbor))
    two_hop_neighbors.discard(node)
    valid_embeddings = [
        doc_embeddings[n]
        for n in two_hop_neighbors if n in doc_embeddings and np.linalg.norm(doc_embeddings[n]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)

# Hybrid embedding with full fallback chain
def get_hybrid_embedding(pid, graph, authors, doc_embeddings, dim=300):
    emb = doc_embeddings.get(pid)
    if emb is not None and np.linalg.norm(emb) > 0:
        embedding_stats["abstract"] += 1
        return emb

    emb = get_average_neighbor_embedding(pid, graph, doc_embeddings, dim)
    if np.linalg.norm(emb) > 0:
        embedding_stats["neighbor_fallback"] += 1
        return emb

    emb = get_2hop_neighbor_embedding(pid, graph, doc_embeddings, dim)
    if np.linalg.norm(emb) > 0:
        embedding_stats["2hop_fallback"] += 1
        return emb

    emb = get_author_based_embedding(pid, authors, doc_embeddings, dim)
    if np.linalg.norm(emb) > 0:
        embedding_stats["author_fallback"] += 1
        return emb

    if np.linalg.norm(global_mean) > 0:
        embedding_stats["global_mean_fallback"] += 1
        return global_mean

    embedding_stats["zero_vector"] += 1
    return np.zeros(dim)

embedding_dim = 300

from networkx.algorithms.link_prediction import jaccard_coefficient

def compute_jaccard_similarity(graph, u, v):
    try:
        return list(jaccard_coefficient(graph, [(u, v)]))[0][2]
    except:
        return 0.0



from networkx.algorithms.link_prediction import adamic_adar_index, preferential_attachment

def compute_adamic_adar(graph, u, v):
    try:
        return list(adamic_adar_index(graph, [(u, v)]))[0][2]
    except:
        return 0.0

def compute_preferential_attachment(graph, u, v):
    try:
        return list(preferential_attachment(graph, [(u, v)]))[0][2]
    except:
        return 0.0

X = []
y = []



# Positive edges
for a, b in tqdm(positive_edges, desc="Creating features for positive edges"):
    a_emb = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)
    b_emb = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)

    X.append([
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        compute_jaccard_similarity(undirected_G, a, b),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])

    y.append(1)

# Negative edges
for a, b in tqdm(negative_edges, desc="Creating features for negative edges"):
    a_emb = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)
    b_emb = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)

    X.append([
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        compute_jaccard_similarity(undirected_G, a, b),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
        
    ])

    y.append(0)

# Convert to NumPy
X_np = np.array(X)
y_np = np.array(y)


model = LogisticRegression(max_iter=500, random_state=42)
model.fit(X_np, y_np)
print("Model trained on full dataset.")

X_test = []
test_edges = []

with open("test.txt", "r") as f:
    for line in f:
        a, b = map(int, line.strip().split(","))
        test_edges.append((a, b))

print(f"Loaded {len(test_edges)} edges from test.txt")

for a, b in tqdm(test_edges, desc="Extracting features from test set"):
    a_emb = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)
    b_emb = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)

    X_test.append([
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        compute_jaccard_similarity(undirected_G, a, b),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])


X_test_np = np.array(X_test)
y_test_probs = model.predict_proba(X_test_np)[:, 1]

submission_df = pd.DataFrame({
    "ID": range(len(y_test_probs)),
    "Label": y_test_probs
})
submission_df.to_csv("adar_preferential_jaccard.csv", index=False)
print("Submission file saved as 'adar_preferential_jaccard.csv'")

# Embedding stats summary
print("\n=== Embedding Source Statistics ===")
for k, v in embedding_stats.items():
    print(f"{k.replace('_', ' ').capitalize():<28}: {v}")

# Cross-validation
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
log_losses = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_np, y_np), 1):
    print(f"\nFold {fold}")
    model = LogisticRegression(max_iter=500)
    model.fit(X_np[train_idx], y_np[train_idx])
    y_pred_proba = model.predict_proba(X_np[val_idx])[:, 1]
    loss = log_loss(y_np[val_idx], y_pred_proba)
    log_losses.append(loss)
    print(f"Log Loss: {loss:.4f}")

print(f"\nAverage Log Loss over 5 folds: {np.mean(log_losses):.4f}")

Creating features for negative edges: 100%|███████████████| 1091955/1091955 [02:48<00:00, 6462.23it/s]


Model trained on full dataset.
Loaded 106692 edges from test.txt


Extracting features from test set: 100%|████████████████████| 106692/106692 [00:17<00:00, 6241.29it/s]


Submission file saved as 'adar_preferential_jaccard.csv'

=== Embedding Source Statistics ===
Abstract                    : 4408323
Neighbor fallback           : 166998
2hop fallback               : 5817
Author fallback             : 54
Global mean fallback        : 12
Zero vector                 : 0

Fold 1
Log Loss: 0.2344

Fold 2
Log Loss: 0.2342

Fold 3
Log Loss: 0.2323

Fold 4
Log Loss: 0.2336

Fold 5
Log Loss: 0.2345

Average Log Loss over 5 folds: 0.2338


**degree centrality**

In [8]:
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

# Load Word2Vec
w2v = KeyedVectors.load_word2vec_format("GoogleNews-vectors-negative300.bin", binary=True)

degree_centrality = nx.degree_centrality(undirected_G)

embedding_stats = {
    "abstract": 0,
    "neighbor_fallback": 0,
    "2hop_fallback": 0,
    "author_fallback": 0,
    "global_mean_fallback": 0,
    "zero_vector": 0
}

# Get embedding from text
def get_embedding(text):
    words = text.split()
    vectors = [w2v[w] for w in words if w in w2v]
    return np.mean(vectors, axis=0) if vectors else np.zeros(w2v.vector_size)

# Create document embeddings
doc_embeddings = {}
for pid, text in tqdm(abstracts.items(), desc="Embedding documents"):
    doc_embeddings[pid] = get_embedding(text)

# Precompute global mean
valid_embeds = [v for v in doc_embeddings.values() if np.linalg.norm(v) > 0]
global_mean = np.mean(valid_embeds, axis=0) if valid_embeds else np.zeros(w2v.vector_size)

# Author-based embedding
def get_author_based_embedding(pid, authors, doc_embeddings, embedding_dim=300):
    author_set = authors.get(pid, set())
    author_docs = [
        doc_id for doc_id, auth_set in authors.items()
        if doc_id != pid and auth_set & author_set
    ]
    valid_embeddings = [
        doc_embeddings[doc_id]
        for doc_id in author_docs
        if np.linalg.norm(doc_embeddings[doc_id]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)

# 1-hop neighbor-based embedding
def get_average_neighbor_embedding(node, graph, doc_embeddings, embedding_dim=300):
    neighbors = list(graph.neighbors(node))
    neighbor_embeds = [
        doc_embeddings[n]
        for n in neighbors if n in doc_embeddings and np.linalg.norm(doc_embeddings[n]) > 0
    ]
    return np.mean(neighbor_embeds, axis=0) if neighbor_embeds else np.zeros(embedding_dim)

# 2-hop neighbor-based embedding
def get_2hop_neighbor_embedding(node, graph, doc_embeddings, embedding_dim=300):
    two_hop_neighbors = set()
    for neighbor in graph.neighbors(node):
        two_hop_neighbors.update(graph.neighbors(neighbor))
    two_hop_neighbors.discard(node)
    valid_embeddings = [
        doc_embeddings[n]
        for n in two_hop_neighbors if n in doc_embeddings and np.linalg.norm(doc_embeddings[n]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)

# Hybrid embedding with full fallback chain
def get_hybrid_embedding(pid, graph, authors, doc_embeddings, dim=300):
    emb = doc_embeddings.get(pid)
    if emb is not None and np.linalg.norm(emb) > 0:
        embedding_stats["abstract"] += 1
        return emb

    emb = get_average_neighbor_embedding(pid, graph, doc_embeddings, dim)
    if np.linalg.norm(emb) > 0:
        embedding_stats["neighbor_fallback"] += 1
        return emb

    emb = get_2hop_neighbor_embedding(pid, graph, doc_embeddings, dim)
    if np.linalg.norm(emb) > 0:
        embedding_stats["2hop_fallback"] += 1
        return emb

    emb = get_author_based_embedding(pid, authors, doc_embeddings, dim)
    if np.linalg.norm(emb) > 0:
        embedding_stats["author_fallback"] += 1
        return emb

    if np.linalg.norm(global_mean) > 0:
        embedding_stats["global_mean_fallback"] += 1
        return global_mean

    embedding_stats["zero_vector"] += 1
    return np.zeros(dim)

embedding_dim = 300

from networkx.algorithms.link_prediction import adamic_adar_index, preferential_attachment

def compute_adamic_adar(graph, u, v):
    try:
        return list(adamic_adar_index(graph, [(u, v)]))[0][2]
    except:
        return 0.0

def compute_preferential_attachment(graph, u, v):
    try:
        return list(preferential_attachment(graph, [(u, v)]))[0][2]
    except:
        return 0.0

X = []
y = []


# Positive edges
for a, b in tqdm(positive_edges, desc="Creating features for positive edges"):
    a_emb = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)
    b_emb = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)

    X.append([
        degree_centrality.get(a, 0.0),
        degree_centrality.get(b, 0.0),
        #compute_adamic_adar(undirected_G, a, b),
        #compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(1)

# Negative edges
for a, b in tqdm(negative_edges, desc="Creating features for negative edges"):
    a_emb = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)
    b_emb = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)

    X.append([
        degree_centrality.get(a, 0.0),
        degree_centrality.get(b, 0.0),
        #compute_adamic_adar(undirected_G, a, b),
        #compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(0)

# Convert to NumPy
X_np = np.array(X)
y_np = np.array(y)


model = LogisticRegression(max_iter=500, random_state=42)
model.fit(X_np, y_np)
print("Model trained on full dataset.")

X_test = []
test_edges = []

with open("test.txt", "r") as f:
    for line in f:
        a, b = map(int, line.strip().split(","))
        test_edges.append((a, b))

print(f"Loaded {len(test_edges)} edges from test.txt")

for a, b in tqdm(test_edges, desc="Extracting features from test set"):
    a_emb = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)
    b_emb = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)

    X_test.append([
        degree_centrality.get(a, 0.0),
        degree_centrality.get(b, 0.0),
        #compute_adamic_adar(undirected_G, a, b),
        #compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])

X_test_np = np.array(X_test)
y_test_probs = model.predict_proba(X_test_np)[:, 1]

submission_df = pd.DataFrame({
    "ID": range(len(y_test_probs)),
    "Label": y_test_probs
})
submission_df.to_csv("adar_preferential_centrality.csv", index=False)
print("Submission file saved as 'adar_centrality.csv'")

# Embedding stats summary
print("\n=== Embedding Source Statistics ===")
for k, v in embedding_stats.items():
    print(f"{k.replace('_', ' ').capitalize():<28}: {v}")

# Cross-validation
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
log_losses = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_np, y_np), 1):
    print(f"\nFold {fold}")
    model = LogisticRegression(max_iter=500)
    model.fit(X_np[train_idx], y_np[train_idx])
    y_pred_proba = model.predict_proba(X_np[val_idx])[:, 1]
    loss = log_loss(y_np[val_idx], y_pred_proba)
    log_losses.append(loss)
    print(f"Log Loss: {loss:.4f}")

print(f"\nAverage Log Loss over 5 folds: {np.mean(log_losses):.4f}")

Creating features for negative edges: 100%|███████████████| 1091955/1091955 [02:30<00:00, 7273.74it/s]


Model trained on full dataset.
Loaded 106692 edges from test.txt


Extracting features from test set: 100%|████████████████████| 106692/106692 [00:14<00:00, 7373.42it/s]


Submission file saved as 'adar_centrality.csv'

=== Embedding Source Statistics ===
Abstract                    : 4408323
Neighbor fallback           : 166998
2hop fallback               : 5817
Author fallback             : 54
Global mean fallback        : 12
Zero vector                 : 0

Fold 1
Log Loss: 0.5199

Fold 2
Log Loss: 0.5213

Fold 3
Log Loss: 0.5204

Fold 4
Log Loss: 0.5206

Fold 5
Log Loss: 0.5190

Average Log Loss over 5 folds: 0.5202


**adar+preferrential+centrality**

In [10]:
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

# Load Word2Vec
w2v = KeyedVectors.load_word2vec_format("GoogleNews-vectors-negative300.bin", binary=True)

#closeness_centrality = nx.closeness_centrality(undirected_G)
with open("closeness_centrality.pkl", "rb") as f:
    closeness_centrality = pickle.load(f)


embedding_stats = {
    "abstract": 0,
    "neighbor_fallback": 0,
    "2hop_fallback": 0,
    "author_fallback": 0,
    "global_mean_fallback": 0,
    "zero_vector": 0
}

# Get embedding from text
def get_embedding(text):
    words = text.split()
    vectors = [w2v[w] for w in words if w in w2v]
    return np.mean(vectors, axis=0) if vectors else np.zeros(w2v.vector_size)

# Create document embeddings
doc_embeddings = {}
for pid, text in tqdm(abstracts.items(), desc="Embedding documents"):
    doc_embeddings[pid] = get_embedding(text)

# Precompute global mean
valid_embeds = [v for v in doc_embeddings.values() if np.linalg.norm(v) > 0]
global_mean = np.mean(valid_embeds, axis=0) if valid_embeds else np.zeros(w2v.vector_size)

# Author-based embedding
def get_author_based_embedding(pid, authors, doc_embeddings, embedding_dim=300):
    author_set = authors.get(pid, set())
    author_docs = [
        doc_id for doc_id, auth_set in authors.items()
        if doc_id != pid and auth_set & author_set
    ]
    valid_embeddings = [
        doc_embeddings[doc_id]
        for doc_id in author_docs
        if np.linalg.norm(doc_embeddings[doc_id]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)

# 1-hop neighbor-based embedding
def get_average_neighbor_embedding(node, graph, doc_embeddings, embedding_dim=300):
    neighbors = list(graph.neighbors(node))
    neighbor_embeds = [
        doc_embeddings[n]
        for n in neighbors if n in doc_embeddings and np.linalg.norm(doc_embeddings[n]) > 0
    ]
    return np.mean(neighbor_embeds, axis=0) if neighbor_embeds else np.zeros(embedding_dim)

# 2-hop neighbor-based embedding
def get_2hop_neighbor_embedding(node, graph, doc_embeddings, embedding_dim=300):
    two_hop_neighbors = set()
    for neighbor in graph.neighbors(node):
        two_hop_neighbors.update(graph.neighbors(neighbor))
    two_hop_neighbors.discard(node)
    valid_embeddings = [
        doc_embeddings[n]
        for n in two_hop_neighbors if n in doc_embeddings and np.linalg.norm(doc_embeddings[n]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)

# Hybrid embedding with full fallback chain
def get_hybrid_embedding(pid, graph, authors, doc_embeddings, dim=300):
    emb = doc_embeddings.get(pid)
    if emb is not None and np.linalg.norm(emb) > 0:
        embedding_stats["abstract"] += 1
        return emb

    emb = get_average_neighbor_embedding(pid, graph, doc_embeddings, dim)
    if np.linalg.norm(emb) > 0:
        embedding_stats["neighbor_fallback"] += 1
        return emb

    emb = get_2hop_neighbor_embedding(pid, graph, doc_embeddings, dim)
    if np.linalg.norm(emb) > 0:
        embedding_stats["2hop_fallback"] += 1
        return emb

    emb = get_author_based_embedding(pid, authors, doc_embeddings, dim)
    if np.linalg.norm(emb) > 0:
        embedding_stats["author_fallback"] += 1
        return emb

    if np.linalg.norm(global_mean) > 0:
        embedding_stats["global_mean_fallback"] += 1
        return global_mean

    embedding_stats["zero_vector"] += 1
    return np.zeros(dim)

embedding_dim = 300

from networkx.algorithms.link_prediction import adamic_adar_index, preferential_attachment

def compute_adamic_adar(graph, u, v):
    try:
        return list(adamic_adar_index(graph, [(u, v)]))[0][2]
    except:
        return 0.0

def compute_preferential_attachment(graph, u, v):
    try:
        return list(preferential_attachment(graph, [(u, v)]))[0][2]
    except:
        return 0.0

X = []
y = []


# Positive edges
for a, b in tqdm(positive_edges, desc="Creating features for positive edges"):
    a_emb = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)
    b_emb = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)

    X.append([
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(1)

# Negative edges
for a, b in tqdm(negative_edges, desc="Creating features for negative edges"):
    a_emb = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)
    b_emb = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)

    X.append([
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(0)

# Convert to NumPy
X_np = np.array(X)
y_np = np.array(y)


model = LogisticRegression(max_iter=500, random_state=42)
model.fit(X_np, y_np)
print("Model trained on full dataset.")

X_test = []
test_edges = []

with open("test.txt", "r") as f:
    for line in f:
        a, b = map(int, line.strip().split(","))
        test_edges.append((a, b))

print(f"Loaded {len(test_edges)} edges from test.txt")

for a, b in tqdm(test_edges, desc="Extracting features from test set"):
    a_emb = get_hybrid_embedding(a, undirected_G, authors, doc_embeddings, embedding_dim)
    b_emb = get_hybrid_embedding(b, undirected_G, authors, doc_embeddings, embedding_dim)

    X_test.append([
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])

X_test_np = np.array(X_test)
y_test_probs = model.predict_proba(X_test_np)[:, 1]

submission_df = pd.DataFrame({
    "ID": range(len(y_test_probs)),
    "Label": y_test_probs
})
submission_df.to_csv("adar_preferential_centrality.csv", index=False)
print("Submission file saved as 'adar_centrality.csv'")

# Embedding stats summary
print("\n=== Embedding Source Statistics ===")
for k, v in embedding_stats.items():
    print(f"{k.replace('_', ' ').capitalize():<28}: {v}")

# Cross-validation
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
log_losses = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_np, y_np), 1):
    print(f"\nFold {fold}")
    model = LogisticRegression(max_iter=500)
    model.fit(X_np[train_idx], y_np[train_idx])
    y_pred_proba = model.predict_proba(X_np[val_idx])[:, 1]
    loss = log_loss(y_np[val_idx], y_pred_proba)
    log_losses.append(loss)
    print(f"Log Loss: {loss:.4f}")

print(f"\nAverage Log Loss over 5 folds: {np.mean(log_losses):.4f}")

Creating features for negative edges: 100%|███████████████| 1091955/1091955 [02:40<00:00, 6800.21it/s]


Model trained on full dataset.
Loaded 106692 edges from test.txt


Extracting features from test set: 100%|████████████████████| 106692/106692 [00:16<00:00, 6665.79it/s]


Submission file saved as 'adar_centrality.csv'

=== Embedding Source Statistics ===
Abstract                    : 4408323
Neighbor fallback           : 166998
2hop fallback               : 5817
Author fallback             : 54
Global mean fallback        : 12
Zero vector                 : 0

Fold 1
Log Loss: 0.2277

Fold 2
Log Loss: 0.2276

Fold 3
Log Loss: 0.2257

Fold 4
Log Loss: 0.2270

Fold 5
Log Loss: 0.2275

Average Log Loss over 5 folds: 0.2271


**PCA**

In [4]:
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.decomposition import PCA



# Load Word2Vec
w2v = KeyedVectors.load_word2vec_format("GoogleNews-vectors-negative300.bin", binary=True)

embedding_dim = 300

#closeness_centrality = nx.closeness_centrality(undirected_G)
with open("closeness_centrality.pkl", "rb") as f:
    closeness_centrality = pickle.load(f)

# Get embedding from text
def get_embedding(text):
    words = text.split()
    vectors = [w2v[w] for w in words if w in w2v]
    return np.mean(vectors, axis=0) if vectors else np.zeros(w2v.vector_size)

raw_embeddings = {}
for pid, text in tqdm(abstracts.items(), desc="Generating raw embeddings"):
    emb = get_embedding(text)
    raw_embeddings[pid] = emb


valid_embeds = [v for v in raw_embeddings.values() if np.linalg.norm(v) > 0]
global_mean = np.mean(valid_embeds, axis=0) if valid_embeds else np.zeros(w2v.vector_size) 

    
def get_author_based_embedding(pid, authors, base_embeddings):
    author_set = authors.get(pid, set())
    author_docs = [
        doc_id for doc_id, auth_set in authors.items()
        if doc_id != pid and auth_set & author_set
    ]
    valid_embeddings = [
        base_embeddings[doc_id]
        for doc_id in author_docs
        if np.linalg.norm(base_embeddings[doc_id]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)

def get_average_neighbor_embedding(node, graph, base_embeddings):
    neighbors = list(graph.neighbors(node))
    neighbor_embeds = [
        base_embeddings[n]
        for n in neighbors if np.linalg.norm(base_embeddings[n]) > 0
    ]
    return np.mean(neighbor_embeds, axis=0) if neighbor_embeds else np.zeros(embedding_dim)

def get_2hop_neighbor_embedding(node, graph, base_embeddings):
    two_hop_neighbors = set()
    for neighbor in graph.neighbors(node):
        two_hop_neighbors.update(graph.neighbors(neighbor))
    two_hop_neighbors.discard(node)
    valid_embeddings = [
        base_embeddings[n]
        for n in two_hop_neighbors if np.linalg.norm(base_embeddings[n]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)

embedding_stats = {
    "abstract": 0,
    "neighbor_fallback": 0,
    "2hop_fallback": 0,
    "author_fallback": 0,
    "global_mean_fallback": 0,
    "zero_vector": 0
}

def get_hybrid_embedding(pid, graph, authors, base_embeddings):
    emb = base_embeddings[pid]
    if np.linalg.norm(emb) > 0:
        embedding_stats["abstract"] += 1
        return emb

    emb = get_average_neighbor_embedding(pid, graph, base_embeddings)
    if np.linalg.norm(emb) > 0:
        embedding_stats["neighbor_fallback"] += 1
        return emb

    emb = get_2hop_neighbor_embedding(pid, graph, base_embeddings)
    if np.linalg.norm(emb) > 0:
        embedding_stats["2hop_fallback"] += 1
        return emb

    emb = get_author_based_embedding(pid, authors, base_embeddings)
    if np.linalg.norm(emb) > 0:
        embedding_stats["author_fallback"] += 1
        return emb

    if np.linalg.norm(global_mean) > 0:
        embedding_stats["global_mean_fallback"] += 1
        return global_mean

    embedding_stats["zero_vector"] += 1
    return np.zeros(embedding_dim)



doc_embeddings = {}
for pid in tqdm(abstracts.keys(), desc="Constructing hybrid embeddings"):
    final_emb = get_hybrid_embedding(pid, undirected_G, authors, raw_embeddings)
    doc_embeddings[pid] = final_emb


print("\n=== Embedding Source Statistics ===")
for k, v in embedding_stats.items():
    print(f"{k.replace('_', ' ').capitalize():<28}: {v}")


embeddings_matrix = np.array(list(doc_embeddings.values()))


pca = PCA(n_components=300)  


reduced_embeddings = pca.fit_transform(embeddings_matrix)


from tqdm import tqdm


reduced_doc_embeddings = {
    pid: reduced_embeddings[idx] for idx, pid in tqdm(enumerate(doc_embeddings.keys()), desc="Creating reduced embeddings")
}



embedding_dim = 300

from networkx.algorithms.link_prediction import adamic_adar_index, preferential_attachment

def compute_adamic_adar(graph, u, v):
    try:
        return list(adamic_adar_index(graph, [(u, v)]))[0][2]
    except:
        return 0.0

def compute_preferential_attachment(graph, u, v):
    try:
        return list(preferential_attachment(graph, [(u, v)]))[0][2]
    except:
        return 0.0

X = []
y = []


# Positive edges
for a, b in tqdm(positive_edges, desc="Creating features for positive edges"):
    a_emb = reduced_doc_embeddings.get(a, np.zeros(embedding_dim))  # Default to zero vector if not found
    b_emb = reduced_doc_embeddings.get(b, np.zeros(embedding_dim)) 

    X.append([
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(1)

# Negative edges
for a, b in tqdm(negative_edges, desc="Creating features for negative edges"):
    a_emb = reduced_doc_embeddings.get(a, np.zeros(embedding_dim))  # Default to zero vector if not found
    b_emb = reduced_doc_embeddings.get(b, np.zeros(embedding_dim)) 

    X.append([
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(0)

# Convert to NumPy
X_np = np.array(X)
y_np = np.array(y)


model = LogisticRegression(max_iter=500, random_state=42)
model.fit(X_np, y_np)
print("Model trained on full dataset.")

X_test = []
test_edges = []

with open("test.txt", "r") as f:
    for line in f:
        a, b = map(int, line.strip().split(","))
        test_edges.append((a, b))

print(f"Loaded {len(test_edges)} edges from test.txt")

for a, b in tqdm(test_edges, desc="Extracting features from test set"):
    a_emb = reduced_doc_embeddings.get(a, np.zeros(embedding_dim))  
    b_emb = reduced_doc_embeddings.get(b, np.zeros(embedding_dim)) 

    X_test.append([
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])

X_test_np = np.array(X_test)
y_test_probs = model.predict_proba(X_test_np)[:, 1]

submission_df = pd.DataFrame({
    "ID": range(len(y_test_probs)),
    "Label": y_test_probs
})
submission_df.to_csv("pca_fixed.csv", index=False)
print("Submission file saved as 'pca_fixed.csv'")



# Cross-validation
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
log_losses = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_np, y_np), 1):
    print(f"\nFold {fold}")
    model = LogisticRegression(max_iter=500)
    model.fit(X_np[train_idx], y_np[train_idx])
    y_pred_proba = model.predict_proba(X_np[val_idx])[:, 1]
    loss = log_loss(y_np[val_idx], y_pred_proba)
    log_losses.append(loss)
    print(f"Log Loss: {loss:.4f}")

print(f"\nAverage Log Loss over 5 folds: {np.mean(log_losses):.4f}")

Constructing hybrid embeddings: 100%|█████████████████████| 138499/138499 [00:01<00:00, 101524.10it/s]



=== Embedding Source Statistics ===
Abstract                    : 131249
Neighbor fallback           : 6919
2hop fallback               : 327
Author fallback             : 3
Global mean fallback        : 1
Zero vector                 : 0


Creating reduced embeddings: 138499it [00:00, 1077251.71it/s]
Creating features for negative edges: 100%|███████████████| 1091955/1091955 [02:45<00:00, 6597.37it/s]


Model trained on full dataset.
Loaded 106692 edges from test.txt


Extracting features from test set: 100%|████████████████████| 106692/106692 [00:17<00:00, 6155.40it/s]


Submission file saved as 'pca_fixed.csv'

Fold 1
Log Loss: 0.1831

Fold 2
Log Loss: 0.1831

Fold 3
Log Loss: 0.1818

Fold 4
Log Loss: 0.1821

Fold 5
Log Loss: 0.1832

Average Log Loss over 5 folds: 0.1826


In [4]:
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.decomposition import PCA



# Load Word2Vec
w2v = KeyedVectors.load_word2vec_format("GoogleNews-vectors-negative300.bin", binary=True)

embedding_dim = 300

#closeness_centrality = nx.closeness_centrality(undirected_G)
with open("closeness_centrality.pkl", "rb") as f:
    closeness_centrality = pickle.load(f)

# Get embedding from text
def get_embedding(text):
    words = text.split()
    vectors = [w2v[w] for w in words if w in w2v]
    return np.mean(vectors, axis=0) if vectors else np.zeros(w2v.vector_size)

raw_embeddings = {}
for pid, text in tqdm(abstracts.items(), desc="Generating raw embeddings"):
    emb = get_embedding(text)
    raw_embeddings[pid] = emb


valid_embeds = [v for v in raw_embeddings.values() if np.linalg.norm(v) > 0]
global_mean = np.mean(valid_embeds, axis=0) if valid_embeds else np.zeros(w2v.vector_size) 

    
def get_author_based_embedding(pid, authors, base_embeddings):
    author_set = authors.get(pid, set())
    author_docs = [
        doc_id for doc_id, auth_set in authors.items()
        if doc_id != pid and auth_set & author_set
    ]
    valid_embeddings = [
        base_embeddings[doc_id]
        for doc_id in author_docs
        if np.linalg.norm(base_embeddings[doc_id]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)

def get_average_neighbor_embedding(node, graph, base_embeddings):
    neighbors = list(graph.neighbors(node))
    neighbor_embeds = [
        base_embeddings[n]
        for n in neighbors if np.linalg.norm(base_embeddings[n]) > 0
    ]
    return np.mean(neighbor_embeds, axis=0) if neighbor_embeds else np.zeros(embedding_dim)

def get_2hop_neighbor_embedding(node, graph, base_embeddings):
    two_hop_neighbors = set()
    for neighbor in graph.neighbors(node):
        two_hop_neighbors.update(graph.neighbors(neighbor))
    two_hop_neighbors.discard(node)
    valid_embeddings = [
        base_embeddings[n]
        for n in two_hop_neighbors if np.linalg.norm(base_embeddings[n]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)

embedding_stats = {
    "abstract": 0,
    "neighbor_fallback": 0,
    "2hop_fallback": 0,
    "author_fallback": 0,
    "global_mean_fallback": 0,
    "zero_vector": 0
}

def get_hybrid_embedding(pid, graph, authors, base_embeddings):
    emb = base_embeddings[pid]
    if np.linalg.norm(emb) > 0:
        embedding_stats["abstract"] += 1
        return emb

    emb = get_average_neighbor_embedding(pid, graph, base_embeddings)
    if np.linalg.norm(emb) > 0:
        embedding_stats["neighbor_fallback"] += 1
        return emb

    emb = get_2hop_neighbor_embedding(pid, graph, base_embeddings)
    if np.linalg.norm(emb) > 0:
        embedding_stats["2hop_fallback"] += 1
        return emb

    emb = get_author_based_embedding(pid, authors, base_embeddings)
    if np.linalg.norm(emb) > 0:
        embedding_stats["author_fallback"] += 1
        return emb

    if np.linalg.norm(global_mean) > 0:
        embedding_stats["global_mean_fallback"] += 1
        return global_mean

    embedding_stats["zero_vector"] += 1
    return np.zeros(embedding_dim)


def compute_personalized_pagerank(graph, source, target, alpha=0.85):
    try:
        ppr = nx.pagerank(graph, alpha=alpha, personalization={source: 1})
        return ppr.get(target, 0.0)
    except Exception as e:
        return 0.0




doc_embeddings = {}
for pid in tqdm(abstracts.keys(), desc="Constructing hybrid embeddings"):
    final_emb = get_hybrid_embedding(pid, undirected_G, authors, raw_embeddings)
    doc_embeddings[pid] = final_emb


print("\n=== Embedding Source Statistics ===")
for k, v in embedding_stats.items():
    print(f"{k.replace('_', ' ').capitalize():<28}: {v}")


embeddings_matrix = np.array(list(doc_embeddings.values()))


pca = PCA(n_components=300)  


reduced_embeddings = pca.fit_transform(embeddings_matrix)


from tqdm import tqdm


reduced_doc_embeddings = {
    pid: reduced_embeddings[idx] for idx, pid in tqdm(enumerate(doc_embeddings.keys()), desc="Creating reduced embeddings")
}



from sklearn.neighbors import NearestNeighbors

# Prepare the matrix
doc_ids = list(reduced_doc_embeddings.keys())
embedding_matrix = np.array([reduced_doc_embeddings[pid] for pid in doc_ids])

# Fit KNN model
knn_model = NearestNeighbors(n_neighbors=6, metric='cosine')  # k+1 because first is self
knn_model.fit(embedding_matrix)

# Map doc_id to index in embedding_matrix
pid_to_index = {pid: idx for idx, pid in enumerate(doc_ids)}




# Precompute KNNs
knn_dict = {}
for pid in tqdm(doc_ids, desc="Precomputing KNNs"):
    idx = pid_to_index[pid]
    distances, indices = knn_model.kneighbors([embedding_matrix[idx]])
    neighbors = [doc_ids[i] for i in indices[0] if doc_ids[i] != pid]
    knn_dict[pid] = neighbors

embedding_dim = 300

def knn_similarity(target_id, knn_source, doc_embeddings):
    similarities = []
    for neighbor in knn_source:
        if neighbor in doc_embeddings:
            sim = cosine_similarity([doc_embeddings[target_id]], [doc_embeddings[neighbor]])[0][0]
            similarities.append(sim)
    return np.mean(similarities) if similarities else 0.0


from networkx.algorithms.link_prediction import adamic_adar_index, preferential_attachment

def compute_adamic_adar(graph, u, v):
    try:
        return list(adamic_adar_index(graph, [(u, v)]))[0][2]
    except:
        return 0.0

def compute_preferential_attachment(graph, u, v):
    try:
        return list(preferential_attachment(graph, [(u, v)]))[0][2]
    except:
        return 0.0

X = []
y = []


# Positive edges
for a, b in tqdm(positive_edges, desc="Creating features for positive edges"):
    a_emb = reduced_doc_embeddings.get(a, np.zeros(embedding_dim))  # Default to zero vector if not found
    b_emb = reduced_doc_embeddings.get(b, np.zeros(embedding_dim)) 


    a_knn = knn_dict.get(a, [])
    b_knn = knn_dict.get(b, [])
    
    knn_sim_a_to_b = knn_similarity(b, a_knn, reduced_doc_embeddings)
    knn_sim_b_to_a = knn_similarity(a, b_knn, reduced_doc_embeddings)

    X.append([
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set())),
        knn_sim_a_to_b,
        knn_sim_b_to_a
    ])

    y.append(1)

# Negative edges
for a, b in tqdm(negative_edges, desc="Creating features for negative edges"):
    a_emb = reduced_doc_embeddings.get(a, np.zeros(embedding_dim))  # Default to zero vector if not found
    b_emb = reduced_doc_embeddings.get(b, np.zeros(embedding_dim)) 

    a_knn = knn_dict.get(a, [])
    b_knn = knn_dict.get(b, [])
    
    knn_sim_a_to_b = knn_similarity(b, a_knn, reduced_doc_embeddings)
    knn_sim_b_to_a = knn_similarity(a, b_knn, reduced_doc_embeddings)
    
    X.append([
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set())),
        knn_sim_a_to_b,
        knn_sim_b_to_a
    ])
    
    y.append(0)

# Convert to NumPy
X_np = np.array(X)
y_np = np.array(y)


model = LogisticRegression(max_iter=500, random_state=42)
model.fit(X_np, y_np)
print("Model trained on full dataset.")

X_test = []
test_edges = []

with open("test.txt", "r") as f:
    for line in f:
        a, b = map(int, line.strip().split(","))
        test_edges.append((a, b))

print(f"Loaded {len(test_edges)} edges from test.txt")

for a, b in tqdm(test_edges, desc="Extracting features from test set"):
    a_emb = reduced_doc_embeddings.get(a, np.zeros(embedding_dim))  
    b_emb = reduced_doc_embeddings.get(b, np.zeros(embedding_dim)) 

    a_knn = knn_dict.get(a, [])
    b_knn = knn_dict.get(b, [])
    
    knn_sim_a_to_b = knn_similarity(b, a_knn, reduced_doc_embeddings)
    knn_sim_b_to_a = knn_similarity(a, b_knn, reduced_doc_embeddings)

    X_test.append([
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set())),
        knn_sim_a_to_b,
        knn_sim_b_to_a
    ])

X_test_np = np.array(X_test)
y_test_probs = model.predict_proba(X_test_np)[:, 1]

submission_df = pd.DataFrame({
    "ID": range(len(y_test_probs)),
    "Label": y_test_probs
})
submission_df.to_csv("knn.csv", index=False)
print("Submission file saved as 'knn.csv'")



# Cross-validation
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
log_losses = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_np, y_np), 1):
    print(f"\nFold {fold}")
    model = LogisticRegression(max_iter=500)
    model.fit(X_np[train_idx], y_np[train_idx])
    y_pred_proba = model.predict_proba(X_np[val_idx])[:, 1]
    loss = log_loss(y_np[val_idx], y_pred_proba)
    log_losses.append(loss)
    print(f"Log Loss: {loss:.4f}")

print(f"\nAverage Log Loss over 5 folds: {np.mean(log_losses):.4f}")

Constructing hybrid embeddings: 100%|█████████████████████| 138499/138499 [00:01<00:00, 136262.56it/s]



=== Embedding Source Statistics ===
Abstract                    : 131249
Neighbor fallback           : 6919
2hop fallback               : 327
Author fallback             : 3
Global mean fallback        : 1
Zero vector                 : 0


Creating reduced embeddings: 138499it [00:00, 1133112.09it/s]
Creating features for negative edges: 100%|████████████████| 1091955/1091955 [28:18<00:00, 642.98it/s]


Model trained on full dataset.
Loaded 106692 edges from test.txt


Extracting features from test set: 100%|█████████████████████| 106692/106692 [02:39<00:00, 667.05it/s]


Submission file saved as 'knn.csv'

Fold 1
Log Loss: 0.1815

Fold 2
Log Loss: 0.1815

Fold 3
Log Loss: 0.1802

Fold 4
Log Loss: 0.1805

Fold 5
Log Loss: 0.1815

Average Log Loss over 5 folds: 0.1810


**clustering coefficient**

In [4]:
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.decomposition import PCA
from networkx.algorithms.cluster import clustering
from sklearn.preprocessing import StandardScaler
import pickle



edges = pd.read_csv("edgelist.txt", header=None, names=["source", "target"])
positive_edges = list(edges.itertuples(index=False, name=None))
G = nx.DiGraph()
G.add_edges_from(edges.itertuples(index=False, name=None))
print(f"Graph loaded with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

# Load negative edges from file
negative_edges = set()
with open("negative_edgelist2.txt", "r") as f:
    for line in f:
        u, v = map(int, line.strip().split(','))
        negative_edges.add((u, v))

print(f"Loaded {len(negative_edges)} negative edges from file.")

undirected_G = G.to_undirected()
print("loading abstracts.txt...")
abstracts = {}
with open("abstracts.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("|--|")
        abstracts[int(parts[0])] = parts[1]
        
print("loading authors.txt...")
authors = {}
with open("authors.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("|--|")
        authors[int(parts[0])] = set(parts[1].split(","))
        


# Load Word2Vec
w2v = KeyedVectors.load_word2vec_format("GoogleNews-vectors-negative300.bin", binary=True)

embedding_dim = 300



# Get embedding from text
def get_embedding(text):
    words = text.split()
    vectors = [w2v[w] for w in words if w in w2v]
    return np.mean(vectors, axis=0) if vectors else np.zeros(w2v.vector_size)

raw_embeddings = {}
for pid, text in tqdm(abstracts.items(), desc="Generating raw embeddings"):
    emb = get_embedding(text)
    raw_embeddings[pid] = emb


valid_embeds = [v for v in raw_embeddings.values() if np.linalg.norm(v) > 0]
global_mean = np.mean(valid_embeds, axis=0) if valid_embeds else np.zeros(w2v.vector_size) 

    
def get_author_based_embedding(pid, authors, base_embeddings):
    author_set = authors.get(pid, set())
    author_docs = [
        doc_id for doc_id, auth_set in authors.items()
        if doc_id != pid and auth_set & author_set
    ]
    valid_embeddings = [
        base_embeddings[doc_id]
        for doc_id in author_docs
        if np.linalg.norm(base_embeddings[doc_id]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)

def get_average_neighbor_embedding(node, graph, base_embeddings):
    neighbors = list(graph.neighbors(node))
    neighbor_embeds = [
        base_embeddings[n]
        for n in neighbors if np.linalg.norm(base_embeddings[n]) > 0
    ]
    return np.mean(neighbor_embeds, axis=0) if neighbor_embeds else np.zeros(embedding_dim)

def get_2hop_neighbor_embedding(node, graph, base_embeddings):
    two_hop_neighbors = set()
    for neighbor in graph.neighbors(node):
        two_hop_neighbors.update(graph.neighbors(neighbor))
    two_hop_neighbors.discard(node)
    valid_embeddings = [
        base_embeddings[n]
        for n in two_hop_neighbors if np.linalg.norm(base_embeddings[n]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)

embedding_stats = {
    "abstract": 0,
    "neighbor_fallback": 0,
    "2hop_fallback": 0,
    "author_fallback": 0,
    "global_mean_fallback": 0,
    "zero_vector": 0
}

def get_hybrid_embedding(pid, graph, authors, base_embeddings):
    emb = base_embeddings[pid]
    if np.linalg.norm(emb) > 0:
        embedding_stats["abstract"] += 1
        return emb

    emb = get_average_neighbor_embedding(pid, graph, base_embeddings)
    if np.linalg.norm(emb) > 0:
        embedding_stats["neighbor_fallback"] += 1
        return emb

    emb = get_2hop_neighbor_embedding(pid, graph, base_embeddings)
    if np.linalg.norm(emb) > 0:
        embedding_stats["2hop_fallback"] += 1
        return emb

    emb = get_author_based_embedding(pid, authors, base_embeddings)
    if np.linalg.norm(emb) > 0:
        embedding_stats["author_fallback"] += 1
        return emb

    if np.linalg.norm(global_mean) > 0:
        embedding_stats["global_mean_fallback"] += 1
        return global_mean

    embedding_stats["zero_vector"] += 1
    return np.zeros(embedding_dim)




doc_embeddings = {}
for pid in tqdm(abstracts.keys(), desc="Constructing hybrid embeddings"):
    final_emb = get_hybrid_embedding(pid, undirected_G, authors, raw_embeddings)
    doc_embeddings[pid] = final_emb


print("\n=== Embedding Source Statistics ===")
for k, v in embedding_stats.items():
    print(f"{k.replace('_', ' ').capitalize():<28}: {v}")


embeddings_matrix = np.array(list(doc_embeddings.values()))

scaler = StandardScaler()
embeddings_matrix = scaler.fit_transform(embeddings_matrix)

pca = PCA(n_components=300)  


reduced_embeddings = pca.fit_transform(embeddings_matrix)




reduced_doc_embeddings = {
    pid: reduced_embeddings[idx] for idx, pid in enumerate(doc_embeddings.keys())
}

import numpy as np

np.save("reduced_w2v_embeddings.npy", reduced_embeddings)



embedding_dim = 300

#closeness_centrality = nx.closeness_centrality(undirected_G)
with open("closeness_centrality.pkl", "rb") as f:
    closeness_centrality = pickle.load(f)

from networkx.algorithms.link_prediction import adamic_adar_index, preferential_attachment

def compute_adamic_adar(graph, u, v):
    try:
        return list(adamic_adar_index(graph, [(u, v)]))[0][2]
    except:
        return 0.0

def compute_preferential_attachment(graph, u, v):
    try:
        return list(preferential_attachment(graph, [(u, v)]))[0][2]
    except:
        return 0.0

import networkx as nx
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression


# Assuming undirected_G is your undirected version of the graph
clustering_coeffs = clustering(undirected_G)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()


X = []
y = []


# Positive edges
for a, b in tqdm(positive_edges, desc="Creating features for positive edges"):
    a_emb = reduced_doc_embeddings.get(a, np.zeros(embedding_dim))  # Default to zero vector if not found
    b_emb = reduced_doc_embeddings.get(b, np.zeros(embedding_dim)) 

    if a_emb.shape != b_emb.shape:
        print(f"Shape mismatch: a_emb {a_emb.shape}, b_emb {b_emb.shape}")
        continue  # Or handle appropriately
        
    X.append([
        clustering_coeffs.get(a, 0.0),  # New
        clustering_coeffs.get(b, 0.0) ,
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(1)

# Negative edges
for a, b in tqdm(negative_edges, desc="Creating features for negative edges"):
    a_emb = reduced_doc_embeddings.get(a, np.zeros(embedding_dim))  # Default to zero vector if not found
    b_emb = reduced_doc_embeddings.get(b, np.zeros(embedding_dim)) 

    if a_emb.shape != b_emb.shape:
        print(f"Shape mismatch: a_emb {a_emb.shape}, b_emb {b_emb.shape}")
        continue  # Or handle appropriately
        
    X.append([
        clustering_coeffs.get(a, 0.0),  # New
        clustering_coeffs.get(b, 0.0) ,
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(0)

# Convert to NumPy
X_np = np.array(X)
y_np = np.array(y)

scaler.fit(X_np)               # Fit on training data
X_np_scaled = scaler.transform(X_np)  # Transform training data

model = LogisticRegression(max_iter=1500, random_state=42)
model.fit(X_np_scaled, y_np)
print("Model trained on full dataset.")

X_test = []
test_edges = []

with open("test.txt", "r") as f:
    for line in f:
        a, b = map(int, line.strip().split(","))
        test_edges.append((a, b))

print(f"Loaded {len(test_edges)} edges from test.txt")

for a, b in tqdm(test_edges, desc="Extracting features from test set"):
    a_emb = reduced_doc_embeddings.get(a, np.zeros(embedding_dim))  
    b_emb = reduced_doc_embeddings.get(b, np.zeros(embedding_dim)) 
    if a_emb.shape != b_emb.shape:
        print(f"Shape mismatch: a_emb {a_emb.shape}, b_emb {b_emb.shape}")
        continue  # Or handle appropriately


    X_test.append([
        clustering_coeffs.get(a, 0.0),  # New
        clustering_coeffs.get(b, 0.0) ,
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_emb], [b_emb])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])

X_test_np = np.array(X_test)
X_test_scaled = scaler.transform(X_test_np)
y_test_probs = model.predict_proba(X_test_scaled)[:, 1]
#y_test_probs = model.predict_proba(X_test_np)[:, 1]

submission_df = pd.DataFrame({
    "ID": range(len(y_test_probs)),
    "Label": y_test_probs
})
submission_df.to_csv("scaled.csv", index=False)
print("Submission file saved as 'scaled.csv'")



#Convert to NumPy
X_np = np.array(X)
y_np = np.array(y)

# Set up 5-fold stratified cross-validation
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
log_losses = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_np, y_np), 1):
    X_train, X_val = X_np[train_idx], X_np[val_idx]
    y_train, y_val = y_np[train_idx], y_np[val_idx]

    # Fit scaler only on training data
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    # Train logistic regression
    model = LogisticRegression(max_iter=1500, random_state=42)
    model.fit(X_train_scaled, y_train)

    # Predict probabilities for log loss
    y_proba = model.predict_proba(X_val_scaled)
    loss = log_loss(y_val, y_proba)
    log_losses.append(loss)

    print(f"Fold {fold}: Log Loss = {loss:.4f}")

# Report average NLL
mean_log_loss = np.mean(log_losses)
print(f"\nAverage Negative Log-Likelihood (Log Loss): {mean_log_loss:.4f}")

Graph loaded with 138499 nodes and 1091955 edges.
Loaded 1091955 negative edges from file.
loading abstracts.txt...
loading authors.txt...


Generating raw embeddings: 100%|██████| 138499/138499 [00:18<00:00, 7362.12it/s]
Constructing hybrid embeddings: 100%|█| 138499/138499 [00:02<00:00, 68598.77it/s



=== Embedding Source Statistics ===
Abstract                    : 131249
Neighbor fallback           : 6919
2hop fallback               : 327
Author fallback             : 3
Global mean fallback        : 1
Zero vector                 : 0


Creating features for positive edges: 100%|█| 1091955/1091955 [02:47<00:00, 6534
Creating features for negative edges: 100%|█| 1091955/1091955 [02:50<00:00, 6391


Model trained on full dataset.
Loaded 106692 edges from test.txt


Extracting features from test set: 100%|█| 106692/106692 [00:15<00:00, 6731.47it


Submission file saved as 'scaled.csv'
Fold 1: Log Loss = 0.1653
Fold 2: Log Loss = 0.1641
Fold 3: Log Loss = 0.1652
Fold 4: Log Loss = 0.1658
Fold 5: Log Loss = 0.1649

Average Negative Log-Likelihood (Log Loss): 0.1651


**sci Bert**

In [ ]:
from transformers import BertTokenizer, BertModel
import torch
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.decomposition import PCA
from networkx.algorithms.cluster import clustering
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm


# Load SciBERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('allenai/scibert_scivocab_uncased')
model = BertModel.from_pretrained('allenai/scibert_scivocab_uncased')


# Function to get SciBERT embedding for a given text
def get_sciBERT_embedding(text):
    # Tokenize the text and get input IDs
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=512)
    
    # Get the embeddings from SciBERT (taking the last hidden state)
    with torch.no_grad():
        outputs = model(**inputs)
        # Use the [CLS] token embedding (index 0) for sentence-level representation
        embedding = outputs.last_hidden_state[:, 0, :].squeeze().numpy()

    return embedding
    
import numpy as np

# Parameters
save_every = 30_000
output_dir = "scibert_chunks"
os.makedirs(output_dir, exist_ok=True)

# Load previously saved embeddings
embeddings = {}
processed_pids = set()

# Find and load all chunk files
chunk_files = sorted([f for f in os.listdir(output_dir) if f.startswith("scibert_chunk_") and f.endswith(".pkl")])
for chunk_file in chunk_files:
    with open(os.path.join(output_dir, chunk_file), "rb") as f:
        chunk_data = pickle.load(f)
        embeddings.update(chunk_data)
        processed_pids.update(chunk_data.keys())
print(f"Loaded {len(embeddings)} embeddings from previous chunks.")

# Resume where you left off
buffer = {}
buffer_index = len(chunk_files)  # Resume chunk numbering

for pid, text in tqdm(abstracts.items(), desc="Generating SciBERT embeddings"):
    if pid in processed_pids:
        continue  # Skip if already processed

    emb = get_sciBERT_embedding(text)
    buffer[pid] = emb
    processed_pids.add(pid)

    if len(buffer) >= save_every:
        chunk_path = os.path.join(output_dir, f"scibert_chunk_{buffer_index}.pkl")
        with open(chunk_path, "wb") as f:
            pickle.dump(buffer, f)
        print(f"Saved {len(buffer)} embeddings to {chunk_path}")
        buffer = {}  # Clear buffer
        buffer_index += 1

# Save any remaining embeddings
if buffer:
    chunk_path = os.path.join(output_dir, f"scibert_chunk_{buffer_index}.pkl")
    with open(chunk_path, "wb") as f:
        pickle.dump(buffer, f)




embedding_matrix = np.array(list(embeddings.values()))

In [1]:
import pickle
import os
from transformers import BertTokenizer, BertModel
import torch
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.decomposition import PCA
from networkx.algorithms.cluster import clustering
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm


edges = pd.read_csv("edgelist.txt", header=None, names=["source", "target"])
positive_edges = list(edges.itertuples(index=False, name=None))
G = nx.DiGraph()
G.add_edges_from(edges.itertuples(index=False, name=None))
print(f"Graph loaded with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

# Load negative edges from file
negative_edges = set()
with open("negative_edgelist.txt", "r") as f:
    for line in f:
        u, v = map(int, line.strip().split(','))
        negative_edges.add((u, v))

print(f"Loaded {len(negative_edges)} negative edges from file.")

undirected_G = G.to_undirected()
print("loading abstracts.txt...")
abstracts = {}
with open("abstracts.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("|--|")
        abstracts[int(parts[0])] = parts[1]
        
print("loading authors.txt...")
authors = {}
with open("authors.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("|--|")
        authors[int(parts[0])] = set(parts[1].split(","))

# Path to your saved SciBERT chunks
output_dir = "scibert_chunks"

# Initialize embedding dictionary
embeddings = {}

# Load all chunks
chunk_files = sorted([
    f for f in os.listdir(output_dir) 
    if f.startswith("scibert_chunk_") and f.endswith(".pkl")
])

for chunk_file in tqdm(chunk_files, desc="Loading SciBERT chunks"):
    chunk_path = os.path.join(output_dir, chunk_file)
    with open(chunk_path, "rb") as f:
        chunk_data = pickle.load(f)
        embeddings.update(chunk_data)

print(f"Loaded {len(embeddings)} total SciBERT embeddings.")


paper_ids = list(abstracts.keys())  # Must be in the same order as embeddings were created



target_embedding = embeddings.get(76)
if target_embedding is None:
    raise ValueError("Embedding for paper 76 not found.")

import numpy as np

# Get the dimensionality of embeddings
embedding_dim = len(target_embedding)
zero_vector = np.zeros(embedding_dim)

# Loop through and zero out matching embeddings
for paper_id, emb in embeddings.items():
    if np.array_equal(emb, target_embedding):
        embeddings[paper_id] = zero_vector

modified_count = sum(np.array_equal(emb, zero_vector) for emb in embeddings.values())
print(f"{modified_count} embeddings were set to zero vectors.")




embedding_dim = 768



valid_embeds = [v for v in embeddings.values() if np.linalg.norm(v) > 0]
global_mean = np.mean(valid_embeds, axis=0) if valid_embeds else np.zeros(w2v.vector_size) 

    
def get_author_based_embedding(pid, authors, base_embeddings):
    author_set = authors.get(pid, set())
    author_docs = [
        doc_id for doc_id, auth_set in authors.items()
        if doc_id != pid and auth_set & author_set
    ]
    valid_embeddings = [
        base_embeddings[doc_id]
        for doc_id in author_docs
        if np.linalg.norm(base_embeddings[doc_id]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)

def get_average_neighbor_embedding(node, graph, base_embeddings):
    neighbors = list(graph.neighbors(node))
    neighbor_embeds = [
        base_embeddings[n]
        for n in neighbors if np.linalg.norm(base_embeddings[n]) > 0
    ]
    return np.mean(neighbor_embeds, axis=0) if neighbor_embeds else np.zeros(embedding_dim)

def get_2hop_neighbor_embedding(node, graph, base_embeddings):
    two_hop_neighbors = set()
    for neighbor in graph.neighbors(node):
        two_hop_neighbors.update(graph.neighbors(neighbor))
    two_hop_neighbors.discard(node)
    valid_embeddings = [
        base_embeddings[n]
        for n in two_hop_neighbors if np.linalg.norm(base_embeddings[n]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)

embedding_stats = {
    "abstract": 0,
    "neighbor_fallback": 0,
    "2hop_fallback": 0,
    "author_fallback": 0,
    "global_mean_fallback": 0,
    "zero_vector": 0
}

def get_hybrid_embedding(pid, graph, authors, base_embeddings):
    emb = base_embeddings[pid]
    if np.linalg.norm(emb) > 0:
        embedding_stats["abstract"] += 1
        return emb

    emb = get_average_neighbor_embedding(pid, graph, base_embeddings)
    if np.linalg.norm(emb) > 0:
        embedding_stats["neighbor_fallback"] += 1
        return emb

    emb = get_2hop_neighbor_embedding(pid, graph, base_embeddings)
    if np.linalg.norm(emb) > 0:
        embedding_stats["2hop_fallback"] += 1
        return emb

    emb = get_author_based_embedding(pid, authors, base_embeddings)
    if np.linalg.norm(emb) > 0:
        embedding_stats["author_fallback"] += 1
        return emb

    if np.linalg.norm(global_mean) > 0:
        embedding_stats["global_mean_fallback"] += 1
        return global_mean

    embedding_stats["zero_vector"] += 1
    return np.zeros(embedding_dim)




doc_embeddings = {}
for pid in tqdm(abstracts.keys(), desc="Constructing hybrid embeddings"):
    final_emb = get_hybrid_embedding(pid, undirected_G, authors, embeddings)
    doc_embeddings[pid] = final_emb




print("\n=== Embedding Source Statistics ===")
for k, v in embedding_stats.items():
    print(f"{k.replace('_', ' ').capitalize():<28}: {v}")


import numpy as np

# Get sorted list of paper IDs to preserve order
paper_ids = sorted(doc_embeddings.keys())
embedding_matrix = np.array([doc_embeddings[pid] for pid in paper_ids])


from sklearn.decomposition import PCA
















# Choose a target dimension (e.g., 100)
pca = PCA(n_components=600, random_state=42)
reduced_embeddings = pca.fit_transform(embedding_matrix)

# Re-map reduced embeddings back to paper IDs
reduced_doc_embeddings = {
    pid: reduced_embeddings[i] for i, pid in enumerate(paper_ids)
}





np.save("reduced_scivocab_embeddings.npy", reduced_embeddings)








#closeness_centrality = nx.closeness_centrality(undirected_G)
with open("closeness_centrality.pkl", "rb") as f:
    closeness_centrality = pickle.load(f)

from networkx.algorithms.link_prediction import adamic_adar_index, preferential_attachment

def compute_adamic_adar(graph, u, v):
    try:
        return list(adamic_adar_index(graph, [(u, v)]))[0][2]
    except:
        return 0.0

def compute_preferential_attachment(graph, u, v):
    try:
        return list(preferential_attachment(graph, [(u, v)]))[0][2]
    except:
        return 0.0

import networkx as nx
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression


# Assuming undirected_G is your undirected version of the graph
clustering_coeffs = clustering(undirected_G)

X = []
y = []


for a, b in tqdm(positive_edges, desc="Creating features for positive edges"):
    #a_w2v = reduced_doc_embeddings.get(a, np.zeros(w2v_dim))  # Existing Word2Vec embeddings
    #b_w2v = reduced_doc_embeddings.get(b, np.zeros(w2v_dim))

    a_sci = reduced_doc_embeddings.get(a, np.zeros(400))  # Reduced SciBERT
    b_sci = reduced_doc_embeddings.get(b, np.zeros(100))


    X.append([
        clustering_coeffs.get(a, 0.0),
        clustering_coeffs.get(b, 0.0),
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        #cosine_similarity([a_w2v], [b_w2v])[0][0],  # Word2Vec sim
        cosine_similarity([a_sci], [b_sci])[0][0],  # SciBERT sim
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(1)

for a, b in tqdm(negative_edges, desc="Creating features for negative edges"):
    #a_w2v = reduced_doc_embeddings.get(a, np.zeros(w2v_dim))  # Existing Word2Vec embeddings
    #b_w2v = reduced_doc_embeddings.get(b, np.zeros(w2v_dim))

    a_sci = reduced_doc_embeddings.get(a, np.zeros(400))  # Reduced SciBERT
    b_sci = reduced_doc_embeddings.get(b, np.zeros(400))


    X.append([
        clustering_coeffs.get(a, 0.0),
        clustering_coeffs.get(b, 0.0),
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        #cosine_similarity([a_w2v], [b_w2v])[0][0],  # Word2Vec sim
        cosine_similarity([a_sci], [b_sci])[0][0],  # SciBERT sim
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(0)

from sklearn.preprocessing import StandardScaler

# Convert X to NumPy array (before scaling)
X_before = np.array(X)



# Apply scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_before)


# Convert to NumPy
X_np_scaled_full = np.array(X_scaled)
y_np = np.array(y)



model = LogisticRegression(max_iter=1500, random_state=42)
model.fit(X_np_scaled_full, y_np)
print("Model trained on full dataset.")

X_test = []
test_edges = []

with open("test.txt", "r") as f:
    for line in f:
        a, b = map(int, line.strip().split(","))
        test_edges.append((a, b))

print(f"Loaded {len(test_edges)} edges from test.txt")


for a, b in tqdm(test_edges, desc="Extracting features from test set"):

    #a_w2v = reduced_doc_embeddings.get(a, np.zeros(w2v_dim))  # Existing Word2Vec embeddings
    #b_w2v = reduced_doc_embeddings.get(b, np.zeros(w2v_dim))

    a_sci = reduced_doc_embeddings.get(a, np.zeros(100))  # Reduced SciBERT
    b_sci = reduced_doc_embeddings.get(b, np.zeros(100))

    X_test.append([
        clustering_coeffs.get(a, 0.0),
        clustering_coeffs.get(b, 0.0),
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        #cosine_similarity([a_w2v], [b_w2v])[0][0],  # Word2Vec sim
        cosine_similarity([a_sci], [b_sci])[0][0],  # SciBERT sim
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    
X_test_np = np.array(X_test)


X_test_scaled = scaler.transform(X_test_np)


y_test_probs = model.predict_proba(X_test_scaled )[:, 1]


submission_df = pd.DataFrame({
    "ID": range(len(y_test_probs)),
    "Label": y_test_probs
})
submission_df.to_csv("sciBert.csv", index=False)
print("Submission file saved as 'scibert.csv'")




from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
import numpy as np

X_np_unscaled = np.array(X)
y_np = np.array(y)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

nll_scores = []
fold = 1

for train_index, val_index in kf.split(X_np_unscaled):
    print(f"\n=== Fold {fold} ===")
    X_train, X_val = X_np_unscaled[train_index], X_np_unscaled[val_index]
    y_train, y_val = y_np[train_index], y_np[val_index]

    # Scale features based on training data
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    # Train logistic regression with probability output
    model = LogisticRegression(max_iter=1500, random_state=42)
    model.fit(X_train_scaled, y_train)

    # Predict probabilities
    y_val_probs = model.predict_proba(X_val_scaled)

    # Compute negative log-likelihood (log loss)
    nll = log_loss(y_val, y_val_probs)
    print(f"Negative Log-Likelihood (Log Loss): {nll:.4f}")
    nll_scores.append(nll)
    
    fold += 1

# Average NLL
avg_nll = np.mean(nll_scores)
print(f"\n=== Average Negative Log-Likelihood over 5 folds: {avg_nll:.4f} ===")





Graph loaded with 138499 nodes and 1091955 edges.
Loaded 1091955 negative edges from file.
loading abstracts.txt...
loading authors.txt...


Loading SciBERT chunks: 100%|███████████████████████████████████████████| 5/5 [00:00<00:00, 13.98it/s]


Loaded 138499 total SciBERT embeddings.
7249 embeddings were set to zero vectors.


Constructing hybrid embeddings: 100%|█████████████████████| 138499/138499 [00:00<00:00, 224714.91it/s]



=== Embedding Source Statistics ===
Abstract                    : 131250
Neighbor fallback           : 6918
2hop fallback               : 327
Author fallback             : 3
Global mean fallback        : 1
Zero vector                 : 0


Creating features for negative edges: 100%|███████████████| 1091955/1091955 [02:44<00:00, 6633.23it/s]


Model trained on full dataset.
Loaded 106692 edges from test.txt


Extracting features from test set: 100%|████████████████████| 106692/106692 [00:15<00:00, 6973.26it/s]


Submission file saved as 'scibert.csv'

=== Fold 1 ===
Negative Log-Likelihood (Log Loss): 0.1906

=== Fold 2 ===
Negative Log-Likelihood (Log Loss): 0.1913

=== Fold 3 ===
Negative Log-Likelihood (Log Loss): 0.1904

=== Fold 4 ===
Negative Log-Likelihood (Log Loss): 0.1915

=== Fold 5 ===
Negative Log-Likelihood (Log Loss): 0.1906

=== Average Negative Log-Likelihood over 5 folds: 0.1909 ===


**generating specter2 embeddings**

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
from tqdm import tqdm
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("allenai/specter2_base")
model = AutoModel.from_pretrained("allenai/specter2_base")

paper_ids = list(abstracts.keys())
abstract_texts = [abstracts[pid] for pid in paper_ids]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

batch_size = 16
all_embeddings = []

print("Tokenizing and encoding abstracts in batches...")
for i in tqdm(range(0, len(abstract_texts), batch_size)):
    batch_texts = abstract_texts[i:i+batch_size]

    inputs = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        return_tensors="pt",
        return_token_type_ids=False,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        embeddings = outputs.last_hidden_state[:, 0, :]  # CLS token
        

    # Append embeddings from this batch
    all_embeddings.append(embeddings.cpu())

# Concatenate all batches into one array
embeddings_np = torch.cat(all_embeddings, dim=0).numpy()
np.save("specter_embeddings.npy", embeddings_np)

print("Total embeddings:", len(embeddings_np))

**specter2 base , w2b, scivocab**

In [2]:
import numpy as np
from sklearn.decomposition import PCA
import pickle
import os
from transformers import BertTokenizer, BertModel
import torch
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.decomposition import PCA
from networkx.algorithms.cluster import clustering
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from sklearn.neighbors import NearestNeighbors



edges = pd.read_csv("edgelist.txt", header=None, names=["source", "target"])
positive_edges = list(edges.itertuples(index=False, name=None))
G = nx.DiGraph()
G.add_edges_from(edges.itertuples(index=False, name=None))
print(f"Graph loaded with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

# Load negative edges from file
negative_edges = set()
with open("negative_edgelist2.txt", "r") as f:
    for line in f:
        u, v = map(int, line.strip().split(','))
        negative_edges.add((u, v))

print(f"Loaded {len(negative_edges)} negative edges from file.")



undirected_G = G.to_undirected()
print("loading abstracts.txt...")
abstracts = {}
with open("abstracts.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("|--|")
        abstracts[int(parts[0])] = parts[1]

print("loading authors.txt...")
authors = {}
with open("authors.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("|--|")
        authors[int(parts[0])] = set(parts[1].split(","))










        
# Make sure you have these first:
embeddings = np.load("specter_embeddings.npy")
paper_ids = list(abstracts.keys())  # Must be in the same order as embeddings were created













index_76 = paper_ids.index(76)

# Get embedding for paper ID 76
if index_76 is not None:
    emb_76 = embeddings[index_76].copy()  # ensure independent copy
    zero_vector = np.zeros_like(emb_76)
    replaced_count = 0
    for i, emb in enumerate(embeddings):
        if np.allclose(emb, emb_76, atol=1e-5, rtol=0):  # check with tight tolerance
            #print("This was an identical embedding: ", i)
            embeddings[i] = zero_vector
            replaced_count += 1
    print(f"\nReplaced {replaced_count} nearly identical embeddings with zero vectors (4 decimal precision).")
embedding_dim = 768



embeddings_dict = {pid: emb for pid, emb in zip(paper_ids, embeddings)}





valid_embeds = [v for v in embeddings_dict.values() if np.linalg.norm(v) > 0]
global_mean = np.mean(valid_embeds, axis=0) if valid_embeds else np.zeros(embedding_dim) 

    
def get_author_based_embedding(pid, authors, base_embeddings):
    author_set = authors.get(pid, set())
    author_docs = [
        doc_id for doc_id, auth_set in authors.items()
        if doc_id != pid and auth_set & author_set
    ]
    valid_embeddings = [
        base_embeddings[doc_id]
        for doc_id in author_docs
        if np.linalg.norm(base_embeddings[doc_id]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)

def get_average_neighbor_embedding(node, graph, base_embeddings):
    neighbors = list(graph.neighbors(node))
    neighbor_embeds = [
        base_embeddings[n]
        for n in neighbors if np.linalg.norm(base_embeddings[n]) > 0
    ]
    return np.mean(neighbor_embeds, axis=0) if neighbor_embeds else np.zeros(embedding_dim)

def get_2hop_neighbor_embedding(node, graph, base_embeddings):
    two_hop_neighbors = set()
    for neighbor in graph.neighbors(node):
        two_hop_neighbors.update(graph.neighbors(neighbor))
    two_hop_neighbors.discard(node)
    valid_embeddings = [
        base_embeddings[n]
        for n in two_hop_neighbors if np.linalg.norm(base_embeddings[n]) > 0
    ]
    return np.mean(valid_embeddings, axis=0) if valid_embeddings else np.zeros(embedding_dim)

embedding_stats = {
    "abstract": 0,
    "neighbor_fallback": 0,
    "2hop_fallback": 0,
    "author_fallback": 0,
    "global_mean_fallback": 0,
    "zero_vector": 0
}

def get_hybrid_embedding(pid, graph, authors, base_embeddings):
    emb = base_embeddings[pid]
    if np.linalg.norm(emb) > 0:
        embedding_stats["abstract"] += 1
        return emb

    emb = get_average_neighbor_embedding(pid, graph, base_embeddings)
    if np.linalg.norm(emb) > 0:
        embedding_stats["neighbor_fallback"] += 1
        return emb

    emb = get_2hop_neighbor_embedding(pid, graph, base_embeddings)
    if np.linalg.norm(emb) > 0:
        embedding_stats["2hop_fallback"] += 1
        return emb

    emb = get_author_based_embedding(pid, authors, base_embeddings)
    if np.linalg.norm(emb) > 0:
        embedding_stats["author_fallback"] += 1
        return emb

    if np.linalg.norm(global_mean) > 0:
        embedding_stats["global_mean_fallback"] += 1
        return global_mean

    embedding_stats["zero_vector"] += 1
    return np.zeros(embedding_dim)


doc_embeddings = {}
for pid in tqdm(abstracts.keys(), desc="Constructing hybrid embeddings"):
    final_emb = get_hybrid_embedding(pid, undirected_G, authors, embeddings)
    doc_embeddings[pid] = final_emb


print("\n=== Embedding Source Statistics ===")
for k, v in embedding_stats.items():
    print(f"{k.replace('_', ' ').capitalize():<28}: {v}")















# Convert dict to NumPy array
doc_matrix = np.array(list(doc_embeddings.values()))

# Now you can get the shape
n_components = min(250, doc_matrix.shape[0], doc_matrix.shape[1])

# Apply PCA
pca = PCA(n_components=n_components, random_state=42)
reduced_vectors = pca.fit_transform(doc_matrix)


paper_to_reduced_vector = {
    pid: reduced_vectors[i] for i, pid in enumerate(paper_ids)
}
















with open("closeness_centrality.pkl", "rb") as f:
    closeness_centrality = pickle.load(f)

from networkx.algorithms.link_prediction import adamic_adar_index, preferential_attachment

def compute_adamic_adar(graph, u, v):
    try:
        return list(adamic_adar_index(graph, [(u, v)]))[0][2]
    except:
        return 0.0

def compute_preferential_attachment(graph, u, v):
    try:
        return list(preferential_attachment(graph, [(u, v)]))[0][2]
    except:
        return 0.0

import networkx as nx
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression


# Assuming undirected_G is your undirected version of the graph
clustering_coeffs = clustering(undirected_G)


import numpy as np

reduced_embeddings = np.load("reduced_w2v_embeddings.npy")


reduced_doc_embeddings = {
    pid: reduced_embeddings[idx] for idx, pid in enumerate(doc_embeddings.keys())
}



import numpy as np

# Step 1: Load the reduced embedding matrix
#reduced_scivocab = np.load("reduced_scivocab_embeddings.npy")

# Step 2: Rebuild the mapping of paper IDs to embeddings
# Make sure paper_ids is loaded or available in the same order as before
#reduced_scivocab_embeddings = {
 #   pid: reduced_scivocab[i] for i, pid in enumerate(paper_ids)
#}











def compute_knn(doc_embeddings, doc_ids, k=5):
    embedding_matrix = np.array([doc_embeddings[pid] for pid in doc_ids])

    knn_model = NearestNeighbors(n_neighbors=k+1, metric='cosine')  # +1 for self
    knn_model.fit(embedding_matrix)

    pid_to_index = {pid: idx for idx, pid in enumerate(doc_ids)}
    
    knn_dict = {}
    for pid in tqdm(doc_ids, desc="Precomputing KNNs"):
        idx = pid_to_index[pid]
        distances, indices = knn_model.kneighbors([embedding_matrix[idx]])
        neighbors = [doc_ids[i] for i in indices[0] if doc_ids[i] != pid]
        knn_dict[pid] = neighbors

    return knn_dict





doc_ids = list(reduced_doc_embeddings.keys())
knn_dict_reduced = compute_knn(reduced_doc_embeddings, doc_ids)

doc_ids_w2v = list(paper_to_reduced_vector.keys())
#knn_dict_w2v = compute_knn(paper_to_reduced_vector, doc_ids_w2v)




def knn_similarity(target_id, knn_source, doc_embeddings):
    similarities = []
    for neighbor in knn_source:
        if neighbor in doc_embeddings:
            sim = cosine_similarity([doc_embeddings[target_id]], [doc_embeddings[neighbor]])[0][0]
            similarities.append(sim)
    return np.mean(similarities) if similarities else 0.0















X = []
y = []


for a, b in tqdm(positive_edges, desc="Creating features for positive edges"):
    a_w2v = reduced_doc_embeddings.get(a, np.zeros(300))  # Existing Word2Vec embeddings
    b_w2v = reduced_doc_embeddings.get(b, np.zeros(300))

   
    a_sci = paper_to_reduced_vector.get(a, np.zeros(n_components))  # ✅
    b_sci = paper_to_reduced_vector.get(b, np.zeros(n_components))  # ✅



   # a_vocab = reduced_scivocab_embeddings.get(a, np.zeros(600))  # Reduced SciBERT
   # b_vocab = reduced_scivocab_embeddings.get(b, np.zeros(600))



   # For SciBERT-based reduced_doc_embeddings
    #a_knn = knn_dict_reduced.get(a, [])
    #b_knn = knn_dict_reduced.get(b, [])
    #sim_sci_a_to_b = knn_similarity(b, a_knn, reduced_doc_embeddings)
    #sim_sci_b_to_a = knn_similarity(a, b_knn, reduced_doc_embeddings)
    
    # For Word2Vec-based paper_to_reduced_vector
    #a_knn_w2v = knn_dict_w2v.get(a, [])
    #b_knn_w2v = knn_dict_w2v.get(b, [])
    #sim_w2v_a_to_b = knn_similarity(b, a_knn_w2v, paper_to_reduced_vector)
    #sim_w2v_b_to_a = knn_similarity(a, b_knn_w2v, paper_to_reduced_vector)


    

    X.append([
        clustering_coeffs.get(a, 0.0),
        clustering_coeffs.get(b, 0.0),
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_w2v], [b_w2v])[0][0],  # Word2Vec sim
        cosine_similarity([a_sci], [b_sci])[0][0],  # SciBERT sim
       # cosine_similarity([a_vocab], [b_vocab])[0][0],  # SciBERT vocab
        len(authors.get(a, set()) & authors.get(b, set())),
        #sim_sci_a_to_b,
        #sim_sci_b_to_a,
        #sim_w2v_a_to_b,
        #sim_w2v_b_to_a
        
    ])
    y.append(1)

for a, b in tqdm(negative_edges, desc="Creating features for negative edges"):
    a_w2v = reduced_doc_embeddings.get(a, np.zeros(300))  # Existing Word2Vec embeddings
    b_w2v = reduced_doc_embeddings.get(b, np.zeros(300))

    a_sci = paper_to_reduced_vector.get(a, np.zeros(n_components))  # ✅
    b_sci = paper_to_reduced_vector.get(b, np.zeros(n_components))  # ✅

   # a_vocab = reduced_scivocab_embeddings.get(a, np.zeros(600))  # Reduced SciBERT
   # b_vocab = reduced_scivocab_embeddings.get(b, np.zeros(600))


    # For SciBERT-based reduced_doc_embeddings
   # a_knn = knn_dict_reduced.get(a, [])
    #b_knn = knn_dict_reduced.get(b, [])
    #sim_sci_a_to_b = knn_similarity(b, a_knn, reduced_doc_embeddings)
    #sim_sci_b_to_a = knn_similarity(a, b_knn, reduced_doc_embeddings)
    
    # For Word2Vec-based paper_to_reduced_vector
    #a_knn_w2v = knn_dict_w2v.get(a, [])
    #b_knn_w2v = knn_dict_w2v.get(b, [])
    #sim_w2v_a_to_b = knn_similarity(b, a_knn_w2v, paper_to_reduced_vector)
    #sim_w2v_b_to_a = knn_similarity(a, b_knn_w2v, paper_to_reduced_vector)

    X.append([
        clustering_coeffs.get(a, 0.0),
        clustering_coeffs.get(b, 0.0),
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_w2v], [b_w2v])[0][0],  # Word2Vec sim
        cosine_similarity([a_sci], [b_sci])[0][0],  # SciBERT sim
        #cosine_similarity([a_vocab], [b_vocab])[0][0],  # SciBERT vocab
        len(authors.get(a, set()) & authors.get(b, set())),
        #sim_sci_a_to_b,
        #sim_sci_b_to_a,
        #sim_w2v_a_to_b,
        #sim_w2v_b_to_a
    ])
    y.append(0)

from sklearn.preprocessing import StandardScaler

# Convert X to NumPy array (before scaling)
X_before = np.array(X)



# Apply scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_before)


# Convert to NumPy
X_np_scaled_full = np.array(X_scaled)
y_np = np.array(y)



model = LogisticRegression(max_iter=1500, random_state=42)
model.fit(X_np_scaled_full, y_np)
print("Model trained on full dataset.")

X_test = []
test_edges = []

with open("test.txt", "r") as f:
    for line in f:
        a, b = map(int, line.strip().split(","))
        test_edges.append((a, b))

print(f"Loaded {len(test_edges)} edges from test.txt")


for a, b in tqdm(test_edges, desc="Extracting features from test set"):

    a_w2v = reduced_doc_embeddings.get(a, np.zeros(300))  # Existing Word2Vec embeddings
    b_w2v = reduced_doc_embeddings.get(b, np.zeros(300))

    a_sci = paper_to_reduced_vector.get(a, np.zeros(n_components))  # ✅
    b_sci = paper_to_reduced_vector.get(b, np.zeros(n_components))  # ✅

   # a_vocab = reduced_scivocab_embeddings.get(a, np.zeros(600))  # Reduced SciBERT
   # b_vocab = reduced_scivocab_embeddings.get(b, np.zeros(600))


     # For SciBERT-based reduced_doc_embeddings
    #a_knn = knn_dict_reduced.get(a, [])
    #b_knn = knn_dict_reduced.get(b, [])
    #sim_sci_a_to_b = knn_similarity(b, a_knn, reduced_doc_embeddings)
    #sim_sci_b_to_a = knn_similarity(a, b_knn, reduced_doc_embeddings)
    
    # For Word2Vec-based paper_to_reduced_vector
   # a_knn_w2v = knn_dict_w2v.get(a, [])
   # b_knn_w2v = knn_dict_w2v.get(b, [])
   # sim_w2v_a_to_b = knn_similarity(b, a_knn_w2v, paper_to_reduced_vector)
    #sim_w2v_b_to_a = knn_similarity(a, b_knn_w2v, paper_to_reduced_vector)

    X_test.append([
        clustering_coeffs.get(a, 0.0),
        clustering_coeffs.get(b, 0.0),
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_w2v], [b_w2v])[0][0],  # Word2Vec sim
        cosine_similarity([a_sci], [b_sci])[0][0],  # SciBERT sim
        #cosine_similarity([a_vocab], [b_vocab])[0][0],  # SciBERT vocab
        len(authors.get(a, set()) & authors.get(b, set())),
       # sim_sci_a_to_b,
        #sim_sci_b_to_a,
        #sim_w2v_a_to_b,
        #sim_w2v_b_to_a
    ])
    
X_test_np = np.array(X_test)


X_test_scaled = scaler.transform(X_test_np)


y_test_probs = model.predict_proba(X_test_scaled )[:, 1]


submission_df = pd.DataFrame({
    "ID": range(len(y_test_probs)),
    "Label": y_test_probs
})
submission_df.to_csv("specter2_fixed.csv", index=False)
print("Submission file saved as 'specter2_fixed.csv'")




from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
import numpy as np

X_np_unscaled = np.array(X)
y_np = np.array(y)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

nll_scores = []
fold = 1

for train_index, val_index in kf.split(X_np_unscaled):
    print(f"\n=== Fold {fold} ===")
    X_train, X_val = X_np_unscaled[train_index], X_np_unscaled[val_index]
    y_train, y_val = y_np[train_index], y_np[val_index]

    # Scale features based on training data
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    # Train logistic regression with probability output
    model = LogisticRegression(max_iter=1500, random_state=42)
    model.fit(X_train_scaled, y_train)

    # Predict probabilities
    y_val_probs = model.predict_proba(X_val_scaled)

    # Compute negative log-likelihood (log loss)
    nll = log_loss(y_val, y_val_probs)
    print(f"Negative Log-Likelihood (Log Loss): {nll:.4f}")
    nll_scores.append(nll)
    
    fold += 1

# Average NLL
avg_nll = np.mean(nll_scores)
print(f"\n=== Average Negative Log-Likelihood over 5 folds: {avg_nll:.4f} ===")


Graph loaded with 138499 nodes and 1091955 edges.
Loaded 1091955 negative edges from file.
loading abstracts.txt...
loading authors.txt...

Replaced 7249 nearly identical embeddings with zero vectors (4 decimal precision).


Constructing hybrid embeddings: 100%|███████████████████████████████████████████| 138499/138499 [00:00<00:00, 221842.97it/s]



=== Embedding Source Statistics ===
Abstract                    : 131250
Neighbor fallback           : 6918
2hop fallback               : 327
Author fallback             : 3
Global mean fallback        : 1
Zero vector                 : 0


Creating features for negative edges: 100%|█████████████████████████████████████| 1091955/1091955 [05:06<00:00, 3566.63it/s]


Model trained on full dataset.
Loaded 106692 edges from test.txt


Extracting features from test set: 100%|██████████████████████████████████████████| 106692/106692 [00:28<00:00, 3681.82it/s]


Submission file saved as 'specter2_fixed.csv'

=== Fold 1 ===
Negative Log-Likelihood (Log Loss): 0.1016

=== Fold 2 ===
Negative Log-Likelihood (Log Loss): 0.0998

=== Fold 3 ===
Negative Log-Likelihood (Log Loss): 0.0992

=== Fold 4 ===
Negative Log-Likelihood (Log Loss): 0.1000

=== Fold 5 ===
Negative Log-Likelihood (Log Loss): 0.1014

=== Average Negative Log-Likelihood over 5 folds: 0.1004 ===


**fix negative edges**

In [3]:
import numpy as np

# Step 1: Load negative edges
negative_edges = set()
with open("negative_edgelist2.txt", "r") as f:
    for line in f:
        u, v = map(int, line.strip().split(','))
        negative_edges.add((u, v))

# Step 2: Load test edges
test_edges = []
with open("test.txt", "r") as f:
    for line in f:
        a, b = map(int, line.strip().split(","))
        test_edges.append((a, b))

# Step 3: Remove test edges from negative edges
for edge in test_edges:
    if edge in negative_edges:
        negative_edges.remove(edge)
print(f"Updated negative_edges size: {len(negative_edges)}")
# Step 4: Load positive edges and all nodes from edgelist.txt






Updated negative_edges size: 1091955
Final number of negative edges: 1091955
Number of test edges still in negative_edges: 0


**tf-idf**

In [2]:
import numpy as np
from sklearn.decomposition import PCA
import pickle
import os
from transformers import BertTokenizer, BertModel
import torch
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.decomposition import PCA
from networkx.algorithms.cluster import clustering
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer




edges = pd.read_csv("edgelist.txt", header=None, names=["source", "target"])
positive_edges = list(edges.itertuples(index=False, name=None))
G = nx.DiGraph()
G.add_edges_from(edges.itertuples(index=False, name=None))
print(f"Graph loaded with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

# Load negative edges from file
negative_edges = set()
with open("negative_edgelist2.txt", "r") as f:
    for line in f:
        u, v = map(int, line.strip().split(','))
        negative_edges.add((u, v))

print(f"Loaded {len(negative_edges)} negative edges from file.")



undirected_G = G.to_undirected()
print("loading abstracts.txt...")
abstracts = {}
with open("abstracts.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("|--|")
        abstracts[int(parts[0])] = parts[1]

print("loading authors.txt...")
authors = {}
with open("authors.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("|--|")
        authors[int(parts[0])] = set(parts[1].split(","))
        

# Load the reduced vectors
reduced_vectors = np.load('reduced_sciBERT_embeddings.npy')

# Load the corresponding paper ids
paper_ids = np.load('paper_ids.npy')

# Rebuild the dictionary mapping paper ids to reduced vectors
paper_to_reduced_vector = {pid: reduced_vectors[i] for i, pid in enumerate(paper_ids)}


with open("closeness_centrality.pkl", "rb") as f:
    closeness_centrality = pickle.load(f)

from networkx.algorithms.link_prediction import adamic_adar_index, preferential_attachment

def compute_adamic_adar(graph, u, v):
    try:
        return list(adamic_adar_index(graph, [(u, v)]))[0][2]
    except:
        return 0.0

def compute_preferential_attachment(graph, u, v):
    try:
        return list(preferential_attachment(graph, [(u, v)]))[0][2]
    except:
        return 0.0





# Assuming undirected_G is your undirected version of the graph
clustering_coeffs = clustering(undirected_G)




reduced_embeddings = np.load("reduced_w2v_embeddings.npy")

paper_ids = list(abstracts.keys())

reduced_doc_embeddings = {
    pid: reduced_embeddings[idx] for idx, pid in enumerate(paper_ids)
}






#tf idf


vectorizer = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf_matrix = vectorizer.fit_transform(abstracts.values())


paper_id_to_tfidf = {i: tfidf_matrix[i] for i in range(len(abstracts))}



n_components= 768
X = []
y = []


for a, b in tqdm(positive_edges, desc="Creating features for positive edges"):
    a_w2v = reduced_doc_embeddings.get(a, np.zeros(300))  # Existing Word2Vec embeddings
    b_w2v = reduced_doc_embeddings.get(b, np.zeros(300))

   
    a_sci = paper_to_reduced_vector.get(a, np.zeros(n_components))  # ✅
    b_sci = paper_to_reduced_vector.get(b, np.zeros(n_components))  # ✅

    a_tfidf = paper_id_to_tfidf.get(a)
    b_tfidf = paper_id_to_tfidf.get(b)
    tfidf_sim = cosine_similarity(a_tfidf, b_tfidf)[0][0] if a_tfidf is not None and b_tfidf is not None else 0.0



    X.append([
        clustering_coeffs.get(a, 0.0),
        clustering_coeffs.get(b, 0.0),
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_w2v], [b_w2v])[0][0],  # Word2Vec sim
        cosine_similarity([a_sci], [b_sci])[0][0],  # SciBERT sim
        tfidf_sim,       
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(1)

for a, b in tqdm(negative_edges, desc="Creating features for negative edges"):
    a_w2v = reduced_doc_embeddings.get(a, np.zeros(300))  # Existing Word2Vec embeddings
    b_w2v = reduced_doc_embeddings.get(b, np.zeros(300))

    a_sci = paper_to_reduced_vector.get(a, np.zeros(n_components))  # ✅
    b_sci = paper_to_reduced_vector.get(b, np.zeros(n_components))  # ✅


    a_tfidf = paper_id_to_tfidf.get(a)
    b_tfidf = paper_id_to_tfidf.get(b)
    tfidf_sim = cosine_similarity(a_tfidf, b_tfidf)[0][0] if a_tfidf is not None and b_tfidf is not None else 0.0


    X.append([
        clustering_coeffs.get(a, 0.0),
        clustering_coeffs.get(b, 0.0),
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_w2v], [b_w2v])[0][0],  # Word2Vec sim
        cosine_similarity([a_sci], [b_sci])[0][0],  # SciBERT sim
        tfidf_sim,
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(0)

from sklearn.preprocessing import StandardScaler

# Convert X to NumPy array (before scaling)
X_before = np.array(X)



# Apply scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_before)


# Convert to NumPy
X_np_scaled_full = np.array(X_scaled)
y_np = np.array(y)



model = LogisticRegression(max_iter=1500, random_state=42)
model.fit(X_np_scaled_full, y_np)
print("Model trained on full dataset.")

X_test = []
test_edges = []

with open("test.txt", "r") as f:
    for line in f:
        a, b = map(int, line.strip().split(","))
        test_edges.append((a, b))

print(f"Loaded {len(test_edges)} edges from test.txt")


for a, b in tqdm(test_edges, desc="Extracting features from test set"):

    a_w2v = reduced_doc_embeddings.get(a, np.zeros(300))  # Existing Word2Vec embeddings
    b_w2v = reduced_doc_embeddings.get(b, np.zeros(300))

    a_sci = paper_to_reduced_vector.get(a, np.zeros(n_components))  # ✅
    b_sci = paper_to_reduced_vector.get(b, np.zeros(n_components))  # ✅


    a_tfidf = paper_id_to_tfidf.get(a)
    b_tfidf = paper_id_to_tfidf.get(b)
    tfidf_sim = cosine_similarity(a_tfidf, b_tfidf)[0][0] if a_tfidf is not None and b_tfidf is not None else 0.0


    X_test.append([
        clustering_coeffs.get(a, 0.0),
        clustering_coeffs.get(b, 0.0),
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_w2v], [b_w2v])[0][0],  # Word2Vec sim
        cosine_similarity([a_sci], [b_sci])[0][0],  # SciBERT sim
        tfidf_sim,
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    
X_test_np = np.array(X_test)


X_test_scaled = scaler.transform(X_test_np)


y_test_probs = model.predict_proba(X_test_scaled )[:, 1]


submission_df = pd.DataFrame({
    "ID": range(len(y_test_probs)),
    "Label": y_test_probs
})
submission_df.to_csv("tf_idf.csv", index=False)
print("Submission file saved as 'tf_idf.csv'")




from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
import numpy as np

X_np_unscaled = np.array(X)
y_np = np.array(y)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

nll_scores = []
fold = 1

for train_index, val_index in kf.split(X_np_unscaled):
    print(f"\n=== Fold {fold} ===")
    X_train, X_val = X_np_unscaled[train_index], X_np_unscaled[val_index]
    y_train, y_val = y_np[train_index], y_np[val_index]

    # Scale features based on training data
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    # Train logistic regression with probability output
    model = LogisticRegression(max_iter=1500, random_state=42)
    model.fit(X_train_scaled, y_train)

    # Predict probabilities
    y_val_probs = model.predict_proba(X_val_scaled)

    # Compute negative log-likelihood (log loss)
    nll = log_loss(y_val, y_val_probs)
    print(f"Negative Log-Likelihood (Log Loss): {nll:.4f}")
    nll_scores.append(nll)
    
    fold += 1

# Average NLL
avg_nll = np.mean(nll_scores)
print(f"\n=== Average Negative Log-Likelihood over 5 folds: {avg_nll:.4f} ===")

Graph loaded with 138499 nodes and 1091955 edges.
Loaded 1091955 negative edges from file.
loading abstracts.txt...
loading authors.txt...


Creating features for negative edges: 100%|██████████████████████████████| 1091955/1091955 [07:59<00:00, 2278.63it/s]


Model trained on full dataset.
Loaded 106692 edges from test.txt


Extracting features from test set: 100%|███████████████████████████████████| 106692/106692 [00:47<00:00, 2237.84it/s]


Submission file saved as 'tf_idf.csv'

=== Fold 1 ===
Negative Log-Likelihood (Log Loss): 0.0970

=== Fold 2 ===
Negative Log-Likelihood (Log Loss): 0.0950

=== Fold 3 ===
Negative Log-Likelihood (Log Loss): 0.0942

=== Fold 4 ===
Negative Log-Likelihood (Log Loss): 0.0952

=== Fold 5 ===
Negative Log-Likelihood (Log Loss): 0.0968

=== Average Negative Log-Likelihood over 5 folds: 0.0956 ===


In [4]:
import nltk
nltk.download('punkt')


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/mariosellinidis/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [6]:
from nltk.tokenize import word_tokenize

text = "This is a test sentence."
print(word_tokenize(text))


['This', 'is', 'a', 'test', 'sentence', '.']


**doc2vec**

In [62]:
import numpy as np
from sklearn.decomposition import PCA
import pickle
import os
from transformers import BertTokenizer, BertModel
import torch
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.decomposition import PCA
from networkx.algorithms.cluster import clustering
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models.doc2vec import TaggedDocument
from nltk.tokenize import word_tokenize
from gensim.models.doc2vec import Doc2Vec





edges = pd.read_csv("edgelist.txt", header=None, names=["source", "target"])
positive_edges = list(edges.itertuples(index=False, name=None))
G = nx.DiGraph()
G.add_edges_from(edges.itertuples(index=False, name=None))
print(f"Graph loaded with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

# Load negative edges from file
negative_edges = set()
with open("negative_edgelist2.txt", "r") as f:
    for line in f:
        u, v = map(int, line.strip().split(','))
        negative_edges.add((u, v))

print(f"Loaded {len(negative_edges)} negative edges from file.")



undirected_G = G.to_undirected()
print("loading abstracts.txt...")
abstracts = {}
with open("abstracts.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("|--|")
        abstracts[int(parts[0])] = parts[1]

print("loading authors.txt...")
authors = {}
with open("authors.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("|--|")
        authors[int(parts[0])] = set(parts[1].split(","))
        

# Load the reduced vectors
reduced_vectors = np.load('reduced_sciBERT_embeddings.npy')

# Load the corresponding paper ids
paper_ids = np.load('paper_ids.npy')

# Rebuild the dictionary mapping paper ids to reduced vectors
paper_to_reduced_vector = {pid: reduced_vectors[i] for i, pid in enumerate(paper_ids)}


with open("closeness_centrality.pkl", "rb") as f:
    closeness_centrality = pickle.load(f)

from networkx.algorithms.link_prediction import adamic_adar_index, preferential_attachment

def compute_adamic_adar(graph, u, v):
    try:
        return list(adamic_adar_index(graph, [(u, v)]))[0][2]
    except:
        return 0.0

def compute_preferential_attachment(graph, u, v):
    try:
        return list(preferential_attachment(graph, [(u, v)]))[0][2]
    except:
        return 0.0





# Assuming undirected_G is your undirected version of the graph
clustering_coeffs = clustering(undirected_G)




reduced_embeddings = np.load("reduced_w2v_embeddings.npy")

paper_ids = list(abstracts.keys())

reduced_doc_embeddings = {
    pid: reduced_embeddings[idx] for idx, pid in enumerate(paper_ids)
}






#tf idf


vectorizer = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf_matrix = vectorizer.fit_transform(abstracts.values())


paper_id_to_tfidf = {i: tfidf_matrix[i] for i in range(len(abstracts))}



#doc2vec







from tqdm import tqdm
from gensim.models.doc2vec import TaggedDocument
from nltk.tokenize import word_tokenize

tagged_docs = []

for paper_id, text in tqdm(abstracts.items(), desc="Tokenizing abstracts"):
    tokens = word_tokenize(text.lower())
    tagged_docs.append(TaggedDocument(words=tokens, tags=[str(paper_id)]))



#doc2vec_model = Doc2Vec(
 #   vector_size=50,  # You can tune this
  #  window=1,
   # min_count=25,
    #workers=4,
    #epochs=400,
    #dm=1  # Distributed Memory (better for similarity tasks)
#)



#doc2vec_model.build_vocab(tagged_docs)
#doc2vec_model.train(tagged_docs, total_examples=doc2vec_model.corpus_count, epochs=doc2vec_model.epochs)

from gensim.models import Doc2Vec
doc2vec_model = Doc2Vec.load("doc2vec_model.model")

paper_to_doc2vec = {
    int(tagged_doc.tags[0]): doc2vec_model.dv[tagged_doc.tags[0]]
    for tagged_doc in tagged_docs
}

##doc2vec_model.save("doc2vec_model.model")




n_components= 768
X = []
y = []


for a, b in tqdm(positive_edges, desc="Creating features for positive edges"):
    a_w2v = reduced_doc_embeddings.get(a, np.zeros(300))  # Existing Word2Vec embeddings
    b_w2v = reduced_doc_embeddings.get(b, np.zeros(300))

   
    a_sci = paper_to_reduced_vector.get(a, np.zeros(n_components))  # ✅
    b_sci = paper_to_reduced_vector.get(b, np.zeros(n_components))  # ✅

    a_tfidf = paper_id_to_tfidf.get(a)
    b_tfidf = paper_id_to_tfidf.get(b)
    tfidf_sim = cosine_similarity(a_tfidf, b_tfidf)[0][0] if a_tfidf is not None and b_tfidf is not None else 0.0



    a_doc2vec = paper_to_doc2vec.get(a, np.zeros(300))
    b_doc2vec = paper_to_doc2vec.get(b, np.zeros(300))
   



    X.append([
        clustering_coeffs.get(a, 0.0),
        clustering_coeffs.get(b, 0.0),
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_w2v], [b_w2v])[0][0],  # Word2Vec sim
        cosine_similarity([a_sci], [b_sci])[0][0],  # SciBERT sim
        cosine_similarity([a_doc2vec], [b_doc2vec])[0][0],
        tfidf_sim,       
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(1)

for a, b in tqdm(negative_edges, desc="Creating features for negative edges"):
    a_w2v = reduced_doc_embeddings.get(a, np.zeros(300))  # Existing Word2Vec embeddings
    b_w2v = reduced_doc_embeddings.get(b, np.zeros(300))

    a_sci = paper_to_reduced_vector.get(a, np.zeros(n_components))  # ✅
    b_sci = paper_to_reduced_vector.get(b, np.zeros(n_components))  # ✅


  #  a_tfidf = paper_id_to_tfidf.get(a)
   # b_tfidf = paper_id_to_tfidf.get(b)
    #tfidf_sim = cosine_similarity(a_tfidf, b_tfidf)[0][0] if a_tfidf is not None and b_tfidf is not None else 0.0


    a_doc2vec = paper_to_doc2vec.get(a, np.zeros(300))
    b_doc2vec = paper_to_doc2vec.get(b, np.zeros(300))



    X.append([
        clustering_coeffs.get(a, 0.0),
        clustering_coeffs.get(b, 0.0),
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_w2v], [b_w2v])[0][0],  # Word2Vec sim
        cosine_similarity([a_sci], [b_sci])[0][0],  # SciBERT sim
        cosine_similarity([a_doc2vec], [b_doc2vec])[0][0],
        tfidf_sim,
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(0)

from sklearn.preprocessing import StandardScaler

# Convert X to NumPy array (before scaling)
X_before = np.array(X)



# Apply scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_before)


# Convert to NumPy
X_np_scaled_full = np.array(X_scaled)
y_np = np.array(y)



model = LogisticRegression(max_iter=1500, random_state=42)
model.fit(X_np_scaled_full, y_np)
print("Model trained on full dataset.")

X_test = []
test_edges = []

with open("test.txt", "r") as f:
    for line in f:
        a, b = map(int, line.strip().split(","))
        test_edges.append((a, b))

print(f"Loaded {len(test_edges)} edges from test.txt")


for a, b in tqdm(test_edges, desc="Extracting features from test set"):

    a_w2v = reduced_doc_embeddings.get(a, np.zeros(300))  # Existing Word2Vec embeddings
    b_w2v = reduced_doc_embeddings.get(b, np.zeros(300))

    a_sci = paper_to_reduced_vector.get(a, np.zeros(n_components))  # ✅
    b_sci = paper_to_reduced_vector.get(b, np.zeros(n_components))  # ✅


   # a_tfidf = paper_id_to_tfidf.get(a)
   # b_tfidf = paper_id_to_tfidf.get(b)
    #tfidf_sim = cosine_similarity(a_tfidf, b_tfidf)[0][0] if a_tfidf is not None and b_tfidf is not None else 0.0

    a_doc2vec = paper_to_doc2vec.get(a, np.zeros(300))
    b_doc2vec = paper_to_doc2vec.get(b, np.zeros(300))
    


    X_test.append([
        clustering_coeffs.get(a, 0.0),
        clustering_coeffs.get(b, 0.0),
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_w2v], [b_w2v])[0][0],  # Word2Vec sim
        cosine_similarity([a_sci], [b_sci])[0][0],  # SciBERT sim
        cosine_similarity([a_doc2vec], [b_doc2vec])[0][0],
        tfidf_sim,
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    
X_test_np = np.array(X_test)


X_test_scaled = scaler.transform(X_test_np)


y_test_probs = model.predict_proba(X_test_scaled )[:, 1]


submission_df = pd.DataFrame({
    "ID": range(len(y_test_probs)),
    "Label": y_test_probs
})
submission_df.to_csv("Doc2Vec.csv", index=False)
print("Submission file saved as 'Doc2Vec.csv'")




from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
import numpy as np

X_np_unscaled = np.array(X)
y_np = np.array(y)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

nll_scores = []
fold = 1

for train_index, val_index in kf.split(X_np_unscaled):
    print(f"\n=== Fold {fold} ===")
    X_train, X_val = X_np_unscaled[train_index], X_np_unscaled[val_index]
    y_train, y_val = y_np[train_index], y_np[val_index]

    # Scale features based on training data
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    # Train logistic regression with probability output
    model = LogisticRegression(max_iter=1500, random_state=42)
    model.fit(X_train_scaled, y_train)

    # Predict probabilities
    y_val_probs = model.predict_proba(X_val_scaled)

    # Compute negative log-likelihood (log loss)
    nll = log_loss(y_val, y_val_probs)
    print(f"Negative Log-Likelihood (Log Loss): {nll:.4f}")
    nll_scores.append(nll)
    
    fold += 1

# Average NLL
avg_nll = np.mean(nll_scores)
print(f"\n=== Average Negative Log-Likelihood over 5 folds: {avg_nll:.4f} ===")

Graph loaded with 138499 nodes and 1091955 edges.
Loaded 1091955 negative edges from file.
loading abstracts.txt...
loading authors.txt...


Tokenizing abstracts: 100%|███████████| 138499/138499 [00:52<00:00, 2636.53it/s]
Creating features for positive edges: 100%|█| 1091955/1091955 [10:22<00:00, 1753
Creating features for negative edges: 100%|█| 1091955/1091955 [07:14<00:00, 2510


Model trained on full dataset.
Loaded 106692 edges from test.txt


Extracting features from test set: 100%|█| 106692/106692 [00:51<00:00, 2068.45it


Submission file saved as 'Doc2Vec.csv'

=== Fold 1 ===
Negative Log-Likelihood (Log Loss): 0.0147

=== Fold 2 ===
Negative Log-Likelihood (Log Loss): 0.0142

=== Fold 3 ===
Negative Log-Likelihood (Log Loss): 0.0143

=== Fold 4 ===
Negative Log-Likelihood (Log Loss): 0.0143

=== Fold 5 ===
Negative Log-Likelihood (Log Loss): 0.0144

=== Average Negative Log-Likelihood over 5 folds: 0.0144 ===


**Node2Vec**

In [2]:
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.decomposition import PCA
import pickle
import os
from transformers import BertTokenizer, BertModel
import torch
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.decomposition import PCA
from networkx.algorithms.cluster import clustering
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models.doc2vec import TaggedDocument
from nltk.tokenize import word_tokenize
from gensim.models.doc2vec import Doc2Vec
from node2vec import Node2Vec


# Load all positive edges from edgelist.txt
edges = pd.read_csv("edgelist.txt", header=None, names=["source", "target"])
positive_edges = list(edges.itertuples(index=False, name=None))

# Split the positive edges into train and validation sets
train_positive_edges, val_positive_edges = train_test_split(
    positive_edges, test_size=0.2, random_state=42
)

# Create the graph only with training edges
G = nx.DiGraph()
G.add_edges_from(train_positive_edges)
undirected_G = G.to_undirected()

print(f"Graph created with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges (training only).")



# Load negative edges from file
negative_edges = set()
with open("negative_edgelist2.txt", "r") as f:
    for line in f:
        u, v = map(int, line.strip().split(','))
        negative_edges.add((u, v))

# Convert to list and split same as positive
negative_edges = list(negative_edges)
train_negative_edges, val_negative_edges = train_test_split(
    negative_edges, test_size=0.2, random_state=42
)

print(f"Loaded {len(negative_edges)} negative edges.")
print(f"Training: {len(train_positive_edges)} pos, {len(train_negative_edges)} neg")
print(f"Validation: {len(val_positive_edges)} pos, {len(val_negative_edges)} neg")

# Optional: Combine training and test sets for classifier input later
train_edges = train_positive_edges + train_negative_edges
val_edges = val_positive_edges + val_negative_edges






abstracts = {}
with open("abstracts.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("|--|")
        abstracts[int(parts[0])] = parts[1]

print("loading authors.txt...")
authors = {}
with open("authors.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("|--|")
        authors[int(parts[0])] = set(parts[1].split(","))
        

# Load the reduced vectors
reduced_vectors = np.load('reduced_sciBERT_embeddings.npy')

# Load the corresponding paper ids
paper_ids = np.load('paper_ids.npy')

# Rebuild the dictionary mapping paper ids to reduced vectors
paper_to_reduced_vector = {pid: reduced_vectors[i] for i, pid in enumerate(paper_ids)}


#with open("closeness_centrality.pkl", "rb") as f:
 #   closeness_centrality = pickle.load(f)

from networkx.algorithms.link_prediction import adamic_adar_index, preferential_attachment

def compute_adamic_adar(graph, u, v):
    try:
        return list(adamic_adar_index(graph, [(u, v)]))[0][2]
    except:
        return 0.0

def compute_preferential_attachment(graph, u, v):
    try:
        return list(preferential_attachment(graph, [(u, v)]))[0][2]
    except:
        return 0.0





# Assuming undirected_G is your undirected version of the graph
clustering_coeffs = clustering(undirected_G)




reduced_embeddings = np.load("reduced_w2v_embeddings.npy")

paper_ids = list(abstracts.keys())

reduced_doc_embeddings = {
    pid: reduced_embeddings[idx] for idx, pid in enumerate(paper_ids)
}






#tf idf


vectorizer = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf_matrix = vectorizer.fit_transform(abstracts.values())


paper_id_to_tfidf = {i: tfidf_matrix[i] for i in range(len(abstracts))}



#doc2vec







from tqdm import tqdm
from gensim.models.doc2vec import TaggedDocument
from nltk.tokenize import word_tokenize

tagged_docs = []

for paper_id, text in tqdm(abstracts.items(), desc="Tokenizing abstracts"):
    tokens = word_tokenize(text.lower())
    tagged_docs.append(TaggedDocument(words=tokens, tags=[str(paper_id)]))


from gensim.models import Doc2Vec
doc2vec_model = Doc2Vec.load("doc2vec_model.model")

paper_to_doc2vec = {
    int(tagged_doc.tags[0]): doc2vec_model.dv[tagged_doc.tags[0]]
    for tagged_doc in tagged_docs
}




#node2vec
node2vec = Node2Vec(G, dimensions=64, walk_length=30, num_walks=20, workers=2)
n2v_model = node2vec.fit(window=6, min_count=1, batch_words=4)


# Create the mapping from paper to node2vec embeddings
paper_to_node2vec = {int(node): n2v_model.wv[str(node)] for node in G.nodes()}



paper_ids = list(paper_to_node2vec.keys())
original_node2vec_matrix = np.array([paper_to_node2vec[pid] for pid in paper_ids])

# 2. Save original Node2Vec embeddings
np.save("node2vec_original.npy", original_node2vec_matrix)
with open("node2vec_ids.npy", "wb") as f:
    np.save(f, paper_ids)










#Optional: Combine training and test sets for classifier input later
#train_edges = train_positive_edges + train_negative_edges
#val_edges = val_positive_edges + val_negative_edges
#train_labels = [1] * len(train_positive_edges) + [0] * len(train_negative_edges)
#val_labels = [1] * len(val_positive_edges) + [0] * len(val_negative_edges)




n_components= 768
X = []
y = []


for a, b in tqdm(train_positive_edges, desc="Creating features for positive edges"):
    a_w2v = reduced_doc_embeddings.get(a, np.zeros(300))  # Existing Word2Vec embeddings
    b_w2v = reduced_doc_embeddings.get(b, np.zeros(300))

   
    a_sci = paper_to_reduced_vector.get(a, np.zeros(n_components))  # ✅
    b_sci = paper_to_reduced_vector.get(b, np.zeros(n_components))  # ✅

    a_tfidf = paper_id_to_tfidf.get(a)
    b_tfidf = paper_id_to_tfidf.get(b)
    tfidf_sim = cosine_similarity(a_tfidf, b_tfidf)[0][0] if a_tfidf is not None and b_tfidf is not None else 0.0



    a_doc2vec = paper_to_doc2vec.get(a, np.zeros(300))
    b_doc2vec = paper_to_doc2vec.get(b, np.zeros(300))


    a_node2vec = paper_to_node2vec.get(a, np.zeros(64))
    b_node2vec = paper_to_node2vec.get(b, np.zeros(64))
    



    X.append([
        clustering_coeffs.get(a, 0.0),
        clustering_coeffs.get(b, 0.0),
        #closeness_centrality.get(a, 0.0),
        #closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_w2v], [b_w2v])[0][0],  # Word2Vec sim
        cosine_similarity([a_sci], [b_sci])[0][0],  # SciBERT sim
        cosine_similarity([a_doc2vec], [b_doc2vec])[0][0],
        tfidf_sim,       
        cosine_similarity([a_node2vec], [b_node2vec])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(1)

for a, b in tqdm(train_negative_edges, desc="Creating features for negative edges"):
    a_w2v = reduced_doc_embeddings.get(a, np.zeros(300))  # Existing Word2Vec embeddings
    b_w2v = reduced_doc_embeddings.get(b, np.zeros(300))

    a_sci = paper_to_reduced_vector.get(a, np.zeros(n_components))  # ✅
    b_sci = paper_to_reduced_vector.get(b, np.zeros(n_components))  # ✅


    a_tfidf = paper_id_to_tfidf.get(a)
    b_tfidf = paper_id_to_tfidf.get(b)
    tfidf_sim = cosine_similarity(a_tfidf, b_tfidf)[0][0] if a_tfidf is not None and b_tfidf is not None else 0.0


    a_doc2vec = paper_to_doc2vec.get(a, np.zeros(300))
    b_doc2vec = paper_to_doc2vec.get(b, np.zeros(300))


    a_node2vec = paper_to_node2vec.get(a, np.zeros(64))
    b_node2vec = paper_to_node2vec.get(b, np.zeros(64))


    X.append([
        clustering_coeffs.get(a, 0.0),
        clustering_coeffs.get(b, 0.0),
        #closeness_centrality.get(a, 0.0),
        #closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_w2v], [b_w2v])[0][0],  # Word2Vec sim
        cosine_similarity([a_sci], [b_sci])[0][0],  # SciBERT sim
        cosine_similarity([a_doc2vec], [b_doc2vec])[0][0],
        tfidf_sim,
        cosine_similarity([a_node2vec], [b_node2vec])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(0)

from sklearn.preprocessing import StandardScaler

# Convert X to NumPy array (before scaling)
X_before = np.array(X)



# Apply scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_before)


# Convert to NumPy
X_np_scaled_full = np.array(X_scaled)
y_np = np.array(y)



model = LogisticRegression(max_iter=1500, random_state=42)
model.fit(X_np_scaled_full, y_np)
print("Model trained on full dataset.")



X_val = []
y_val = []

# Positive validation edges
for a, b in tqdm(val_positive_edges, desc="Validation: positive"):
    a_w2v = reduced_doc_embeddings.get(a, np.zeros(300))
    b_w2v = reduced_doc_embeddings.get(b, np.zeros(300))

    a_sci = paper_to_reduced_vector.get(a, np.zeros(n_components))
    b_sci = paper_to_reduced_vector.get(b, np.zeros(n_components))

    a_tfidf = paper_id_to_tfidf.get(a)
    b_tfidf = paper_id_to_tfidf.get(b)
    tfidf_sim = cosine_similarity(a_tfidf, b_tfidf)[0][0] if a_tfidf is not None and b_tfidf is not None else 0.0

    a_doc2vec = paper_to_doc2vec.get(a, np.zeros(300))
    b_doc2vec = paper_to_doc2vec.get(b, np.zeros(300))

    a_node2vec = paper_to_node2vec.get(a, np.zeros(64))
    b_node2vec = paper_to_node2vec.get(b, np.zeros(64))
  

    X_val.append([
        clustering_coeffs.get(a, 0.0),
        clustering_coeffs.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_w2v], [b_w2v])[0][0],
        cosine_similarity([a_sci], [b_sci])[0][0],
        cosine_similarity([a_doc2vec], [b_doc2vec])[0][0],
        tfidf_sim,
        cosine_similarity([a_node2vec], [b_node2vec])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y_val.append(1)

# Negative validation edges
for a, b in tqdm(val_negative_edges, desc="Validation: negative"):
    a_w2v = reduced_doc_embeddings.get(a, np.zeros(300))
    b_w2v = reduced_doc_embeddings.get(b, np.zeros(300))

    a_sci = paper_to_reduced_vector.get(a, np.zeros(n_components))
    b_sci = paper_to_reduced_vector.get(b, np.zeros(n_components))

    a_tfidf = paper_id_to_tfidf.get(a)
    b_tfidf = paper_id_to_tfidf.get(b)
    tfidf_sim = cosine_similarity(a_tfidf, b_tfidf)[0][0] if a_tfidf is not None and b_tfidf is not None else 0.0

    a_doc2vec = paper_to_doc2vec.get(a, np.zeros(300))
    b_doc2vec = paper_to_doc2vec.get(b, np.zeros(300))

    a_node2vec = paper_to_node2vec.get(a, np.zeros(64))
    b_node2vec = paper_to_node2vec.get(b, np.zeros(64))
    

    X_val.append([
        clustering_coeffs.get(a, 0.0),
        clustering_coeffs.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_w2v], [b_w2v])[0][0],
        cosine_similarity([a_sci], [b_sci])[0][0],
        cosine_similarity([a_doc2vec], [b_doc2vec])[0][0],
        tfidf_sim,
        cosine_similarity([a_node2vec], [b_node2vec])[0][0],
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y_val.append(0)


X_val_scaled = scaler.transform(np.array(X_val))
y_val = np.array(y_val)



from sklearn.metrics import log_loss

# Predict class probabilities for validation set
y_proba = model.predict_proba(X_val_scaled)[:, 1]  # Probabilities for the positive class

# Compute Negative Log-Likelihood (Log Loss)
nll = log_loss(y_val, y_proba)

print(f"Negative Log-Likelihood (NLL): {nll:.4f}")




Graph created with 135492 nodes and 873564 edges (training only).
Loaded 1091955 negative edges.
Training: 873564 pos, 873564 neg
Validation: 218391 pos, 218391 neg
loading authors.txt...


Tokenizing abstracts: 100%|███████████████████████████████████████████████████████| 138499/138499 [00:51<00:00, 2713.21it/s]


Computing transition probabilities:   0%|          | 0/135492 [00:00<?, ?it/s]

Creating features for negative edges: 100%|███████████████████████████████████████| 873564/873564 [10:07<00:00, 1438.53it/s]


Model trained on full dataset.


Validation: negative: 100%|███████████████████████████████████████████████████████| 218391/218391 [02:31<00:00, 1438.62it/s]


Negative Log-Likelihood (NLL): 0.1478


**Allternative Node2vec**

In [2]:
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.decomposition import PCA
import pickle
import os
from transformers import BertTokenizer, BertModel
import torch
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.decomposition import PCA
from networkx.algorithms.cluster import clustering
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models.doc2vec import TaggedDocument
from nltk.tokenize import word_tokenize
from gensim.models.doc2vec import Doc2Vec
from node2vec import Node2Vec


# Load all positive edges from edgelist.txt
edges = pd.read_csv("edgelist.txt", header=None, names=["source", "target"])
positive_edges = list(edges.itertuples(index=False, name=None))

# Split the positive edges into train and validation sets
train_positive_edges, val_positive_edges = train_test_split(
    positive_edges, test_size=0.2, random_state=42
)

# Create the graph only with training edges
G = nx.DiGraph()
G.add_edges_from(train_positive_edges)
undirected_G = G.to_undirected()

print(f"Graph created with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges (training only).")



# Load negative edges from file
negative_edges = set()
with open("negative_edgelist2.txt", "r") as f:
    for line in f:
        u, v = map(int, line.strip().split(','))
        negative_edges.add((u, v))

# Convert to list and split same as positive
negative_edges = list(negative_edges)
train_negative_edges, val_negative_edges = train_test_split(
    negative_edges, test_size=0.2, random_state=42
)

print(f"Loaded {len(negative_edges)} negative edges.")
print(f"Training: {len(train_positive_edges)} pos, {len(train_negative_edges)} neg")
print(f"Validation: {len(val_positive_edges)} pos, {len(val_negative_edges)} neg")

# Optional: Combine training and test sets for classifier input later
train_edges = train_positive_edges + train_negative_edges
val_edges = val_positive_edges + val_negative_edges






abstracts = {}
with open("abstracts.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("|--|")
        abstracts[int(parts[0])] = parts[1]

print("loading authors.txt...")
authors = {}
with open("authors.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("|--|")
        authors[int(parts[0])] = set(parts[1].split(","))
        

# Load the reduced vectors
reduced_vectors = np.load('reduced_sciBERT_embeddings.npy')

# Load the corresponding paper ids
paper_ids = np.load('paper_ids.npy')

# Rebuild the dictionary mapping paper ids to reduced vectors
paper_to_reduced_vector = {pid: reduced_vectors[i] for i, pid in enumerate(paper_ids)}


#with open("closeness_centrality.pkl", "rb") as f:
 #   closeness_centrality = pickle.load(f)

from networkx.algorithms.link_prediction import adamic_adar_index, preferential_attachment

def compute_adamic_adar(graph, u, v):
    try:
        return list(adamic_adar_index(graph, [(u, v)]))[0][2]
    except:
        return 0.0

def compute_preferential_attachment(graph, u, v):
    try:
        return list(preferential_attachment(graph, [(u, v)]))[0][2]
    except:
        return 0.0





# Assuming undirected_G is your undirected version of the graph
clustering_coeffs = clustering(undirected_G)




reduced_embeddings = np.load("reduced_w2v_embeddings.npy")

paper_ids = list(abstracts.keys())

reduced_doc_embeddings = {
    pid: reduced_embeddings[idx] for idx, pid in enumerate(paper_ids)
}






#tf idf


vectorizer = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf_matrix = vectorizer.fit_transform(abstracts.values())


paper_id_to_tfidf = {i: tfidf_matrix[i] for i in range(len(abstracts))}



#doc2vec







from tqdm import tqdm
from gensim.models.doc2vec import TaggedDocument
from nltk.tokenize import word_tokenize

tagged_docs = []

for paper_id, text in tqdm(abstracts.items(), desc="Tokenizing abstracts"):
    tokens = word_tokenize(text.lower())
    tagged_docs.append(TaggedDocument(words=tokens, tags=[str(paper_id)]))


from gensim.models import Doc2Vec
doc2vec_model = Doc2Vec.load("doc2vec_model.model")

paper_to_doc2vec = {
    int(tagged_doc.tags[0]): doc2vec_model.dv[tagged_doc.tags[0]]
    for tagged_doc in tagged_docs
}




#node2vec
node2vec = Node2Vec(G, dimensions=64, walk_length=30, num_walks=20, workers=2)
n2v_model = node2vec.fit(window=6, min_count=1, batch_words=4)


# Create the mapping from paper to node2vec embeddings
paper_to_node2vec = {int(node): n2v_model.wv[str(node)] for node in G.nodes()}



paper_ids = list(paper_to_node2vec.keys())
original_node2vec_matrix = np.array([paper_to_node2vec[pid] for pid in paper_ids])

# 2. Save original Node2Vec embeddings
np.save("node2vec_original.npy", original_node2vec_matrix)
with open("node2vec_ids.npy", "wb") as f:
    np.save(f, paper_ids)










#Optional: Combine training and test sets for classifier input later
#train_edges = train_positive_edges + train_negative_edges
#val_edges = val_positive_edges + val_negative_edges
#train_labels = [1] * len(train_positive_edges) + [0] * len(train_negative_edges)
#val_labels = [1] * len(val_positive_edges) + [0] * len(val_negative_edges)




n_components= 768
X = []
y = []


for a, b in tqdm(train_positive_edges, desc="Creating features for positive edges"):
    a_w2v = reduced_doc_embeddings.get(a, np.zeros(300))  # Existing Word2Vec embeddings
    b_w2v = reduced_doc_embeddings.get(b, np.zeros(300))

   
    a_sci = paper_to_reduced_vector.get(a, np.zeros(n_components))  # ✅
    b_sci = paper_to_reduced_vector.get(b, np.zeros(n_components))  # ✅

    a_tfidf = paper_id_to_tfidf.get(a)
    b_tfidf = paper_id_to_tfidf.get(b)
    tfidf_sim = cosine_similarity(a_tfidf, b_tfidf)[0][0] if a_tfidf is not None and b_tfidf is not None else 0.0



    a_doc2vec = paper_to_doc2vec.get(a, np.zeros(300))
    b_doc2vec = paper_to_doc2vec.get(b, np.zeros(300))


    a_node2vec = paper_to_node2vec.get(a, np.zeros(64))
    b_node2vec = paper_to_node2vec.get(b, np.zeros(64))
    



    X.append([
        clustering_coeffs.get(a, 0.0),
        clustering_coeffs.get(b, 0.0),
        #closeness_centrality.get(a, 0.0),
        #closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_w2v], [b_w2v])[0][0],  # Word2Vec sim
        cosine_similarity([a_sci], [b_sci])[0][0],  # SciBERT sim
        cosine_similarity([a_doc2vec], [b_doc2vec])[0][0],
        tfidf_sim,       
        *a_node2vec,
        *b_node2vec,
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(1)

for a, b in tqdm(train_negative_edges, desc="Creating features for negative edges"):
    a_w2v = reduced_doc_embeddings.get(a, np.zeros(300))  # Existing Word2Vec embeddings
    b_w2v = reduced_doc_embeddings.get(b, np.zeros(300))

    a_sci = paper_to_reduced_vector.get(a, np.zeros(n_components))  # ✅
    b_sci = paper_to_reduced_vector.get(b, np.zeros(n_components))  # ✅


    a_tfidf = paper_id_to_tfidf.get(a)
    b_tfidf = paper_id_to_tfidf.get(b)
    tfidf_sim = cosine_similarity(a_tfidf, b_tfidf)[0][0] if a_tfidf is not None and b_tfidf is not None else 0.0


    a_doc2vec = paper_to_doc2vec.get(a, np.zeros(300))
    b_doc2vec = paper_to_doc2vec.get(b, np.zeros(300))


    a_node2vec = paper_to_node2vec.get(a, np.zeros(64))
    b_node2vec = paper_to_node2vec.get(b, np.zeros(64))


    X.append([
        clustering_coeffs.get(a, 0.0),
        clustering_coeffs.get(b, 0.0),
        #closeness_centrality.get(a, 0.0),
        #closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_w2v], [b_w2v])[0][0],  # Word2Vec sim
        cosine_similarity([a_sci], [b_sci])[0][0],  # SciBERT sim
        cosine_similarity([a_doc2vec], [b_doc2vec])[0][0],
        tfidf_sim,
        *a_node2vec,
        *b_node2vec,
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(0)

from sklearn.preprocessing import StandardScaler

# Convert X to NumPy array (before scaling)
X_before = np.array(X)



# Apply scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_before)


# Convert to NumPy
X_np_scaled_full = np.array(X_scaled)
y_np = np.array(y)



model = LogisticRegression(max_iter=1500, random_state=42)
model.fit(X_np_scaled_full, y_np)
print("Model trained on full dataset.")



X_val = []
y_val = []

# Positive validation edges
for a, b in tqdm(val_positive_edges, desc="Validation: positive"):
    a_w2v = reduced_doc_embeddings.get(a, np.zeros(300))
    b_w2v = reduced_doc_embeddings.get(b, np.zeros(300))

    a_sci = paper_to_reduced_vector.get(a, np.zeros(n_components))
    b_sci = paper_to_reduced_vector.get(b, np.zeros(n_components))

    a_tfidf = paper_id_to_tfidf.get(a)
    b_tfidf = paper_id_to_tfidf.get(b)
    tfidf_sim = cosine_similarity(a_tfidf, b_tfidf)[0][0] if a_tfidf is not None and b_tfidf is not None else 0.0

    a_doc2vec = paper_to_doc2vec.get(a, np.zeros(300))
    b_doc2vec = paper_to_doc2vec.get(b, np.zeros(300))

    a_node2vec = paper_to_node2vec.get(a, np.zeros(64))
    b_node2vec = paper_to_node2vec.get(b, np.zeros(64))
  

    X_val.append([
        clustering_coeffs.get(a, 0.0),
        clustering_coeffs.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_w2v], [b_w2v])[0][0],
        cosine_similarity([a_sci], [b_sci])[0][0],
        cosine_similarity([a_doc2vec], [b_doc2vec])[0][0],
        tfidf_sim,
        *a_node2vec,
        *b_node2vec,
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y_val.append(1)

# Negative validation edges
for a, b in tqdm(val_negative_edges, desc="Validation: negative"):
    a_w2v = reduced_doc_embeddings.get(a, np.zeros(300))
    b_w2v = reduced_doc_embeddings.get(b, np.zeros(300))

    a_sci = paper_to_reduced_vector.get(a, np.zeros(n_components))
    b_sci = paper_to_reduced_vector.get(b, np.zeros(n_components))

    a_tfidf = paper_id_to_tfidf.get(a)
    b_tfidf = paper_id_to_tfidf.get(b)
    tfidf_sim = cosine_similarity(a_tfidf, b_tfidf)[0][0] if a_tfidf is not None and b_tfidf is not None else 0.0

    a_doc2vec = paper_to_doc2vec.get(a, np.zeros(300))
    b_doc2vec = paper_to_doc2vec.get(b, np.zeros(300))

    a_node2vec = paper_to_node2vec.get(a, np.zeros(64))
    b_node2vec = paper_to_node2vec.get(b, np.zeros(64))
    

    X_val.append([
        clustering_coeffs.get(a, 0.0),
        clustering_coeffs.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_w2v], [b_w2v])[0][0],
        cosine_similarity([a_sci], [b_sci])[0][0],
        cosine_similarity([a_doc2vec], [b_doc2vec])[0][0],
        tfidf_sim,
        *a_node2vec,
        *b_node2vec,
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y_val.append(0)


X_val_scaled = scaler.transform(np.array(X_val))
y_val = np.array(y_val)



from sklearn.metrics import log_loss

# Predict class probabilities for validation set
y_proba = model.predict_proba(X_val_scaled)[:, 1]  # Probabilities for the positive class

# Compute Negative Log-Likelihood (Log Loss)
nll = log_loss(y_val, y_proba)

print(f"Negative Log-Likelihood (NLL): {nll:.4f}")

Graph created with 135492 nodes and 873564 edges (training only).
Loaded 1091955 negative edges.
Training: 873564 pos, 873564 neg
Validation: 218391 pos, 218391 neg
loading authors.txt...


Tokenizing abstracts: 100%|███████████| 138499/138499 [00:49<00:00, 2802.56it/s]


Computing transition probabilities:   0%|          | 0/135492 [00:00<?, ?it/s]

Generating walks (CPU: 2): 100%|██████████| 10/10 [00:13<00:00,  1.39s/it]
Creating features for positive edges: 100%|█| 873564/873564 [08:35<00:00, 1693.9
Creating features for negative edges: 100%|█| 873564/873564 [09:26<00:00, 1543.1


Model trained on full dataset.


Validation: negative: 100%|███████████| 218391/218391 [02:13<00:00, 1638.92it/s]


Negative Log-Likelihood (NLL): 0.1242


**GNN**

In [33]:
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.decomposition import PCA
import pickle
import os
from transformers import BertTokenizer, BertModel
import torch
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.decomposition import PCA
from networkx.algorithms.cluster import clustering
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models.doc2vec import TaggedDocument
from nltk.tokenize import word_tokenize
from gensim.models.doc2vec import Doc2Vec
from node2vec import Node2Vec
from gensim.models.doc2vec import TaggedDocument
from gensim.models import Doc2Vec
from networkx.algorithms.link_prediction import adamic_adar_index, preferential_attachment




# Load all positive edges from edgelist.txt
edges = pd.read_csv("edgelist.txt", header=None, names=["source", "target"])
positive_edges = list(edges.itertuples(index=False, name=None))

# Split the positive edges into train and validation sets
train_positive_edges, val_positive_edges = train_test_split(
    positive_edges, test_size=0.2, random_state=42
)

# Create the graph only with training edges
G = nx.DiGraph()
G.add_edges_from(train_positive_edges)
undirected_G = G.to_undirected()

print(f"Graph created with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges (training only).")



# Load negative edges from file
negative_edges = set()
with open("negative_edgelist2.txt", "r") as f:
    for line in f:
        u, v = map(int, line.strip().split(','))
        negative_edges.add((u, v))

# Convert to list and split same as positive
negative_edges = list(negative_edges)
train_negative_edges, val_negative_edges = train_test_split(
    negative_edges, test_size=0.2, random_state=42
)

print(f"Loaded {len(negative_edges)} negative edges.")
print(f"Training: {len(train_positive_edges)} pos, {len(train_negative_edges)} neg")
print(f"Validation: {len(val_positive_edges)} pos, {len(val_negative_edges)} neg")

# Optional: Combine training and test sets for classifier input later
train_edges = train_positive_edges + train_negative_edges
val_edges = val_positive_edges + val_negative_edges




abstracts = {}
with open("abstracts.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("|--|")
        abstracts[int(parts[0])] = parts[1]

print("loading authors.txt...")
authors = {}
with open("authors.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("|--|")
        authors[int(parts[0])] = set(parts[1].split(","))
        

# Load the reduced vectors
reduced_vectors = np.load('reduced_sciBERT_embeddings.npy')

# Load the corresponding paper ids
paper_ids = np.load('paper_ids.npy')

# Rebuild the dictionary mapping paper ids to reduced vectors
paper_to_reduced_vector = {pid: reduced_vectors[i] for i, pid in enumerate(paper_ids)}



# Assume undirected_G is already defined
#closeness_centrality = nx.closeness_centrality(undirected_G)

#with open("closeness_centrality_Train.json", "w") as f:
 #   json.dump(closeness_centrality, f)

with open('closeness_centrality_Train.json', 'r') as f:
    closeness_centrality = json.load(f)



def compute_adamic_adar(graph, u, v):
    try:
        return list(adamic_adar_index(graph, [(u, v)]))[0][2]
    except:
        return 0.0

def compute_preferential_attachment(graph, u, v):
    try:
        return list(preferential_attachment(graph, [(u, v)]))[0][2]
    except:
        return 0.0

def compute_jaccard_similarity(graph, u, v):
    try:
        return list(jaccard_coefficient(graph, [(u, v)]))[0][2]
    except:
        return 0.0

# Assuming undirected_G is your undirected version of the graph
clustering_coeffs = clustering(undirected_G)


reduced_embeddings = np.load("reduced_w2v_embeddings.npy")

paper_ids = list(abstracts.keys())

reduced_doc_embeddings = {
    pid: reduced_embeddings[idx] for idx, pid in enumerate(paper_ids)
}




#tf idf


vectorizer = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf_matrix = vectorizer.fit_transform(abstracts.values())
tfidf_dim = tfidf_matrix.shape[1]


paper_id_to_tfidf = {i: tfidf_matrix[i] for i in range(len(abstracts))}



#doc2vec


tagged_docs = []

for paper_id, text in tqdm(abstracts.items(), desc="Tokenizing abstracts"):
    tokens = word_tokenize(text.lower())
    tagged_docs.append(TaggedDocument(words=tokens, tags=[str(paper_id)]))



doc2vec_model = Doc2Vec.load("doc2vec_model.model")

paper_to_doc2vec = {
    int(tagged_doc.tags[0]): doc2vec_model.dv[tagged_doc.tags[0]]
    for tagged_doc in tagged_docs
}









#gnn




import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from tqdm import tqdm
import numpy as np

# Map paper IDs to index
paper_id_to_index = {pid: i for i, pid in enumerate(paper_ids)}






#x_features = []
#for pid in paper_ids:
    # Base embeddings (Specter2)
 #   spec_vec = paper_to_reduced_vector.get(pid, np.zeros(250))

    # Add optional ones
  #  w2v = reduced_doc_embeddings.get(pid, np.zeros(300))
    
    # Structural scalar features
   # cc = clustering_coeffs.get(pid, 0.0)
    #closeness = closeness_centrality.get(pid, 0.0)

    # Concatenate everything
    #combined = np.concatenate([spec_vec, w2v, [cc, closeness]])
    #x_features.append(combined)

#x = torch.tensor(np.array(x_features), dtype=torch.float)




# Input features (Specter2 vectors)
#x = torch.tensor(np.array([paper_to_reduced_vector[pid] for pid in paper_ids]), dtype=torch.float)

# Graph edges for message passing
#edge_index = torch.tensor([
 #   [paper_id_to_index[u], paper_id_to_index[v]] 
  #  for u, v in train_positive_edges 
   # if u in paper_id_to_index and v in paper_id_to_index
#], dtype=torch.long).t().contiguous()

# Utility: Edge list to tensor
#def edge_pairs_to_tensor(edges):
 #   return torch.tensor([
  #      [paper_id_to_index[u], paper_id_to_index[v]] 
   ##     for u, v in edges 
     #   if u in paper_id_to_index and v in paper_id_to_index
    #], dtype=torch.long).t()

# Prepare training edges
#pos_edge_index = edge_pairs_to_tensor(train_positive_edges)
#neg_edge_index = edge_pairs_to_tensor(train_negative_edges)

# Combine and shuffle positive/negative edges
#all_edge_index = torch.cat([pos_edge_index, neg_edge_index], dim=1)
#labels = torch.cat([
 #   torch.ones(pos_edge_index.shape[1]), 
  #  torch.zeros(neg_edge_index.shape[1])
#])
#perm = torch.randperm(all_edge_index.shape[1])
#all_edge_index = all_edge_index[:, perm]
#labels = labels[perm]

# GraphSAGE model with MLP decoder
#class GraphSAGE_LinkPredictor(torch.nn.Module):
 #   def __init__(self, in_channels, hidden_channels, out_channels):
  #      super().__init__()
   #     self.conv1 = SAGEConv(in_channels, hidden_channels)
    #    self.conv2 = SAGEConv(hidden_channels, out_channels)
     #   self.decoder = torch.nn.Sequential(
      #      torch.nn.Linear(out_channels * 2, 64),
       #     torch.nn.ReLU(),
        #    torch.nn.Linear(64, 1)
        #)

    #def encode(self, x, edge_index):
     #   x = F.relu(self.conv1(x, edge_index))
      #  x = self.conv2(x, edge_index)
       # return x

    #def decode(self, z, edge_pairs):
     #   z_i = z[edge_pairs[0]]
      #  z_j = z[edge_pairs[1]]
       # return self.decoder(torch.cat([z_i, z_j], dim=1)).squeeze()

# Model setup
#model = GraphSAGE_LinkPredictor(in_channels=x.shape[1], hidden_channels=256, out_channels=128)
#optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

#criterion = torch.nn.BCEWithLogitsLoss()

# Training loop
#model.train()
#for epoch in tqdm(range(100), desc="Training GraphSAGE"):
 #   optimizer.zero_grad()
  #  z = model.encode(x, edge_index)
   # preds = model.decode(z, all_edge_index)
   # loss = criterion(preds, labels)
   # loss.backward()
   # optimizer.step()

    #if epoch % 10 == 0:
      #  with torch.no_grad():
     #       pred_probs = torch.sigmoid(preds)
       #     pred_labels = (pred_probs > 0.5).float()
        #    acc = accuracy_score(labels.cpu(), pred_labels.cpu())
         #   f1 = f1_score(labels.cpu(), pred_labels.cpu())
          #  roc_auc = roc_auc_score(labels.cpu(), pred_probs.cpu())
           # print(f"Epoch {epoch} | Loss: {loss.item():.4f} | Acc: {acc:.4f} | F1: {f1:.4f} | ROC-AUC: {roc_auc:.4f}")

# Extract GNN embeddings
#model.eval()
#with torch.no_grad():
 #   gnn_embeddings = model.encode(x, edge_index).cpu()

# Combine Specter2 + GNN embeddings
#combined_embeddings = torch.cat([x.cpu(), gnn_embeddings], dim=1)

# Optional: PCA for dimensionality reduction
#pca = PCA(n_components=50)
#reduced_combined_np = pca.fit_transform(combined_embeddings.numpy())


#np.save("reduced_combined_embeddings.npy", reduced_combined_np)

reduced_combined_np = np.load("reduced_combined_embeddings.npy")


def get_combined_vector(pid):
    return reduced_combined_np[paper_id_to_index[pid]]







degree_centrality = nx.degree_centrality(undirected_G)





n_components= 768
X = []
y = []


for a, b in tqdm(train_positive_edges, desc="Creating features for positive edges"):
    a_w2v = reduced_doc_embeddings.get(a, np.zeros(300))  
    b_w2v = reduced_doc_embeddings.get(b, np.zeros(300))

   
    a_sci = paper_to_reduced_vector.get(a, np.zeros(n_components))  
    b_sci = paper_to_reduced_vector.get(b, np.zeros(n_components)) 

    a_tfidf = paper_id_to_tfidf.get(a)
    b_tfidf = paper_id_to_tfidf.get(b)
    tfidf_sim = cosine_similarity(a_tfidf, b_tfidf)[0][0] if a_tfidf is not None and b_tfidf is not None else 0.0



    a_doc2vec = paper_to_doc2vec.get(a, np.zeros(300))
    b_doc2vec = paper_to_doc2vec.get(b, np.zeros(300))


    emb_u = get_combined_vector(a)
    emb_v = get_combined_vector(b)

     
    
   
    X.append([
        degree_centrality.get(a, 0.0),
        degree_centrality.get(b, 0.0),
        clustering_coeffs.get(a, 0.0),
        clustering_coeffs.get(b, 0.0),
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_w2v], [b_w2v])[0][0],  # Word2Vec sim
        cosine_similarity([a_sci], [b_sci])[0][0],  # SciBERT sim
        cosine_similarity([a_doc2vec], [b_doc2vec])[0][0],
        tfidf_sim,     
        *emb_u,
        *emb_v,
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(1)




for a, b in tqdm(train_negative_edges, desc="Creating features for negative edges"):
    a_w2v = reduced_doc_embeddings.get(a, np.zeros(300))  
    b_w2v = reduced_doc_embeddings.get(b, np.zeros(300))

    a_sci = paper_to_reduced_vector.get(a, np.zeros(n_components)) 
    b_sci = paper_to_reduced_vector.get(b, np.zeros(n_components))


    a_tfidf = paper_id_to_tfidf.get(a)
    b_tfidf = paper_id_to_tfidf.get(b)
    tfidf_sim = cosine_similarity(a_tfidf, b_tfidf)[0][0] if a_tfidf is not None and b_tfidf is not None else 0.0


    a_doc2vec = paper_to_doc2vec.get(a, np.zeros(300))
    b_doc2vec = paper_to_doc2vec.get(b, np.zeros(300))

    emb_u = get_combined_vector(a)
    emb_v = get_combined_vector(b)

   
    
    
    X.append([
        degree_centrality.get(a, 0.0),
        degree_centrality.get(b, 0.0),
        clustering_coeffs.get(a, 0.0),
        clustering_coeffs.get(b, 0.0),
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_w2v], [b_w2v])[0][0],  # Word2Vec sim
        cosine_similarity([a_sci], [b_sci])[0][0],  # SciBERT sim
        cosine_similarity([a_doc2vec], [b_doc2vec])[0][0],
        tfidf_sim,
        *emb_u,
        *emb_v,
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(0)

from sklearn.preprocessing import StandardScaler



X_np = np.array(X)
y_np = np.array(y)

np.save("X_train.npy", X_np)
np.save("y_train.npy", y_np)

# Convert X to NumPy array (before scaling)
X_before = np.array(X)



# Apply scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_before)


# Convert to NumPy
X_np_scaled_full = np.array(X_scaled)
y_np = np.array(y)


from sklearn.utils import shuffle

# Shuffle the data and labels in unison
X_np_scaled_full, y_np = shuffle(X_np_scaled_full, y_np, random_state=42)

# Now train the model
model = LogisticRegression(max_iter=1500, random_state=42)
model.fit(X_np_scaled_full, y_np)
print("Model trained on shuffled dataset.")








X_val = []
y_val = []

# Positive validation edges
for a, b in tqdm(val_positive_edges, desc="Validation: positive"):
    a_w2v = reduced_doc_embeddings.get(a, np.zeros(300))
    b_w2v = reduced_doc_embeddings.get(b, np.zeros(300))

    a_sci = paper_to_reduced_vector.get(a, np.zeros(n_components))
    b_sci = paper_to_reduced_vector.get(b, np.zeros(n_components))

    a_tfidf = paper_id_to_tfidf.get(a)
    b_tfidf = paper_id_to_tfidf.get(b)
    tfidf_sim = cosine_similarity(a_tfidf, b_tfidf)[0][0] if a_tfidf is not None and b_tfidf is not None else 0.0

    a_doc2vec = paper_to_doc2vec.get(a, np.zeros(300))
    b_doc2vec = paper_to_doc2vec.get(b, np.zeros(300))


    emb_u = get_combined_vector(a)
    emb_v = get_combined_vector(b)

  

    X_val.append([
        degree_centrality.get(a, 0.0),
        degree_centrality.get(b, 0.0),
        clustering_coeffs.get(a, 0.0),
        clustering_coeffs.get(b, 0.0),
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_w2v], [b_w2v])[0][0],
        cosine_similarity([a_sci], [b_sci])[0][0],
        cosine_similarity([a_doc2vec], [b_doc2vec])[0][0],
        tfidf_sim,
        *emb_u,
        *emb_v,
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y_val.append(1)

# Negative validation edges
for a, b in tqdm(val_negative_edges, desc="Validation: negative"):
    a_w2v = reduced_doc_embeddings.get(a, np.zeros(300))
    b_w2v = reduced_doc_embeddings.get(b, np.zeros(300))

    a_sci = paper_to_reduced_vector.get(a, np.zeros(n_components))
    b_sci = paper_to_reduced_vector.get(b, np.zeros(n_components))

    a_tfidf = paper_id_to_tfidf.get(a)
    b_tfidf = paper_id_to_tfidf.get(b)
    tfidf_sim = cosine_similarity(a_tfidf, b_tfidf)[0][0] if a_tfidf is not None and b_tfidf is not None else 0.0

    a_doc2vec = paper_to_doc2vec.get(a, np.zeros(300))
    b_doc2vec = paper_to_doc2vec.get(b, np.zeros(300))

    emb_u = get_combined_vector(a)
    emb_v = get_combined_vector(b)
    
    X_val.append([
        
        degree_centrality.get(a, 0.0),
        degree_centrality.get(b, 0.0),
        clustering_coeffs.get(a, 0.0),
        clustering_coeffs.get(b, 0.0),
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_w2v], [b_w2v])[0][0],
        cosine_similarity([a_sci], [b_sci])[0][0],
        cosine_similarity([a_doc2vec], [b_doc2vec])[0][0],
        tfidf_sim,
        *emb_u,
        *emb_v,
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y_val.append(0)


X_val= np.array(X_val)
y_val = np.array(y_val)


np.save("X_val_train.npy",X_val)
np.save("y_val_train.npy", y_val)



X_val_scaled = scaler.transform(X_val)




from sklearn.metrics import log_loss

# Predict class probabilities for validation set
y_proba = model.predict_proba(X_val_scaled)[:, 1]  # Probabilities for the positive class

# Compute Negative Log-Likelihood (Log Loss)
nll = log_loss(y_val, y_proba)

print(f"Negative Log-Likelihood (NLL) with jaccard: {nll:.4f}")

Graph created with 135492 nodes and 873564 edges (training only).
Loaded 1091955 negative edges.
Training: 873564 pos, 873564 neg
Validation: 218391 pos, 218391 neg
loading authors.txt...


Tokenizing abstracts: 100%|███████████| 138499/138499 [00:50<00:00, 2748.93it/s]
Creating features for positive edges: 100%|█| 873564/873564 [08:41<00:00, 1675.8
Creating features for negative edges: 100%|█| 873564/873564 [09:24<00:00, 1548.0


Model trained on shuffled dataset.


Validation: negative: 100%|███████████| 218391/218391 [03:35<00:00, 1011.26it/s]


Negative Log-Likelihood (NLL) with jaccard: 0.1095


**kaggle version**

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
import pickle
import os
from transformers import BertTokenizer, BertModel
import torch
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.decomposition import PCA
from networkx.algorithms.cluster import clustering
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models.doc2vec import TaggedDocument
from nltk.tokenize import word_tokenize
from gensim.models.doc2vec import Doc2Vec





edges = pd.read_csv("edgelist.txt", header=None, names=["source", "target"])
positive_edges = list(edges.itertuples(index=False, name=None))
G = nx.DiGraph()
G.add_edges_from(edges.itertuples(index=False, name=None))
print(f"Graph loaded with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

# Load negative edges from file
negative_edges = set()
with open("negative_edgelist2.txt", "r") as f:
    for line in f:
        u, v = map(int, line.strip().split(','))
        negative_edges.add((u, v))

print(f"Loaded {len(negative_edges)} negative edges from file.")



undirected_G = G.to_undirected()
print("loading abstracts.txt...")
abstracts = {}
with open("abstracts.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("|--|")
        abstracts[int(parts[0])] = parts[1]

print("loading authors.txt...")
authors = {}
with open("authors.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("|--|")
        authors[int(parts[0])] = set(parts[1].split(","))
        

# Load the reduced vectors
reduced_vectors = np.load('reduced_sciBERT_embeddings.npy')

# Load the corresponding paper ids
paper_ids = np.load('paper_ids.npy')

# Rebuild the dictionary mapping paper ids to reduced vectors
paper_to_reduced_vector = {pid: reduced_vectors[i] for i, pid in enumerate(paper_ids)}


with open("closeness_centrality.pkl", "rb") as f:
    closeness_centrality = pickle.load(f)

from networkx.algorithms.link_prediction import adamic_adar_index, preferential_attachment

def compute_adamic_adar(graph, u, v):
    try:
        return list(adamic_adar_index(graph, [(u, v)]))[0][2]
    except:
        return 0.0

def compute_preferential_attachment(graph, u, v):
    try:
        return list(preferential_attachment(graph, [(u, v)]))[0][2]
    except:
        return 0.0





# Assuming undirected_G is your undirected version of the graph
clustering_coeffs = clustering(undirected_G)




reduced_embeddings = np.load("reduced_w2v_embeddings.npy")

paper_ids = list(abstracts.keys())

reduced_doc_embeddings = {
    pid: reduced_embeddings[idx] for idx, pid in enumerate(paper_ids)
}






#tf idf


vectorizer = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf_matrix = vectorizer.fit_transform(abstracts.values())


paper_id_to_tfidf = {i: tfidf_matrix[i] for i in range(len(abstracts))}



#doc2vec







from tqdm import tqdm
from gensim.models.doc2vec import TaggedDocument
from nltk.tokenize import word_tokenize

tagged_docs = []

for paper_id, text in tqdm(abstracts.items(), desc="Tokenizing abstracts"):
    tokens = word_tokenize(text.lower())
    tagged_docs.append(TaggedDocument(words=tokens, tags=[str(paper_id)]))



#doc2vec_model = Doc2Vec(
 #   vector_size=50,  # You can tune this
  #  window=1,
   # min_count=25,
    #workers=4,
    #epochs=400,
    #dm=1  # Distributed Memory (better for similarity tasks)
#)



#doc2vec_model.build_vocab(tagged_docs)
#doc2vec_model.train(tagged_docs, total_examples=doc2vec_model.corpus_count, epochs=doc2vec_model.epochs)

from gensim.models import Doc2Vec
doc2vec_model = Doc2Vec.load("doc2vec_model.model")

paper_to_doc2vec = {
    int(tagged_doc.tags[0]): doc2vec_model.dv[tagged_doc.tags[0]]
    for tagged_doc in tagged_docs
}

##doc2vec_model.save("doc2vec_model.model")






#gnn




import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from tqdm import tqdm
import numpy as np

# Map paper IDs to index
paper_id_to_index = {pid: i for i, pid in enumerate(paper_ids)}

# Input features (Specter2 vectors)
#x = torch.tensor(np.array([paper_to_reduced_vector[pid] for pid in paper_ids]), dtype=torch.float)

#x_features = []
#for pid in paper_ids:
    # Base embeddings (Specter2)
 #   spec_vec = paper_to_reduced_vector.get(pid, np.zeros(250))

    # Add optional ones
  #  w2v = reduced_doc_embeddings.get(pid, np.zeros(300))
    
    # Structural scalar features
   # cc = clustering_coeffs.get(pid, 0.0)
   # closeness = closeness_centrality.get(pid, 0.0)

    # Concatenate everything
    #combined = np.concatenate([spec_vec, w2v, [cc, closeness]])
    #x_features.append(combined)

#x = torch.tensor(np.array(x_features), dtype=torch.float)

# Graph edges for message passing
#edge_index = torch.tensor([
 #   [paper_id_to_index[u], paper_id_to_index[v]] 
  #  for u, v in positive_edges 
   # if u in paper_id_to_index and v in paper_id_to_index
#], dtype=torch.long).t().contiguous()

## Utility: Edge list to tensor
#def edge_pairs_to_tensor(edges):
 #   return torch.tensor([
  #      [paper_id_to_index[u], paper_id_to_index[v]] 
   #     for u, v in edges 
    #    if u in paper_id_to_index and v in paper_id_to_index
    #], dtype=torch.long).t()

# Prepare training edges
#pos_edge_index = edge_pairs_to_tensor(positive_edges)
#neg_edge_index = edge_pairs_to_tensor(negative_edges)

# Combine and shuffle positive/negative edges
#all_edge_index = torch.cat([pos_edge_index, neg_edge_index], dim=1)
#labels = torch.cat([
 #   torch.ones(pos_edge_index.shape[1]), 
  #  torch.zeros(neg_edge_index.shape[1])
#])
#perm = torch.randperm(all_edge_index.shape[1])
#all_edge_index = all_edge_index[:, perm]
#labels = labels[perm]

# GraphSAGE model with MLP decoder
#class GraphSAGE_LinkPredictor(torch.nn.Module):
 #   def __init__(self, in_channels, hidden_channels, out_channels):
  #      super().__init__()
   ##     self.conv1 = SAGEConv(in_channels, hidden_channels)
      #  self.conv2 = SAGEConv(hidden_channels, out_channels)
     #   self.decoder = torch.nn.Sequential(
       #     torch.nn.Linear(out_channels * 2, 64),
        #    torch.nn.ReLU(),
         #   torch.nn.Linear(64, 1)
        #)

    #def encode(self, x, edge_index):
     #   x = F.relu(self.conv1(x, edge_index))
      #  x = self.conv2(x, edge_index)
       # return x

    #def decode(self, z, edge_pairs):
     #   z_i = z[edge_pairs[0]]
      #  z_j = z[edge_pairs[1]]
       # return self.decoder(torch.cat([z_i, z_j], dim=1)).squeeze()

# Model setup
#model = GraphSAGE_LinkPredictor(in_channels=x.shape[1], hidden_channels=256, out_channels=128)
#optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
#criterion = torch.nn.BCEWithLogitsLoss()

# Training loop
#model.train()
#for epoch in tqdm(range(100), desc="Training GraphSAGE"):
 #   optimizer.zero_grad()
  ##  z = model.encode(x, edge_index)
    #preds = model.decode(z, all_edge_index)
    #loss = criterion(preds, labels)
    #loss.backward()
    #optimizer.step()

    #if epoch % 10 == 0:
     #   with torch.no_grad():
      #      pred_probs = torch.sigmoid(preds)
       #     pred_labels = (pred_probs > 0.5).float()
        #    acc = accuracy_score(labels.cpu(), pred_labels.cpu())
         #   f1 = f1_score(labels.cpu(), pred_labels.cpu())
          #  roc_auc = roc_auc_score(labels.cpu(), pred_probs.cpu())
           # print(f"Epoch {epoch} | Loss: {loss.item():.4f} | Acc: {acc:.4f} | F1: {f1:.4f} | ROC-AUC: {roc_auc:.4f}")

# Extract GNN embeddings
#model.eval()
#with torch.no_grad():
 #   gnn_embeddings = model.encode(x, edge_index).cpu()

# Combine Specter2 + GNN embeddings
#combined_embeddings = torch.cat([x.cpu(), gnn_embeddings], dim=1)

# Optional: PCA for dimensionality reduction
#pca = PCA(n_components=50)
#reduced_combined_np = pca.fit_transform(combined_embeddings.numpy())

#np.save("reduced_combined_embeddings_kaggle.npy", reduced_combined_np)

def get_combined_vector(pid):
    return reduced_combined_np [paper_id_to_index[pid]]


degree_centrality = nx.degree_centrality(undirected_G)








reduced_combined_np = np.load("reduced_combined_embeddings_kaggle.npy")











n_components= 768
X = []
y = []


for a, b in tqdm(positive_edges, desc="Creating features for positive edges"):
    a_w2v = reduced_doc_embeddings.get(a, np.zeros(300))  # Existing Word2Vec embeddings
    b_w2v = reduced_doc_embeddings.get(b, np.zeros(300))

   
    a_sci = paper_to_reduced_vector.get(a, np.zeros(n_components))  # ✅
    b_sci = paper_to_reduced_vector.get(b, np.zeros(n_components))  # ✅

    a_tfidf = paper_id_to_tfidf.get(a)
    b_tfidf = paper_id_to_tfidf.get(b)
    tfidf_sim = cosine_similarity(a_tfidf, b_tfidf)[0][0] if a_tfidf is not None and b_tfidf is not None else 0.0



    a_doc2vec = paper_to_doc2vec.get(a, np.zeros(300))
    b_doc2vec = paper_to_doc2vec.get(b, np.zeros(300))
   


    emb_u = get_combined_vector(a)
    emb_v = get_combined_vector(b)


    X.append([
        degree_centrality.get(a, 0.0),
        degree_centrality.get(b, 0.0),
        clustering_coeffs.get(a, 0.0),
        clustering_coeffs.get(b, 0.0),
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_w2v], [b_w2v])[0][0],  # Word2Vec sim
        cosine_similarity([a_sci], [b_sci])[0][0],  # SciBERT sim
        cosine_similarity([a_doc2vec], [b_doc2vec])[0][0],
        tfidf_sim,       
        *emb_u,
        *emb_v,
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(1)

for a, b in tqdm(negative_edges, desc="Creating features for negative edges"):
    a_w2v = reduced_doc_embeddings.get(a, np.zeros(300))  # Existing Word2Vec embeddings
    b_w2v = reduced_doc_embeddings.get(b, np.zeros(300))

    a_sci = paper_to_reduced_vector.get(a, np.zeros(n_components))  # ✅
    b_sci = paper_to_reduced_vector.get(b, np.zeros(n_components))  # ✅


    a_tfidf = paper_id_to_tfidf.get(a)
    b_tfidf = paper_id_to_tfidf.get(b)
    tfidf_sim = cosine_similarity(a_tfidf, b_tfidf)[0][0] if a_tfidf is not None and b_tfidf is not None else 0.0


    a_doc2vec = paper_to_doc2vec.get(a, np.zeros(300))
    b_doc2vec = paper_to_doc2vec.get(b, np.zeros(300))



    emb_u = get_combined_vector(a)
    emb_v = get_combined_vector(b)



    
    X.append([
       
        degree_centrality.get(a, 0.0),
        degree_centrality.get(b, 0.0),
        clustering_coeffs.get(a, 0.0),
        clustering_coeffs.get(b, 0.0),
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_w2v], [b_w2v])[0][0],  # Word2Vec sim
        cosine_similarity([a_sci], [b_sci])[0][0],  # SciBERT sim
        cosine_similarity([a_doc2vec], [b_doc2vec])[0][0],
        tfidf_sim,
        *emb_u,
        *emb_v,
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y.append(0)

from sklearn.preprocessing import StandardScaler


X_np = np.array(X)
y_np = np.array(y)

np.save("X.npy", X_np)
np.save("y.npy", y_np)


# Convert X to NumPy array (before scaling)
X_before = np.array(X)



# Apply scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_before)


# Convert to NumPy
X_np_scaled_full = np.array(X_scaled)
y_np = np.array(y)




from sklearn.utils import shuffle

# Shuffle the data and labels in unison
X_np_scaled_full, y_np = shuffle(X_np_scaled_full, y_np, random_state=42)

# Now train the model
model = LogisticRegression(max_iter=1500, random_state=42)
model.fit(X_np_scaled_full, y_np)
print("Model trained on shuffled dataset.")






X_test = []
test_edges = []

with open("test.txt", "r") as f:
    for line in f:
        a, b = map(int, line.strip().split(","))
        test_edges.append((a, b))

print(f"Loaded {len(test_edges)} edges from test.txt")


for a, b in tqdm(test_edges, desc="Extracting features from test set"):

    a_w2v = reduced_doc_embeddings.get(a, np.zeros(300))  # Existing Word2Vec embeddings
    b_w2v = reduced_doc_embeddings.get(b, np.zeros(300))

    a_sci = paper_to_reduced_vector.get(a, np.zeros(n_components))  # ✅
    b_sci = paper_to_reduced_vector.get(b, np.zeros(n_components))  # ✅


    a_tfidf = paper_id_to_tfidf.get(a)
    b_tfidf = paper_id_to_tfidf.get(b)
    tfidf_sim = cosine_similarity(a_tfidf, b_tfidf)[0][0] if a_tfidf is not None and b_tfidf is not None else 0.0

    a_doc2vec = paper_to_doc2vec.get(a, np.zeros(300))
    b_doc2vec = paper_to_doc2vec.get(b, np.zeros(300))
    

    emb_u = get_combined_vector(a)
    emb_v = get_combined_vector(b)
    

    X_test.append([
        degree_centrality.get(a, 0.0),
        degree_centrality.get(b, 0.0),
        clustering_coeffs.get(a, 0.0),
        clustering_coeffs.get(b, 0.0),
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_w2v], [b_w2v])[0][0],  # Word2Vec sim
        cosine_similarity([a_sci], [b_sci])[0][0],  # SciBERT sim
        cosine_similarity([a_doc2vec], [b_doc2vec])[0][0],
        tfidf_sim,
        *emb_u,
        *emb_v,
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    
X_test_np = np.array(X_test)



import numpy as np

# Save the test feature matrix
np.save("X_test.npy", X_test_np)
print("✅ Saved X_test to X_test.npy")



X_test_scaled = scaler.transform(X_test_np)


y_test_probs = model.predict_proba(X_test_scaled )[:, 1]


submission_df = pd.DataFrame({
    "ID": range(len(y_test_probs)),
    "Label": y_test_probs
})
submission_df.to_csv("finalized.csv", index=False)
print("Submission file saved as 'GraphSageTuned.csv'")

**node2vec+GNN**

In [2]:

from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.decomposition import PCA
import pickle
import os
from transformers import BertTokenizer, BertModel
import torch
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.decomposition import PCA
from networkx.algorithms.cluster import clustering
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models.doc2vec import TaggedDocument
from nltk.tokenize import word_tokenize
from gensim.models.doc2vec import Doc2Vec
from node2vec import Node2Vec
from gensim.models.doc2vec import TaggedDocument
from gensim.models import Doc2Vec
from networkx.algorithms.link_prediction import adamic_adar_index, preferential_attachment
import numpy as np
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle



# Load all positive edges from edgelist.txt
edges = pd.read_csv("edgelist.txt", header=None, names=["source", "target"])
positive_edges = list(edges.itertuples(index=False, name=None))

# Split the positive edges into train and validation sets
train_positive_edges, val_positive_edges = train_test_split(
    positive_edges, test_size=0.2, random_state=42
)

# Create the graph only with training edges
G = nx.DiGraph()
G.add_edges_from(train_positive_edges)
undirected_G = G.to_undirected()

print(f"Graph created with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges (training only).")



# Load negative edges from file
negative_edges = set()
with open("negative_edgelist2.txt", "r") as f:
    for line in f:
        u, v = map(int, line.strip().split(','))
        negative_edges.add((u, v))

# Convert to list and split same as positive
negative_edges = list(negative_edges)
train_negative_edges, val_negative_edges = train_test_split(
    negative_edges, test_size=0.2, random_state=42
)

print(f"Loaded {len(negative_edges)} negative edges.")
print(f"Training: {len(train_positive_edges)} pos, {len(train_negative_edges)} neg")
print(f"Validation: {len(val_positive_edges)} pos, {len(val_negative_edges)} neg")

# Optional: Combine training and test sets for classifier input later
train_edges = train_positive_edges + train_negative_edges
val_edges = val_positive_edges + val_negative_edges




abstracts = {}
with open("abstracts.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("|--|")
        abstracts[int(parts[0])] = parts[1]

print("loading authors.txt...")
authors = {}
with open("authors.txt", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("|--|")
        authors[int(parts[0])] = set(parts[1].split(","))
        

# Load the reduced vectors
reduced_vectors = np.load('reduced_sciBERT_embeddings.npy')

# Load the corresponding paper ids
paper_ids = np.load('paper_ids.npy')

# Rebuild the dictionary mapping paper ids to reduced vectors
paper_to_reduced_vector = {pid: reduced_vectors[i] for i, pid in enumerate(paper_ids)}



# Assume undirected_G is already defined
#closeness_centrality = nx.closeness_centrality(undirected_G)

#with open("closeness_centrality_Train.json", "w") as f:
 #   json.dump(closeness_centrality, f)

with open('closeness_centrality_Train.json', 'r') as f:
    closeness_centrality = json.load(f)



def compute_adamic_adar(graph, u, v):
    try:
        return list(adamic_adar_index(graph, [(u, v)]))[0][2]
    except:
        return 0.0

def compute_preferential_attachment(graph, u, v):
    try:
        return list(preferential_attachment(graph, [(u, v)]))[0][2]
    except:
        return 0.0

def compute_jaccard_similarity(graph, u, v):
    try:
        return list(jaccard_coefficient(graph, [(u, v)]))[0][2]
    except:
        return 0.0

# Assuming undirected_G is your undirected version of the graph
clustering_coeffs = clustering(undirected_G)


reduced_embeddings = np.load("reduced_w2v_embeddings.npy")

paper_ids = list(abstracts.keys())

reduced_doc_embeddings = {
    pid: reduced_embeddings[idx] for idx, pid in enumerate(paper_ids)
}




#tf idf


vectorizer = TfidfVectorizer(stop_words='english', max_features=10000)
tfidf_matrix = vectorizer.fit_transform(abstracts.values())
tfidf_dim = tfidf_matrix.shape[1]


paper_id_to_tfidf = {i: tfidf_matrix[i] for i in range(len(abstracts))}



#doc2vec


tagged_docs = []

for paper_id, text in tqdm(abstracts.items(), desc="Tokenizing abstracts"):
    tokens = word_tokenize(text.lower())
    tagged_docs.append(TaggedDocument(words=tokens, tags=[str(paper_id)]))



doc2vec_model = Doc2Vec.load("doc2vec_model.model")

paper_to_doc2vec = {
    int(tagged_doc.tags[0]): doc2vec_model.dv[tagged_doc.tags[0]]
    for tagged_doc in tagged_docs
}









#gnn




import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from tqdm import tqdm
import numpy as np

# Map paper IDs to index
paper_id_to_index = {pid: i for i, pid in enumerate(paper_ids)}






#x_features = []
#for pid in paper_ids:
    # Base embeddings (Specter2)
 #   spec_vec = paper_to_reduced_vector.get(pid, np.zeros(250))

    # Add optional ones
  #  w2v = reduced_doc_embeddings.get(pid, np.zeros(300))
    
    # Structural scalar features
   # cc = clustering_coeffs.get(pid, 0.0)
    #closeness = closeness_centrality.get(pid, 0.0)

    # Concatenate everything
    #combined = np.concatenate([spec_vec, w2v, [cc, closeness]])
    #x_features.append(combined)

#x = torch.tensor(np.array(x_features), dtype=torch.float)




# Input features (Specter2 vectors)
#x = torch.tensor(np.array([paper_to_reduced_vector[pid] for pid in paper_ids]), dtype=torch.float)

# Graph edges for message passing
#edge_index = torch.tensor([
 #   [paper_id_to_index[u], paper_id_to_index[v]] 
  #  for u, v in train_positive_edges 
   # if u in paper_id_to_index and v in paper_id_to_index
#], dtype=torch.long).t().contiguous()

# Utility: Edge list to tensor
#def edge_pairs_to_tensor(edges):
 #   return torch.tensor([
  #      [paper_id_to_index[u], paper_id_to_index[v]] 
   ##     for u, v in edges 
     #   if u in paper_id_to_index and v in paper_id_to_index
    #], dtype=torch.long).t()

# Prepare training edges
#pos_edge_index = edge_pairs_to_tensor(train_positive_edges)
#neg_edge_index = edge_pairs_to_tensor(train_negative_edges)

# Combine and shuffle positive/negative edges
#all_edge_index = torch.cat([pos_edge_index, neg_edge_index], dim=1)
#labels = torch.cat([
 #   torch.ones(pos_edge_index.shape[1]), 
  #  torch.zeros(neg_edge_index.shape[1])
#])
#perm = torch.randperm(all_edge_index.shape[1])
#all_edge_index = all_edge_index[:, perm]
#labels = labels[perm]

# GraphSAGE model with MLP decoder
#class GraphSAGE_LinkPredictor(torch.nn.Module):
 #   def __init__(self, in_channels, hidden_channels, out_channels):
  #      super().__init__()
   #     self.conv1 = SAGEConv(in_channels, hidden_channels)
    #    self.conv2 = SAGEConv(hidden_channels, out_channels)
     #   self.decoder = torch.nn.Sequential(
      #      torch.nn.Linear(out_channels * 2, 64),
       #     torch.nn.ReLU(),
        #    torch.nn.Linear(64, 1)
        #)

    #def encode(self, x, edge_index):
     #   x = F.relu(self.conv1(x, edge_index))
      #  x = self.conv2(x, edge_index)
       # return x

    #def decode(self, z, edge_pairs):
     #   z_i = z[edge_pairs[0]]
      #  z_j = z[edge_pairs[1]]
       # return self.decoder(torch.cat([z_i, z_j], dim=1)).squeeze()

# Model setup
#model = GraphSAGE_LinkPredictor(in_channels=x.shape[1], hidden_channels=256, out_channels=128)
#optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

#criterion = torch.nn.BCEWithLogitsLoss()

# Training loop
#model.train()
#for epoch in tqdm(range(100), desc="Training GraphSAGE"):
 #   optimizer.zero_grad()
  #  z = model.encode(x, edge_index)
   # preds = model.decode(z, all_edge_index)
   # loss = criterion(preds, labels)
   # loss.backward()
   # optimizer.step()

    #if epoch % 10 == 0:
      #  with torch.no_grad():
     #       pred_probs = torch.sigmoid(preds)
       #     pred_labels = (pred_probs > 0.5).float()
        #    acc = accuracy_score(labels.cpu(), pred_labels.cpu())
         #   f1 = f1_score(labels.cpu(), pred_labels.cpu())
          #  roc_auc = roc_auc_score(labels.cpu(), pred_probs.cpu())
           # print(f"Epoch {epoch} | Loss: {loss.item():.4f} | Acc: {acc:.4f} | F1: {f1:.4f} | ROC-AUC: {roc_auc:.4f}")

# Extract GNN embeddings
#model.eval()
#with torch.no_grad():
 #   gnn_embeddings = model.encode(x, edge_index).cpu()

# Combine Specter2 + GNN embeddings
#combined_embeddings = torch.cat([x.cpu(), gnn_embeddings], dim=1)

# Optional: PCA for dimensionality reduction
#pca = PCA(n_components=50)
#reduced_combined_np = pca.fit_transform(combined_embeddings.numpy())


#np.save("reduced_combined_embeddings.npy", reduced_combined_np)

reduced_combined_np = np.load("reduced_combined_embeddings.npy")


def get_combined_vector(pid):
    return reduced_combined_np[paper_id_to_index[pid]]







degree_centrality = nx.degree_centrality(undirected_G)







#node2vec
node2vec = Node2Vec(G, dimensions=64, walk_length=30, num_walks=20, workers=2)
n2v_model = node2vec.fit(window=6, min_count=1, batch_words=4)


# Create the mapping from paper to node2vec embeddings
paper_to_node2vec = {int(node): n2v_model.wv[str(node)] for node in G.nodes()}



paper_ids = list(paper_to_node2vec.keys())
original_node2vec_matrix = np.array([paper_to_node2vec[pid] for pid in paper_ids])

# 2. Save original Node2Vec embeddings
np.save("node2vec_original.npy", original_node2vec_matrix)
with open("node2vec_ids.npy", "wb") as f:
    np.save(f, paper_ids)

n_components= 768

# Memory-map the array
X_memmap = np.load("X_train.npy", mmap_mode='r')
y_memmap = np.load("y_train.npy", mmap_mode='r')

# Shuffle indices (not data)
indices = np.arange(len(y_memmap))
np.random.seed(42)
np.random.shuffle(indices)

# Initialize scaler and classifier
scaler = StandardScaler()
clf = SGDClassifier(loss="log_loss", max_iter=1, tol=None, random_state=42, warm_start=True)

batch_size = 5000  # You can tune this to avoid memory overload
n_batches = int(np.ceil(len(indices) / batch_size))

#First pass to fit the scaler
for i in tqdm(range(n_batches), desc="Fitting scaler"):
    batch_idx = indices[i*batch_size : (i+1)*batch_size]
    X_batch = X_memmap[batch_idx]
    scaler.partial_fit(X_batch)

# Second pass to train the classifier incrementally
for i in tqdm(range(n_batches), desc="Training classifier"):
    batch_idx = indices[i*batch_size : (i+1)*batch_size]
    X_batch = scaler.transform(X_memmap[batch_idx])
    y_batch = y_memmap[batch_idx]
    clf.partial_fit(X_batch, y_batch, classes=[0, 1])  # Specify all classes in first pass

print("Model trained using SGDClassifier and batching.")





X_val = []
y_val = []

# Positive validation edges
for a, b in tqdm(val_positive_edges, desc="Validation: positive"):
    a_w2v = reduced_doc_embeddings.get(a, np.zeros(300))
    b_w2v = reduced_doc_embeddings.get(b, np.zeros(300))

    a_sci = paper_to_reduced_vector.get(a, np.zeros(n_components))
    b_sci = paper_to_reduced_vector.get(b, np.zeros(n_components))

    a_tfidf = paper_id_to_tfidf.get(a)
    b_tfidf = paper_id_to_tfidf.get(b)
    tfidf_sim = cosine_similarity(a_tfidf, b_tfidf)[0][0] if a_tfidf is not None and b_tfidf is not None else 0.0

    a_doc2vec = paper_to_doc2vec.get(a, np.zeros(300))
    b_doc2vec = paper_to_doc2vec.get(b, np.zeros(300))


    emb_u = get_combined_vector(a)
    emb_v = get_combined_vector(b)

  

    a_node2vec = paper_to_node2vec.get(a, np.zeros(64))
    b_node2vec = paper_to_node2vec.get(b, np.zeros(64))
    
    X_val.append([
        degree_centrality.get(a, 0.0),
        degree_centrality.get(b, 0.0),
        clustering_coeffs.get(a, 0.0),
        clustering_coeffs.get(b, 0.0),
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_w2v], [b_w2v])[0][0],
        cosine_similarity([a_sci], [b_sci])[0][0],
        cosine_similarity([a_doc2vec], [b_doc2vec])[0][0],
        tfidf_sim,
        *emb_u,
        *emb_v,
        *a_node2vec,
        *b_node2vec,
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y_val.append(1)

# Negative validation edges
for a, b in tqdm(val_negative_edges, desc="Validation: negative"):
    a_w2v = reduced_doc_embeddings.get(a, np.zeros(300))
    b_w2v = reduced_doc_embeddings.get(b, np.zeros(300))

    a_sci = paper_to_reduced_vector.get(a, np.zeros(n_components))
    b_sci = paper_to_reduced_vector.get(b, np.zeros(n_components))

    a_tfidf = paper_id_to_tfidf.get(a)
    b_tfidf = paper_id_to_tfidf.get(b)
    tfidf_sim = cosine_similarity(a_tfidf, b_tfidf)[0][0] if a_tfidf is not None and b_tfidf is not None else 0.0

    a_doc2vec = paper_to_doc2vec.get(a, np.zeros(300))
    b_doc2vec = paper_to_doc2vec.get(b, np.zeros(300))

    emb_u = get_combined_vector(a)
    emb_v = get_combined_vector(b)

    a_node2vec = paper_to_node2vec.get(a, np.zeros(64))
    b_node2vec = paper_to_node2vec.get(b, np.zeros(64))
    
    X_val.append([
        
        degree_centrality.get(a, 0.0),
        degree_centrality.get(b, 0.0),
        clustering_coeffs.get(a, 0.0),
        clustering_coeffs.get(b, 0.0),
        closeness_centrality.get(a, 0.0),
        closeness_centrality.get(b, 0.0),
        compute_adamic_adar(undirected_G, a, b),
        compute_preferential_attachment(undirected_G, a, b),
        cosine_similarity([a_w2v], [b_w2v])[0][0],
        cosine_similarity([a_sci], [b_sci])[0][0],
        cosine_similarity([a_doc2vec], [b_doc2vec])[0][0],
        tfidf_sim,
        *emb_u,
        *emb_v,
        *a_node2vec,
        *b_node2vec,
        len(authors.get(a, set()) & authors.get(b, set()))
    ])
    y_val.append(0)


X_val= np.array(X_val)
y_val = np.array(y_val)


np.save("X_val_train.npy",X_val)
np.save("y_val_train.npy", y_val)



X_val_scaled = scaler.transform(X_val)




# Predict class probabilities for validation set
y_proba = clf.predict_proba(X_val_scaled)[:, 1]  # Use `clf`, not `model`

# Compute Negative Log-Likelihood (Log Loss)
nll = log_loss(y_val, y_proba)

print(f"Negative Log-Likelihood (NLL): {nll:.4f}")

Graph created with 135492 nodes and 873564 edges (training only).
Loaded 1091955 negative edges.
Training: 873564 pos, 873564 neg
Validation: 218391 pos, 218391 neg
loading authors.txt...


Tokenizing abstracts: 100%|███████████| 138499/138499 [00:49<00:00, 2803.73it/s]


Computing transition probabilities:   0%|          | 0/135492 [00:00<?, ?it/s]

Training classifier: 100%|████████████████████| 350/350 [02:59<00:00,  1.95it/s]


Model trained using SGDClassifier and batching.


Validation: negative: 100%|███████████| 218391/218391 [02:09<00:00, 1688.79it/s]


Negative Log-Likelihood (NLL): 0.1231


**classifiers**

**regression**

In [2]:
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.decomposition import PCA
import pickle
import os
from transformers import BertTokenizer, BertModel
import torch
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.decomposition import PCA
from networkx.algorithms.cluster import clustering
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models.doc2vec import TaggedDocument
from nltk.tokenize import word_tokenize
from gensim.models.doc2vec import Doc2Vec
from node2vec import Node2Vec
from gensim.models.doc2vec import TaggedDocument
from gensim.models import Doc2Vec
from networkx.algorithms.link_prediction import adamic_adar_index, preferential_attachment
from sklearn.utils import shuffle
from sklearn.metrics import log_loss


print("loading features...")
X_np = np.load("X_train.npy")
y_np = np.load("y_train.npy")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_np)

X_np_scaled_full = np.array(X_scaled)


X_np_scaled_full, y_np = shuffle(X_np_scaled_full, y_np, random_state=42)

print("training model..")
model = LogisticRegression(C=0.02, max_iter=1500, random_state=42)

model.fit(X_np_scaled_full, y_np)


X_val = np.load("X_val_train.npy")
y_val = np.load("y_val_train.npy")

X_val_scaled = scaler.transform(X_val)

y_proba = model.predict_proba(X_val_scaled)[:, 1]  # Probabilities for the positive class


nll = log_loss(y_val, y_proba)

print(f"Negative Log-Likelihood (NLL) : {nll:.4f}")

loading features...
training model..
Negative Log-Likelihood (NLL) : 0.1088


**LogisticRegressionKaggle**

In [4]:
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.decomposition import PCA
import pickle
import os
from transformers import BertTokenizer, BertModel
import torch
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.decomposition import PCA
from networkx.algorithms.cluster import clustering
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models.doc2vec import TaggedDocument
from nltk.tokenize import word_tokenize
from gensim.models.doc2vec import Doc2Vec
from node2vec import Node2Vec
from gensim.models.doc2vec import TaggedDocument
from gensim.models import Doc2Vec
from networkx.algorithms.link_prediction import adamic_adar_index, preferential_attachment
from sklearn.utils import shuffle
from sklearn.metrics import log_loss


print("loading features...")
X_np = np.load("X.npy")
y_np = np.load("y.npy")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_np)

X_np_scaled_full = np.array(X_scaled)


X_np_scaled_full, y_np = shuffle(X_np_scaled_full, y_np, random_state=42)

print("training model..")
model = LogisticRegression(C=0.02, max_iter=1500, random_state=42)

model.fit(X_np_scaled_full, y_np)



X_test_np =np.load("X_test.npy")



X_test_scaled = scaler.transform(X_test_np)


y_test_probs = model.predict_proba(X_test_scaled )[:, 1]


submission_df = pd.DataFrame({
    "ID": range(len(y_test_probs)),
    "Label": y_test_probs
})
submission_df.to_csv("regression.csv", index=False)
print("Submission file saved as 'regression.csv'")

loading features...
training model..
Submission file saved as 'regression.csv'


**RandomForest**

In [2]:
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.decomposition import PCA
import pickle
import os
from transformers import BertTokenizer, BertModel
import torch
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.decomposition import PCA
from networkx.algorithms.cluster import clustering
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models.doc2vec import TaggedDocument
from nltk.tokenize import word_tokenize
from gensim.models.doc2vec import Doc2Vec
from node2vec import Node2Vec
from gensim.models.doc2vec import TaggedDocument
from gensim.models import Doc2Vec
from networkx.algorithms.link_prediction import adamic_adar_index, preferential_attachment
from sklearn.utils import shuffle
from sklearn.metrics import log_loss
from sklearn.ensemble import RandomForestClassifier

print("loading features...")
X_np = np.load("X_train.npy")
y_np = np.load("y_train.npy")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_np)

X_np_scaled_full = np.array(X_scaled)


X_np_scaled_full, y_np = shuffle(X_np_scaled_full, y_np, random_state=42)

print("training model..")
model = RandomForestClassifier(
    n_estimators=21,
    max_depth=19,
    min_samples_split=5,
    min_samples_leaf=20,
    random_state=42
)

model.fit(X_np_scaled_full, y_np)


X_val = np.load("X_val_train.npy")
y_val = np.load("y_val_train.npy")

X_val_scaled = scaler.transform(X_val)

y_proba = model.predict_proba(X_val_scaled)[:, 1]  # Probabilities for the positive class


nll = log_loss(y_val, y_proba)

print(f"Negative Log-Likelihood (NLL) : {nll:.4f}")

loading features...
training model..
Negative Log-Likelihood (NLL) : 0.1152


**XGBoost**

In [122]:
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.decomposition import PCA
import pickle
import os
from transformers import BertTokenizer, BertModel
import torch
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.decomposition import PCA
from networkx.algorithms.cluster import clustering
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models.doc2vec import TaggedDocument
from nltk.tokenize import word_tokenize
from gensim.models.doc2vec import Doc2Vec
from node2vec import Node2Vec
from gensim.models.doc2vec import TaggedDocument
from gensim.models import Doc2Vec
from networkx.algorithms.link_prediction import adamic_adar_index, preferential_attachment
from sklearn.utils import shuffle
from sklearn.metrics import log_loss
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

print("loading features...")
X_np = np.load("X_train.npy")
y_np = np.load("y_train.npy")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_np)

X_np_scaled_full = np.array(X_scaled)


X_np_scaled_full, y_np = shuffle(X_np_scaled_full, y_np, random_state=42)

print("training model..")
model = XGBClassifier(
    n_estimators=180,
    max_depth=3,
    learning_rate=0.06,
    subsample=0.8,
    colsample_bytree=0.9,
    eval_metric='logloss',
    random_state=42
)


model.fit(X_np_scaled_full, y_np)


X_val = np.load("X_val_train.npy")
y_val = np.load("y_val_train.npy")

X_val_scaled = scaler.transform(X_val)

y_proba = model.predict_proba(X_val_scaled)[:, 1]  # Probabilities for the positive class


nll = log_loss(y_val, y_proba)

print(f"Negative Log-Likelihood (NLL) : {nll:.4f}")

loading features...
training model..
Negative Log-Likelihood (NLL) : 0.1020


**XGBoostKaggle**

In [126]:
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.decomposition import PCA
import pickle
import os
from transformers import BertTokenizer, BertModel
import torch
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.decomposition import PCA
from networkx.algorithms.cluster import clustering
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models.doc2vec import TaggedDocument
from nltk.tokenize import word_tokenize
from gensim.models.doc2vec import Doc2Vec
from node2vec import Node2Vec
from gensim.models.doc2vec import TaggedDocument
from gensim.models import Doc2Vec
from networkx.algorithms.link_prediction import adamic_adar_index, preferential_attachment
from sklearn.utils import shuffle
from sklearn.metrics import log_loss


print("loading features...")
X_np = np.load("X.npy")
y_np = np.load("y.npy")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_np)

X_np_scaled_full = np.array(X_scaled)


X_np_scaled_full, y_np = shuffle(X_np_scaled_full, y_np, random_state=42)


print("training model..")
model = XGBClassifier(
    n_estimators=180,
    max_depth=3,
    learning_rate=0.06,
    subsample=0.8,
    colsample_bytree=0.9,
    eval_metric='logloss',
    random_state=42
)


model.fit(X_np_scaled_full, y_np)



X_test_np =np.load("X_test.npy")



X_test_scaled = scaler.transform(X_test_np)


y_test_probs = model.predict_proba(X_test_scaled )[:, 1]


submission_df = pd.DataFrame({
    "ID": range(len(y_test_probs)),
    "Label": y_test_probs
})
submission_df.to_csv("XGboost.csv", index=False)
print("Submission file saved as 'XGboost.csv'")

loading features...
training model..
Submission file saved as 'XGboost.csv'


**LightGBM**

In [66]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle
from sklearn.metrics import log_loss
from lightgbm import LGBMClassifier

print("loading features...")
X_np = np.load("X_train.npy")
y_np = np.load("y_train.npy")

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_np)

# Create consistent feature names
num_features = X_scaled.shape[1]
feature_names = [f"f{i}" for i in range(num_features)]

# Convert to DataFrame with feature names
X_df = pd.DataFrame(X_scaled, columns=feature_names)
X_df, y_np = shuffle(X_df, y_np, random_state=42)

# Train the model
print("training model..")
model = LGBMClassifier(
    num_leaves=27,
    n_estimators=120,
    learning_rate=0.05,
    colsample_bytree=1,
    random_state=42,
    verbose=-1  # suppress training info
)

model.fit(X_df, y_np)

# Load and preprocess validation data
X_val = np.load("X_val_train.npy")
y_val = np.load("y_val_train.npy")
X_val_scaled = scaler.transform(X_val)

# Use the same feature names for validation set
X_val_df = pd.DataFrame(X_val_scaled, columns=feature_names)

# Predict probabilities
y_proba = model.predict_proba(X_val_df)[:, 1]

# Evaluate with log loss
nll = log_loss(y_val, y_proba)
print(f"Negative Log-Likelihood (NLL): {nll:.4f}")


loading features...
training model..
Negative Log-Likelihood (NLL): 0.1016


**LightGBMKaggle**

In [70]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle
from lightgbm import LGBMClassifier

print("loading features...")
X_np = np.load("X.npy")
y_np = np.load("y.npy")

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_np)

# Give feature names to the DataFrame
num_features = X_scaled.shape[1]
feature_names = [f"f{i}" for i in range(num_features)]
X_df = pd.DataFrame(X_scaled, columns=feature_names)

# Shuffle data
X_df, y_np = shuffle(X_df, y_np, random_state=42)

# Train model
print("training model..")
model = LGBMClassifier(
    num_leaves=27,
    n_estimators=120,
    learning_rate=0.05,
    colsample_bytree=1,
    random_state=42,
    verbose=-1
)
model.fit(X_df, y_np)

# Load and scale test data
X_test_np = np.load("X_test.npy")
X_test_scaled = scaler.transform(X_test_np)

# Convert to DataFrame with same feature names
X_test_df = pd.DataFrame(X_test_scaled, columns=feature_names)

# Predict
y_test_probs = model.predict_proba(X_test_df)[:, 1]

# Save submission
submission_df = pd.DataFrame({
    "ID": range(len(y_test_probs)),
    "Label": y_test_probs
})
submission_df.to_csv("LightGBM.csv", index=False)
print("Submission file saved as 'LightGBM.csv'")


loading features...
training model..
Submission file saved as 'LightGBM.csv'


**CatBoost**

In [60]:
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.decomposition import PCA
import pickle
import os
from transformers import BertTokenizer, BertModel
import torch
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.decomposition import PCA
from networkx.algorithms.cluster import clustering
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models.doc2vec import TaggedDocument
from nltk.tokenize import word_tokenize
from gensim.models.doc2vec import Doc2Vec
from node2vec import Node2Vec
from gensim.models.doc2vec import TaggedDocument
from gensim.models import Doc2Vec
from networkx.algorithms.link_prediction import adamic_adar_index, preferential_attachment
from sklearn.utils import shuffle
from sklearn.metrics import log_loss
from sklearn.svm import SVC
from catboost import CatBoostClassifier

print("loading features...")
X_np = np.load("X_train.npy")
y_np = np.load("y_train.npy")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_np)

X_np_scaled_full, y_np = shuffle(X_scaled, y_np, random_state=42)

print("training model..")
model = CatBoostClassifier(
    iterations=150,            # Number of boosting rounds (trees)
    depth=3,                  # Tree depth
    learning_rate=0.07,       # Step size shrinkage
    subsample=0.8,            # Row sampling rate (like XGBoost)
    colsample_bylevel=0.9,    # Feature sampling rate (like colsample_bytree)
    eval_metric='Logloss',    # Loss function
    random_seed=42,
    verbose=100               # Print training progress every 100 iterations
)

model.fit(X_np_scaled_full, y_np)

X_val = np.load("X_val_train.npy")
X_val_scaled = scaler.transform(X_val)
y_val = np.load("y_val_train.npy")

y_proba = model.predict_proba(X_val_scaled)[:, 1]

from sklearn.metrics import log_loss
nll = log_loss(y_val, y_proba)
print(f"Negative Log-Likelihood (NLL) : {nll:.4f}")


loading features...
training model..
0:	learn: 0.5775814	total: 320ms	remaining: 47.7s
100:	learn: 0.0940965	total: 10.5s	remaining: 5.11s
149:	learn: 0.0884048	total: 15.3s	remaining: 0us
Negative Log-Likelihood (NLL) : 0.1013


**CatBoostKaggle**

In [161]:
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.decomposition import PCA
import pickle
import os
from transformers import BertTokenizer, BertModel
import torch
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.decomposition import PCA
from networkx.algorithms.cluster import clustering
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models.doc2vec import TaggedDocument
from nltk.tokenize import word_tokenize
from gensim.models.doc2vec import Doc2Vec
from node2vec import Node2Vec
from gensim.models.doc2vec import TaggedDocument
from gensim.models import Doc2Vec
from networkx.algorithms.link_prediction import adamic_adar_index, preferential_attachment
from sklearn.utils import shuffle
from sklearn.metrics import log_loss
from catboost import CatBoostClassifier

print("loading features...")
X_np = np.load("X.npy")
y_np = np.load("y.npy")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_np)

X_np_scaled_full = np.array(X_scaled)


X_np_scaled_full, y_np = shuffle(X_np_scaled_full, y_np, random_state=42)


print("training model..")
model = CatBoostClassifier(
    iterations=160,            # Number of boosting rounds (trees)
    depth=3,                  # Tree depth
    learning_rate=0.07,       # Step size shrinkage
    subsample=0.8,            # Row sampling rate (like XGBoost)
    colsample_bylevel=0.9,    # Feature sampling rate (like colsample_bytree)
    eval_metric='Logloss',    # Loss function
    random_seed=42,
    verbose=100               # Print training progress every 100 iterations
)




model.fit(X_np_scaled_full, y_np)



X_test_np =np.load("X_test.npy")



X_test_scaled = scaler.transform(X_test_np)


y_test_probs = model.predict_proba(X_test_scaled )[:, 1]


submission_df = pd.DataFrame({
    "ID": range(len(y_test_probs)),
    "Label": y_test_probs
})
submission_df.to_csv("CatBoost.csv", index=False)
print("Submission file saved as 'CatBoost.csv'")

loading features...
training model..
0:	learn: 0.5794784	total: 295ms	remaining: 46.9s
100:	learn: 0.0857339	total: 13.2s	remaining: 7.71s
159:	learn: 0.0795352	total: 20.3s	remaining: 0us
Submission file saved as 'CatBoost.csv'


**HistGradientBoosting**

In [21]:
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.decomposition import PCA
from networkx.algorithms.cluster import clustering
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models.doc2vec import TaggedDocument
from nltk.tokenize import word_tokenize
from gensim.models.doc2vec import Doc2Vec
from node2vec import Node2Vec
from gensim.models.doc2vec import TaggedDocument
from gensim.models import Doc2Vec
from networkx.algorithms.link_prediction import adamic_adar_index, preferential_attachment
from sklearn.utils import shuffle
from sklearn.metrics import log_loss
from catboost import CatBoostClassifier
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import log_loss

print("loading features...")
X_np = np.load("X_train.npy")
y_np = np.load("y_train.npy")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_np)

X_np_scaled_full, y_np = shuffle(X_scaled, y_np, random_state=42)

print("training model..")
model = HistGradientBoostingClassifier(
    learning_rate=0.06,
    max_iter=140,
    max_leaf_nodes=11,
    max_depth=None,
    min_samples_leaf=4,
    l2_regularization=0.0,
    early_stopping=True,
    random_state=42,
    verbose=1
)

model.fit(X_np_scaled_full, y_np)

X_val = np.load("X_val_train.npy")
X_val_scaled = scaler.transform(X_val)
y_val = np.load("y_val_train.npy")

y_proba = model.predict_proba(X_val_scaled)[:, 1]

nll = log_loss(y_val, y_proba)
print(f"Negative Log-Likelihood (NLL) : {nll:.4f}")



loading features...
training model..
Binning 1.421 GB of training data: 3.583 s
Binning 0.158 GB of validation data: 0.358 s
Fitting gradient boosted rounds:
Fit 140 trees in 23.429 s, (1540 total leaves)
Time spent computing histograms: 13.109s
Time spent finding best splits:  0.375s
Time spent applying splits:      1.329s
Time spent predicting:           0.203s
Negative Log-Likelihood (NLL) : 0.1003


**HistGradientBoostingKaggle**

In [159]:
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.decomposition import PCA
import pickle
import os
from transformers import BertTokenizer, BertModel
import torch
from gensim.models import KeyedVectors
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import networkx as nx
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from sklearn.decomposition import PCA
from networkx.algorithms.cluster import clustering
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models.doc2vec import TaggedDocument
from nltk.tokenize import word_tokenize
from gensim.models.doc2vec import Doc2Vec
from node2vec import Node2Vec
from gensim.models.doc2vec import TaggedDocument
from gensim.models import Doc2Vec
from networkx.algorithms.link_prediction import adamic_adar_index, preferential_attachment
from sklearn.utils import shuffle
from sklearn.metrics import log_loss
from catboost import CatBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

print("loading features...")
X_np = np.load("X.npy")
y_np = np.load("y.npy")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_np)

X_np_scaled_full = np.array(X_scaled)


X_np_scaled_full, y_np = shuffle(X_np_scaled_full, y_np, random_state=42)


print("training model..")
model = HistGradientBoostingClassifier(
    learning_rate=0.07, 
    max_iter=220, 
    max_leaf_nodes=12,
    max_depth=None,
    min_samples_leaf=4,
    l2_regularization=0.0,
    early_stopping=True,
    random_state=42,
    verbose=1
)


model.fit(X_np_scaled_full, y_np)



X_test_np =np.load("X_test.npy")



X_test_scaled = scaler.transform(X_test_np)


y_test_probs = model.predict_proba(X_test_scaled )[:, 1]


submission_df = pd.DataFrame({
    "ID": range(len(y_test_probs)),
    "Label": y_test_probs
})
submission_df.to_csv("HistGradientBoosting.csv", index=False)
print("Submission file saved as 'HistGradientBoosting.csv'")

loading features...
training model..
Binning 1.777 GB of training data: 4.464 s
Binning 0.197 GB of validation data: 0.467 s
Fitting gradient boosted rounds:
Fit 230 trees in 39.172 s, (2760 total leaves)
Time spent computing histograms: 23.543s
Time spent finding best splits:  0.527s
Time spent applying splits:      2.250s
Time spent predicting:           0.400s
Submission file saved as 'HistGradientBoosting.csv'


**MLP**

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import log_loss
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle
from tqdm import tqdm

# Load and scale features
print("Loading features...")
X_np = np.load("X_train.npy")
y_np = np.load("y_train.npy")
X_val = np.load("X_val_train.npy")
y_val = np.load("y_val_train.npy")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_np)
X_val_scaled = scaler.transform(X_val)

X_scaled, y_np = shuffle(X_scaled, y_np, random_state=42)

# Define hyperparameter grid
param_grid = {
    'hidden_layer_sizes': [(64,), (128,), (256,),
                           (128, 64), (256, 128), 
                           (256, 128, 64)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam', 'sgd'],
    'alpha': [0.0001, 0.001],   # L2 penalty
    'learning_rate_init': [0.001, 0.01]
}

print("Training MLP models with various configurations...")
results = []

for params in tqdm(list(ParameterGrid(param_grid)), desc="Tuning MLP"):
    model = MLPClassifier(
        max_iter=100,           # You can increase to 200+ if needed
        random_state=42,
        early_stopping=True,
        n_iter_no_change=10,
        verbose=False,
        **params
    )

    try:
        model.fit(X_scaled, y_np)
        y_pred_proba = model.predict_proba(X_val_scaled)[:, 1]
        loss = log_loss(y_val, y_pred_proba)
        results.append((loss, params))
        print(f"NLL: {loss:.4f} | Params: {params}")
    except Exception as e:
        print(f"Failed for params: {params} with error: {e}")

# Sort and print best config
results.sort(key=lambda x: x[0])
print("\nBest Configuration:")
print(f"Log Loss: {results[0][0]:.4f}")
print("Params:", results[0][1])


Loading features...
Training MLP models with various configurations...


Tuning MLP:   1%|▌                                                 | 1/96 [02:59<4:44:08, 179.46s/it]

NLL: 0.2338 | Params: {'activation': 'relu', 'alpha': 0.0001, 'hidden_layer_sizes': (64,), 'learning_rate_init': 0.001, 'solver': 'adam'}


Tuning MLP:   2%|█                                                 | 2/96 [07:39<6:14:13, 238.86s/it]

NLL: 0.1783 | Params: {'activation': 'relu', 'alpha': 0.0001, 'hidden_layer_sizes': (64,), 'learning_rate_init': 0.001, 'solver': 'sgd'}


...

NLL: 0.2323 | Params: {'activation': 'relu', 'alpha': 0.0001, 'hidden_layer_sizes': (256, 128, 64), 'learning_rate_init': 0.01, 'solver': 'sgd'}


Tuning MLP:  26%|███████████▉                                  | 25/96 [3:05:33<11:59:27, 608.00s/it]

NLL: 0.2408 | Params: {'activation': 'relu', 'alpha': 0.001, 'hidden_layer_sizes': (64,), 'learning_rate_init': 0.001, 'solver': 'adam'}


/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")
Tuning MLP:  27%|████████████▍                                 | 26/96 [3:10:38<10:03:11, 517.02s/it]

NLL: 0.1536 | Params: {'activation': 'relu', 'alpha': 0.001, 'hidden_layer_sizes': (64,), 'learning_rate_init': 0.001, 'solver': 'sgd'}


In [4]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import log_loss
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle
from tqdm import tqdm

# Load and scale features
print("Loading features...")
X_np = np.load("X_train.npy")
y_np = np.load("y_train.npy")
X_val = np.load("X_val_train.npy")
y_val = np.load("y_val_train.npy")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_np)
X_val_scaled = scaler.transform(X_val)

X_scaled, y_np = shuffle(X_scaled, y_np, random_state=42)

# Define hyperparameter grid
#param_grid = {
    #'hidden_layer_sizes': [
       #(3, 4),
        
    #],
    #'activation': ['relu'],
    #'solver': ['adam'],
    #'alpha': [0.001],
   # 'learning_rate_init': [0.001],  # refine around 0.001
   # 'momentum': [0.6],  # SGD-specific
    #'nesterovs_momentum': [True, False]
#} MAX_ITER 1 

# Define hyperparameter grid
param_grid = {
    'hidden_layer_sizes': [
       (3, 4)
    ],
    'activation': ['relu'],
    'solver': ['adam'],
    'alpha': [0.001],
    'learning_rate_init': [0.001],  # refine around 0.001
    'momentum': [0.6],  # SGD-specific
    #'nesterovs_momentum': [True, False]
}



print("Training MLP models with various configurations...")
results = []

for params in tqdm(list(ParameterGrid(param_grid)), desc="Tuning MLP"):
    model = MLPClassifier(
        max_iter = 8,           # You can increase to 200+ if needed
        random_state=42,
        early_stopping=True,
        verbose=False,
        **params
    )

    try:
        model.fit(X_scaled, y_np)
        y_pred_proba = model.predict_proba(X_val_scaled)[:, 1]
        loss = log_loss(y_val, y_pred_proba)
        results.append((loss, params))
        print(f"NLL: {loss:.4f} | Params: {params}")
    except Exception as e:
        print(f"Failed for params: {params} with error: {e}")

# Sort and print best config
results.sort(key=lambda x: x[0])
print("\nBest Configuration:")
print(f"Log Loss: {results[0][0]:.4f}")
print("Params:", results[0][1])



Loading features...
Training MLP models with various configurations...


Tuning MLP:   0%|                                         | 0/1 [00:00<?, ?it/s]/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (8) reached and the optimization hasn't converged yet.
  warnings.warn(
Tuning MLP: 100%|█████████████████████████████████| 1/1 [00:34<00:00, 34.00s/it]

NLL: 0.1031 | Params: {'activation': 'relu', 'alpha': 0.001, 'hidden_layer_sizes': (3, 4), 'learning_rate_init': 0.001, 'momentum': 0.6, 'solver': 'adam'}

Best Configuration:
Log Loss: 0.1031
Params: {'activation': 'relu', 'alpha': 0.001, 'hidden_layer_sizes': (3, 4), 'learning_rate_init': 0.001, 'momentum': 0.6, 'solver': 'adam'}
